In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# CELL 1: IMPORT LIBRARIES AND PRINT VERSIONS
# ============================================================

import os, sys, gc, json, time, random, shutil, hashlib, warnings, platform, subprocess, itertools, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy import fft as sp_fft
from tqdm.auto import tqdm

import sklearn
import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")

print("Python       :", sys.version.split()[0])
print("Platform     :", platform.platform())
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("Matplotlib   :", matplotlib.__version__)
print("Pillow       :", Image.__version__ if hasattr(Image, "__version__") else "n/a")
print("TensorFlow   :", tf.__version__)
print("Keras        :", keras.__version__)

In [ ]:
# ============================================================
# CELL 2: GPU / TENSORFLOW / CUDA CHECK AND MIXED PRECISION
# ============================================================

print("=" * 70)
print("nvidia-smi")
print("=" * 70)
try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(out.stdout if out.stdout else out.stderr)
except FileNotFoundError:
    print("nvidia-smi not found (no NVIDIA driver visible).")

gpus = tf.config.list_physical_devices("GPU")
print("Physical GPUs visible to TensorFlow:", gpus)

build = tf.sysconfig.get_build_info()
print("TF built with CUDA :", tf.test.is_built_with_cuda())
print("CUDA version (TF)  :", build.get("cuda_version"))
print("cuDNN version (TF) :", build.get("cudnn_version"))

GPU_INFO = {"available": len(gpus) > 0, "name": None, "compute_capability": None}
MIXED_PRECISION = False

if gpus:
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except RuntimeError as e:
            print("Memory growth could not be set (GPU already initialised):", e)
    details = tf.config.experimental.get_device_details(gpus[0])
    GPU_INFO["name"] = details.get("device_name")
    GPU_INFO["compute_capability"] = details.get("compute_capability")
    print("Using GPU          :", GPU_INFO["name"], "| compute capability:", GPU_INFO["compute_capability"])
    print("NOTE: only GPU:0 is used (no multi-GPU strategy), to keep runs simple and reproducible.")

    cc = GPU_INFO["compute_capability"]
    if cc is not None and tuple(cc) >= (7, 0):
        keras.mixed_precision.set_global_policy("mixed_float16")
        MIXED_PRECISION = True
    else:
        keras.mixed_precision.set_global_policy("float32")

    with tf.device("/GPU:0"):
        a = tf.random.uniform((2048, 2048)); b = tf.random.uniform((2048, 2048))
        t0 = time.time(); c = tf.matmul(a, b); _ = c.numpy(); dt = time.time() - t0
    print(f"GPU matmul test OK ({dt*1000:.1f} ms).")
else:
    print("\n*** WARNING: NO GPU DETECTED. In Kaggle: Settings -> Accelerator -> choose a GPU, then restart. ***")
    keras.mixed_precision.set_global_policy("float32")

print("Global dtype policy:", keras.mixed_precision.global_policy().name)
print("MIXED_PRECISION    :", MIXED_PRECISION)

In [ ]:
# ============================================================
# CELL 3: GLOBAL CONFIGURATION, SEEDS, DIRECTORIES, LABEL MAPPING
# ============================================================

SEED = 42

def set_global_seed(seed: int = SEED):
    """Seeds Python, NumPy and TensorFlow. (GPU kernels can still be mildly non-deterministic.)"""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        pass

set_global_seed(SEED)

# ---------------- Directories ----------------
BASE = Path("/kaggle/working/brain_tumor_project")
DIRS = {k: BASE / k for k in
        ["data", "splits", "models", "checkpoints", "histories", "results",
         "figures", "cv", "tuning", "logs", "gradcam", "final"]}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# ---------------- Classes (same numbering as the reference paper) ----------------
CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]   # labels 0,1,2,3
CLASS_DISPLAY = {0: "Glioma", 1: "Meningioma", 2: "Pituitary", 3: "No Tumor"}
NUM_CLASSES = len(CLASS_NAMES)
LABEL_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}

_FOLDER_ALIASES = {
    "gliomatumor": 0, "glioma": 0,
    "meningiomatumor": 1, "meningioma": 1,
    "pituitarytumor": 2, "pituitary": 2,
    "notumor": 3, "nontumor": 3, "normal": 3, "healthy": 3,
}
def canonical_label(folder_name: str):
    """Maps a folder name like 'glioma_tumor' / 'no_tumor' to its integer label, or None."""
    key = folder_name.lower().replace("_", "").replace(" ", "").replace("-", "")
    return _FOLDER_ALIASES.get(key)

# ---------------- Experiment configuration ----------------
IMG_SIZE = 224
SPLIT_CONFIGS = {
    "70_30": {"train_frac": 0.70, "test_frac": 0.30, "cv_folds": 3},
    "80_20": {"train_frac": 0.80, "test_frac": 0.20, "cv_folds": 5},
    "90_10": {"train_frac": 0.90, "test_frac": 0.10, "cv_folds": 10},
}
VAL_FRACTION = 0.15            # fraction of each training portion held out for early stopping / LR scheduling
ARCHITECTURES = ["custom_cnn", "resnet50", "vgg19", "densenet121", "efficientnetb0"]
ACTIVATIONS = ["relu", "gelu"]
MANDATORY_CV_ARCHS = ["efficientnetb0", "densenet121"]

# ---------------- Duplicate / near-duplicate detection thresholds ----------------
NEARDUP = {
    "phash_strong_max": 6,     # pHash Hamming distance (64 bits) for the 'strong' rule
    "dhash_strong_max": 8,     # dHash Hamming distance (64 bits) for the 'strong' rule
    "corr_min": 0.98,          # correlation of 32x32 grayscale thumbnails for the 'pixel' rule
    "phash_pixel_max": 12,     # pHash upper bound accompanying the 'pixel' rule (limits false positives)
}

CONFIG = {
    "seed": SEED, "img_size": IMG_SIZE, "class_names": CLASS_NAMES,
    "split_configs": SPLIT_CONFIGS, "val_fraction": VAL_FRACTION,
    "architectures": ARCHITECTURES, "activations": ACTIVATIONS,
    "mandatory_cv_architectures": MANDATORY_CV_ARCHS, "near_duplicate_thresholds": NEARDUP,
    "mixed_precision": MIXED_PRECISION, "gpu": GPU_INFO,
    "tf_version": tf.__version__, "keras_version": keras.__version__,
}
with open(DIRS["logs"] / "config_base.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print("Project base directory:", BASE)
print("Class mapping:", {i: n for i, n in enumerate(CLASS_NAMES)})
print("Seed:", SEED, "| Image size:", IMG_SIZE, "| Validation fraction:", VAL_FRACTION)
print("Split configs:", SPLIT_CONFIGS)

In [ ]:
# ============================================================
# CELL 4: RUN REGISTRY, JSON UTILITIES, BACKUP AND RESTORE
# ============================================================

class NumpyEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        if isinstance(o, Path): return str(o)
        return super().default(o)

def save_json(obj, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, cls=NumpyEncoder)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path) as f:
        return json.load(f)

# ---------------- OPTIONAL: restore a previous session ----------------
# Set to the folder of an attached Kaggle Dataset that contains a previous backup, e.g.
# RESTORE_FROM = "/kaggle/input/btc-project-backup"
RESTORE_FROM = "/kaggle/input/datasets/<your-kaggle-username>/backup"

if RESTORE_FROM is not None:
    hits = list(Path(RESTORE_FROM).rglob("run_registry.json"))
    if not hits:
        print("No run_registry.json found under", RESTORE_FROM, "- nothing restored.")
    else:
        src_root = hits[0].parent.parent      # .../<project>/logs/run_registry.json -> <project>
        shutil.copytree(src_root, BASE, dirs_exist_ok=True)
        print("Restored previous project state from:", src_root)

# ---------------- Run registry ----------------
REGISTRY_PATH = DIRS["logs"] / "run_registry.json"

def _load_registry():
    return load_json(REGISTRY_PATH, default={})

def run_is_done(run_id: str) -> bool:
    return run_id in _load_registry()

def register_run(run_id: str, info: dict):
    reg = _load_registry()
    reg[run_id] = {**info, "completed_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    save_json(reg, REGISTRY_PATH)

def list_runs(prefix: str = ""):
    reg = _load_registry()
    return sorted(k for k in reg if k.startswith(prefix))

def backup_project_zip(zip_name: str = "project_backup.zip"):
    """Zips the whole project folder into /kaggle/working/<zip_name> (outside the project dir)."""
    zpath = Path("/kaggle/working") / zip_name
    if zpath.exists(): zpath.unlink()
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in BASE.rglob("*"):
            if p.is_file():
                zf.write(p, p.relative_to(BASE.parent))
    print(f"Backup written: {zpath} ({zpath.stat().st_size/1e6:.1f} MB)")
    return zpath

print("Registry file :", REGISTRY_PATH)
print("Runs recorded :", len(_load_registry()))

In [ ]:
# ============================================================
# CELL 5: DATASET DISCOVERY, PATH VERIFICATION AND RAW INDEX
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def find_dataset_root(max_depth: int = 4):
    """Finds the folder whose name contains 'brain-tumor-classification' under /kaggle/input."""
    inp = Path("/kaggle/input")
    assert inp.exists(), "/kaggle/input does not exist - are you running inside Kaggle?"
    for dirpath, dirnames, _ in os.walk(inp):
        depth = len(Path(dirpath).relative_to(inp).parts)
        if "brain-tumor-classification" in Path(dirpath).name.lower():
            return Path(dirpath)
        if depth >= max_depth:
            dirnames[:] = []
    return None

DATA_ROOT = find_dataset_root()
if DATA_ROOT is None:
    print("Dataset not found. Contents of /kaggle/input:")
    for p in sorted(Path("/kaggle/input").iterdir()): print("  ", p)
    raise FileNotFoundError("Add the dataset 'sartajbhuvaji/brain-tumor-classification-mri' via 'Add Input'.")
print("DATA_ROOT:", DATA_ROOT)

# ---- Directory tree (depth <= 3) with number of image files directly inside each folder ----
print("\nDirectory structure (images directly inside each folder):")
for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
    rel = Path(dirpath).relative_to(DATA_ROOT)
    if len(rel.parts) > 3: 
        dirnames[:] = []; continue
    n_img = sum(Path(f).suffix.lower() in IMG_EXTS for f in filenames)
    print(f"  {'  '*len(rel.parts)}{rel if str(rel) != '.' else '.'}/  [{n_img} images, {len(dirnames)} subfolders]")

# ---- Build index ----
records, ext_counter, non_image, unmapped = [], Counter(), [], []
for dirpath, _, filenames in os.walk(DATA_ROOT):
    for fn in filenames:
        p = Path(dirpath) / fn
        ext = p.suffix.lower()
        ext_counter[ext] += 1
        if ext not in IMG_EXTS:
            non_image.append(str(p)); continue
        label = canonical_label(p.parent.name)
        if label is None:
            unmapped.append(str(p)); continue
        parts_lower = [x.lower() for x in p.relative_to(DATA_ROOT).parts]
        if "training" in parts_lower:  orig_split = "Training"
        elif "testing" in parts_lower: orig_split = "Testing"
        else:                          orig_split = "Unknown"
        records.append({"path": str(p), "rel_path": str(p.relative_to(DATA_ROOT)), "filename": fn,
                        "class_folder": p.parent.name, "label": label, "class_name": CLASS_NAMES[label],
                        "orig_split": orig_split, "ext": ext})

df = (pd.DataFrame(records).sort_values("rel_path").reset_index(drop=True))   # sorted -> reproducible ids
df.insert(0, "image_id", np.arange(len(df)))

print("\nFile extensions found:", dict(ext_counter))
print("Non-image files ignored:", len(non_image), non_image[:5])
print("Image files in unrecognised class folders:", len(unmapped), unmapped[:5])
assert len(unmapped) == 0, "Some images sit in folders that do not map to the 4 classes - inspect above."
assert set(df["label"].unique()) == {0, 1, 2, 3}, "Did not find all four classes."
assert df["rel_path"].is_unique

df.to_csv(DIRS["data"] / "raw_index.csv", index=False)
print(f"\nRaw index built: {len(df)} images, saved to {DIRS['data'] / 'raw_index.csv'}")
print(df.head())

In [ ]:
# ============================================================
# CELL 6: CLASS COUNTS, DISTRIBUTION PLOTS, IMBALANCE ANALYSIS
# ============================================================
from sklearn.utils.class_weight import compute_class_weight

print("Total images indexed:", len(df))
ct = pd.crosstab(df["class_name"], df["orig_split"]).reindex(CLASS_NAMES)
ct["Total"] = ct.sum(axis=1)
ct.loc["Total"] = ct.sum()
print("\nClass-wise counts by ORIGINAL folder:")
print(ct)

# ---- Compare with the numbers printed in the paper (Table 2) ----
paper = pd.DataFrame({"Training": [826, 822, 827, 395], "Testing": [100, 115, 74, 105]}, index=CLASS_NAMES)
paper["Total"] = paper.sum(axis=1)
paper.loc["Total"] = paper.sum()
diff = ct.reindex(columns=paper.columns, fill_value=0) - paper
print("\nPaper Table 2 counts:\n", paper)
print("\nDifference (ours - paper); all zeros means identical:\n", diff)

# ---- Plots ----
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
tot = df["class_name"].value_counts().reindex(CLASS_NAMES)
bars = axes[0].bar([CLASS_DISPLAY[i] for i in range(4)], tot.values,
                   color=sns.color_palette("viridis", 4), edgecolor="black")
for b, v in zip(bars, tot.values):
    axes[0].text(b.get_x() + b.get_width() / 2, v + 10, f"{v}\n({100*v/len(df):.1f}%)", ha="center", fontsize=11)
axes[0].set_title("Overall class distribution (raw dataset)", fontsize=14)
axes[0].set_xlabel("Class", fontsize=12); axes[0].set_ylabel("Number of images", fontsize=12)
axes[0].set_ylim(0, tot.max() * 1.18)

plot_df = df.groupby(["class_name", "orig_split"]).size().unstack(fill_value=0).reindex(CLASS_NAMES)
plot_df.index = [CLASS_DISPLAY[i] for i in range(4)]
plot_df.plot(kind="bar", ax=axes[1], edgecolor="black", rot=0, color=["#4c72b0", "#dd8452", "#999999"][:plot_df.shape[1]])
axes[1].set_title("Class distribution by original folder", fontsize=14)
axes[1].set_xlabel("Class", fontsize=12); axes[1].set_ylabel("Number of images", fontsize=12)
axes[1].legend(title="Original folder", fontsize=11)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "01_class_distribution_raw.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- Imbalance analysis ----
counts = tot.values.astype(float)
print("\nImbalance ratio (max/min):", round(counts.max() / counts.min(), 3))
print("Class shares (%):", {CLASS_DISPLAY[i]: round(100 * c / counts.sum(), 2) for i, c in enumerate(counts)})
cw = compute_class_weight("balanced", classes=np.arange(4), y=df["label"].values)
print("Balanced class weights (raw data, preview only):", {CLASS_DISPLAY[i]: round(w, 3) for i, w in enumerate(cw)})
print("NOTE: final class weights will be recomputed from each TRAINING subset only.")

# Train vs Test folder composition (distribution shift between the original folders)
shares = pd.crosstab(df["orig_split"], df["class_name"], normalize="index").reindex(columns=CLASS_NAMES) * 100
print("\nClass share (%) inside each ORIGINAL folder:\n", shares.round(1))

In [ ]:
# ============================================================
# CELL 7: INTEGRITY CHECK, DIMENSIONS, FORMATS, CHANNELS, INTENSITY
# ============================================================

def inspect_image(path: str) -> dict:
    rec = {"is_corrupt": False, "error": "", "width": np.nan, "height": np.nan, "format": None,
           "mode": None, "gray_like": np.nan, "int_mean": np.nan, "int_std": np.nan,
           "int_min": np.nan, "int_max": np.nan, "file_kb": os.path.getsize(path) / 1024}
    try:
        with Image.open(path) as im:
            im.verify()                      # structural check
    except Exception as e:
        rec["is_corrupt"] = True; rec["error"] = f"verify failed: {e!r}"; return rec
    try:
        with Image.open(path) as im:
            rec["format"], rec["mode"] = im.format, im.mode
            rec["width"], rec["height"] = im.size
            im.load()                        # full decode
            if im.mode == "L":
                rec["gray_like"] = True
            else:
                a = np.asarray(im.convert("RGB"))
                rec["gray_like"] = bool(np.array_equal(a[..., 0], a[..., 1]) and np.array_equal(a[..., 1], a[..., 2]))
            g = np.asarray(im.convert("L"), dtype=np.float32)
            rec["int_mean"], rec["int_std"] = float(g.mean()), float(g.std())
            rec["int_min"], rec["int_max"] = float(g.min()), float(g.max())
    except Exception as e:
        rec["is_corrupt"] = True; rec["error"] = f"decode failed: {e!r}"
    return rec

t0 = time.time()
meta = pd.DataFrame([inspect_image(p) for p in tqdm(df["path"], desc="Inspecting images")])
df = pd.concat([df.drop(columns=[c for c in meta.columns if c in df.columns]), meta], axis=1)
df["is_blank"] = (df["int_std"] < 1.0) & (~df["is_corrupt"])
print(f"Inspection finished in {time.time()-t0:.1f}s")

print("\nCorrupted / unreadable images:", int(df["is_corrupt"].sum()))
if df["is_corrupt"].any():
    print(df.loc[df["is_corrupt"], ["image_id", "rel_path", "error"]].to_string())
print("Blank / near-constant images (std < 1):", int(df["is_blank"].sum()))
if df["is_blank"].any():
    print(df.loc[df["is_blank"], ["image_id", "rel_path", "class_name"]].to_string())

ok = df[~df["is_corrupt"]]
print("\nFile formats:", ok["format"].value_counts().to_dict())
print("Colour modes:", ok["mode"].value_counts().to_dict())
print("Grayscale-like (identical channels) images:", int(ok["gray_like"].sum()), "of", len(ok))
print("Truly coloured images (channels differ):", int((~ok["gray_like"].astype(bool)).sum()))
print("Unique image sizes:", ok.groupby(["width", "height"]).ngroups)
print("\nTop 10 image sizes (W x H):")
print(ok.groupby(["width", "height"]).size().sort_values(ascending=False).head(10))
print("\nSize statistics by class:")
print(ok.groupby("class_name")[["width", "height", "file_kb", "int_mean", "int_std"]].agg(["min", "median", "max"]).round(1))
print("\nNon-square images:", int((ok["width"] != ok["height"]).sum()))

# ---- Plots ----
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for i, cname in enumerate(CLASS_NAMES):
    s = ok[ok["class_name"] == cname]
    axes[0].scatter(s["width"] + np.random.RandomState(i).uniform(-4, 4, len(s)),
                    s["height"] + np.random.RandomState(i + 9).uniform(-4, 4, len(s)),
                    s=8, alpha=0.4, label=CLASS_DISPLAY[i])
axes[0].set_title("Image width vs height (jittered)", fontsize=14)
axes[0].set_xlabel("Width (px)", fontsize=12); axes[0].set_ylabel("Height (px)", fontsize=12); axes[0].legend(fontsize=10)

top = ok.groupby(["width", "height"]).size().sort_values(ascending=False).head(10)
axes[1].barh([f"{w}x{h}" for w, h in top.index][::-1], top.values[::-1], color="#4c72b0", edgecolor="black")
axes[1].set_title("Ten most frequent image sizes", fontsize=14)
axes[1].set_xlabel("Number of images", fontsize=12); axes[1].set_ylabel("Size (W x H)", fontsize=12)

for i, cname in enumerate(CLASS_NAMES):
    axes[2].hist(ok.loc[ok["class_name"] == cname, "int_mean"], bins=40, alpha=0.5, label=CLASS_DISPLAY[i])
axes[2].set_title("Mean pixel intensity per image, by class", fontsize=14)
axes[2].set_xlabel("Mean intensity (0-255)", fontsize=12); axes[2].set_ylabel("Number of images", fontsize=12); axes[2].legend(fontsize=10)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "02_image_properties.png", dpi=150, bbox_inches="tight")
plt.show()

df.to_csv(DIRS["data"] / "raw_index_with_meta.csv", index=False)
print("Saved:", DIRS["data"] / "raw_index_with_meta.csv")

In [ ]:
# ============================================================
# CELL 8: REPRESENTATIVE MRI IMAGES FROM ALL FOUR CLASSES
# ============================================================

rng = np.random.RandomState(SEED)
n_per_class = 5
fig, axes = plt.subplots(4, n_per_class, figsize=(4 * n_per_class, 16))
ok = df[~df["is_corrupt"]]
for r, cname in enumerate(CLASS_NAMES):
    pool = ok[ok["class_name"] == cname]
    chosen = pool.iloc[rng.choice(len(pool), n_per_class, replace=False)]
    for c, (_, row) in enumerate(chosen.iterrows()):
        with Image.open(row["path"]) as im:
            axes[r, c].imshow(np.asarray(im.convert("L")), cmap="gray")
        axes[r, c].set_title(f"{CLASS_DISPLAY[r]} | id {row['image_id']}\n{row['orig_split']} | {int(row['width'])}x{int(row['height'])}", fontsize=10)
        axes[r, c].axis("off")
fig.suptitle("Representative MRI images (random, seed=42)", fontsize=18, y=1.0)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "03_sample_images.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CELL 9: EXACT DUPLICATE DETECTION (FILE BYTES AND DECODED PIXELS)
# ============================================================

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_pixels(path: str) -> str:
    with Image.open(path) as im:
        g = np.asarray(im.convert("L"))
    h = hashlib.sha256()
    h.update(str(g.shape).encode()); h.update(g.tobytes())
    return h.hexdigest()

df["sha256"] = None
df["pixel_sha"] = None
valid_idx = df.index[~df["is_corrupt"]]
sha_list, pix_list = [], []
for p in tqdm(df.loc[valid_idx, "path"], desc="Hashing"):
    sha_list.append(sha256_file(p)); pix_list.append(sha256_pixels(p))
df.loc[valid_idx, "sha256"] = sha_list
df.loc[valid_idx, "pixel_sha"] = pix_list

dfv = df[~df["is_corrupt"]]

def summarise_groups(dfv, key):
    rows = []
    for h, g in dfv.groupby(key):
        if len(g) < 2: continue
        rows.append({"hash": h, "n_copies": len(g), "image_ids": list(g["image_id"]),
                     "classes": sorted(g["class_name"].unique()), "orig_splits": sorted(g["orig_split"].unique()),
                     "label_conflict": g["label"].nunique() > 1,
                     "cross_train_test": {"Training", "Testing"} <= set(g["orig_split"])})
    return pd.DataFrame(rows)

byte_groups = summarise_groups(dfv, "sha256")
EXACT_GROUPS = summarise_groups(dfv, "pixel_sha")

def report(name, gdf):
    if gdf.empty:
        print(f"{name}: no duplicate groups found."); return
    print(f"{name}: {len(gdf)} groups | {int(gdf['n_copies'].sum())} images involved | "
          f"{int((gdf['n_copies']-1).sum())} redundant copies")
    print(f"   groups spanning original Training AND Testing : {int(gdf['cross_train_test'].sum())}")
    print(f"   groups with conflicting class labels          : {int(gdf['label_conflict'].sum())}")

print(f"Valid (non-corrupt) images hashed: {len(dfv)}")
report("Exact duplicates by FILE BYTES   ", byte_groups)
report("Exact duplicates by DECODED PIXELS", EXACT_GROUPS)

if not EXACT_GROUPS.empty:
    print("\nCopy-count distribution (pixel-level):", EXACT_GROUPS["n_copies"].value_counts().sort_index().to_dict())
    print("\nFirst 10 exact-duplicate groups:")
    print(EXACT_GROUPS.head(10)[["n_copies", "image_ids", "classes", "orig_splits", "label_conflict"]].to_string())
    EXACT_GROUPS.to_csv(DIRS["data"] / "exact_duplicate_groups.csv", index=False)

df.to_csv(DIRS["data"] / "raw_index_with_meta_hashes.csv", index=False)

In [ ]:
# ============================================================
# CELL 10: NEAR-DUPLICATE DETECTION (pHASH + dHASH + THUMBNAIL CORRELATION)
# ============================================================

def perceptual_features(path: str):
    with Image.open(path) as im:
        g = im.convert("L")
        t32 = np.asarray(g.resize((32, 32), Image.LANCZOS), dtype=np.float32)
        t98 = np.asarray(g.resize((9, 8), Image.LANCZOS), dtype=np.float32)    # 8 rows x 9 cols
    dct = sp_fft.dctn(t32, norm="ortho")[:8, :8].flatten()
    phash = (dct > np.median(dct[1:])).astype(np.uint8)                          # 64 bits
    dhash = (t98[:, 1:] > t98[:, :-1]).astype(np.uint8).flatten()                # 64 bits
    th = t32.flatten(); th = (th - th.mean()) / (th.std() + 1e-6)
    return phash, dhash, th

dfv = df[~df["is_corrupt"]].reset_index(drop=True)
ids = dfv["image_id"].values
P, D, T = [], [], []
for p in tqdm(dfv["path"], desc="Perceptual features"):
    a, b, c = perceptual_features(p); P.append(a); D.append(b); T.append(c)
P = np.stack(P).astype(np.float32); D = np.stack(D).astype(np.float32); T = np.stack(T).astype(np.float32)

def hamming_matrix(B):
    return B @ (1 - B).T + (1 - B) @ B.T

Dp = hamming_matrix(P)             # (N, N) pHash distances
Dd = hamming_matrix(D)             # (N, N) dHash distances
Cc = (T @ T.T) / T.shape[1]        # (N, N) thumbnail correlations
upper = np.triu(np.ones_like(Dp, dtype=bool), k=1)

# ---- Sensitivity table: how many pairs at each threshold ----
print("Threshold sensitivity (pairs with i<j):")
sens = []
for t in [0, 2, 4, 6, 8, 10, 12]:
    sens.append({"pHash <= t": t, "pairs": int(((Dp <= t) & upper).sum()),
                 "pHash<=t & dHash<=t+2": int(((Dp <= t) & (Dd <= t + 2) & upper).sum())})
print(pd.DataFrame(sens).to_string(index=False))
print("Pairs with thumbnail correlation >= 0.99 / 0.98 / 0.95:",
      [int(((Cc >= c) & upper).sum()) for c in (0.99, 0.98, 0.95)])

# ---- Final rule ----
strong = (Dp <= NEARDUP["phash_strong_max"]) & (Dd <= NEARDUP["dhash_strong_max"])
pixel  = (Cc >= NEARDUP["corr_min"]) & (Dp <= NEARDUP["phash_pixel_max"])
near_mask = (strong | pixel) & upper
ia, ib = np.where(near_mask)

by_id = df.set_index("image_id")
NEAR_PAIRS = pd.DataFrame({
    "id_a": ids[ia], "id_b": ids[ib],
    "phash": Dp[ia, ib].astype(int), "dhash": Dd[ia, ib].astype(int), "corr": Cc[ia, ib].round(4),
})
NEAR_PAIRS["class_a"] = by_id.loc[NEAR_PAIRS["id_a"], "class_name"].values
NEAR_PAIRS["class_b"] = by_id.loc[NEAR_PAIRS["id_b"], "class_name"].values
NEAR_PAIRS["split_a"] = by_id.loc[NEAR_PAIRS["id_a"], "orig_split"].values
NEAR_PAIRS["split_b"] = by_id.loc[NEAR_PAIRS["id_b"], "orig_split"].values
NEAR_PAIRS["exact"] = (by_id.loc[NEAR_PAIRS["id_a"], "pixel_sha"].values == by_id.loc[NEAR_PAIRS["id_b"], "pixel_sha"].values)
NEAR_PAIRS["label_conflict"] = NEAR_PAIRS["class_a"] != NEAR_PAIRS["class_b"]
NEAR_PAIRS["cross_train_test"] = NEAR_PAIRS["split_a"] != NEAR_PAIRS["split_b"]

print(f"\nNear-duplicate pairs (final rule): {len(NEAR_PAIRS)}")
print("   of which exact (identical pixels)      :", int(NEAR_PAIRS["exact"].sum()))
print("   of which NON-exact near-duplicates     :", int((~NEAR_PAIRS["exact"]).sum()))
print("   pairs spanning Training and Testing    :", int(NEAR_PAIRS["cross_train_test"].sum()))
print("   pairs with conflicting class labels    :", int(NEAR_PAIRS["label_conflict"].sum()))
NEAR_PAIRS.to_csv(DIRS["data"] / "near_duplicate_pairs.csv", index=False)

# ---- Visual inspection of non-exact near-duplicate pairs ----
cand = NEAR_PAIRS[~NEAR_PAIRS["exact"]].sort_values(["cross_train_test", "phash"], ascending=[False, True]).head(6)
if len(cand):
    fig, axes = plt.subplots(len(cand), 2, figsize=(7, 3.4 * len(cand)))
    axes = np.atleast_2d(axes)
    for r, (_, row) in enumerate(cand.iterrows()):
        for c, key in enumerate(["id_a", "id_b"]):
            rr = by_id.loc[row[key]]
            with Image.open(rr["path"]) as im:
                axes[r, c].imshow(np.asarray(im.convert("L")), cmap="gray")
            axes[r, c].set_title(f"id {row[key]} | {rr['class_name']} | {rr['orig_split']}\n"
                                 f"pHash={row['phash']} dHash={row['dhash']} corr={row['corr']}", fontsize=9)
            axes[r, c].axis("off")
    plt.suptitle("Non-exact near-duplicate pairs (cross-folder pairs first)", fontsize=13)
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "04_near_duplicate_examples.png", dpi=130, bbox_inches="tight")
    plt.show()
else:
    print("No non-exact near-duplicate pairs to display.")

del Dp, Dd, Cc, T, P, D, near_mask, strong, pixel, upper
gc.collect()

In [ ]:
# ============================================================
# CELL 11: DUPLICATE POLICY, GROUP IDS, LEAKAGE REPORT, CLEANED INDEX
# ============================================================

class UnionFind:
    def __init__(self, items):
        self.p = {i: i for i in items}
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            if ra < rb: self.p[rb] = ra
            else:       self.p[ra] = rb          # root is always the smallest image_id

dfv = df[~df["is_corrupt"]].copy()

# ---------- (A) RAW leakage analysis: original Testing images that have a copy in Training ----------
uf_raw = UnionFind(dfv["image_id"].tolist())
for _, g in dfv.groupby("pixel_sha"):
    ids_g = g["image_id"].tolist()
    for j in ids_g[1:]: uf_raw.union(ids_g[0], j)
for a, b in zip(NEAR_PAIRS["id_a"], NEAR_PAIRS["id_b"]): uf_raw.union(int(a), int(b))
dfv["raw_comp"] = dfv["image_id"].map(uf_raw.find)
comp_splits = dfv.groupby("raw_comp")["orig_split"].agg(lambda s: set(s))
dfv["comp_has_train"] = dfv["raw_comp"].map(lambda c: "Training" in comp_splits[c])
dfv["comp_has_test"]  = dfv["raw_comp"].map(lambda c: "Testing" in comp_splits[c])

test_imgs = dfv[dfv["orig_split"] == "Testing"]
leaky = test_imgs[test_imgs["comp_has_train"]]
print("=" * 70)
print("LEAKAGE REPORT FOR THE ORIGINAL KAGGLE TRAINING/TESTING FOLDERS")
print("=" * 70)
print(f"Original Testing images                              : {len(test_imgs)}")
print(f"  ...with an exact/near-duplicate in Training folder : {len(leaky)} ({100*len(leaky)/max(len(test_imgs),1):.1f}%)")
if len(leaky):
    print("  by class:", leaky["class_name"].value_counts().to_dict())
print(f"Duplicate components spanning both folders           : {int((comp_splits.map(lambda s: len(s) > 1)).sum())}")

# ---------- (B) Removal policy ----------
removed = []
def add_removed(row, reason, kept=None):
    removed.append({"image_id": int(row["image_id"]), "rel_path": row["rel_path"], "class_name": row["class_name"],
                    "orig_split": row["orig_split"], "reason": reason, "kept_image_id": kept})

for _, r in df[df["is_corrupt"]].iterrows():
    add_removed(r, "corrupt_or_unreadable")

for h, g in dfv.groupby("pixel_sha"):
    if len(g) < 2: continue
    if g["label"].nunique() > 1:
        for _, r in g.iterrows(): add_removed(r, "exact_duplicate_label_conflict")
    else:
        keep = int(g["image_id"].min())
        for _, r in g[g["image_id"] != keep].iterrows(): add_removed(r, "exact_duplicate_redundant", kept=keep)

removed_df = pd.DataFrame(removed, columns=["image_id", "rel_path", "class_name", "orig_split", "reason", "kept_image_id"])
removed_ids = set(removed_df["image_id"])
removed_df.to_csv(DIRS["data"] / "removed_images.csv", index=False)

print("\n" + "=" * 70)
print("REMOVAL LOG (nothing is removed silently)")
print("=" * 70)
print(f"Images excluded in total: {len(removed_df)}")
if len(removed_df):
    print(removed_df["reason"].value_counts().to_string())
    print("\nBy class and original folder:")
    print(removed_df.groupby(["class_name", "orig_split"]).size().to_string())
print("Full log saved to:", DIRS["data"] / "removed_images.csv")

# ---------- (C) Cleaned set and near-duplicate GROUP ids ----------
df_clean = df[(~df["is_corrupt"]) & (~df["image_id"].isin(removed_ids))].copy().sort_values("image_id").reset_index(drop=True)
clean_ids = set(df_clean["image_id"])
uf = UnionFind(df_clean["image_id"].tolist())
np_clean = NEAR_PAIRS[NEAR_PAIRS["id_a"].isin(clean_ids) & NEAR_PAIRS["id_b"].isin(clean_ids)]
for a, b in zip(np_clean["id_a"], np_clean["id_b"]): uf.union(int(a), int(b))

root = df_clean["image_id"].map(uf.find)
root_to_gid = {r: g for g, r in enumerate(sorted(root.unique()))}
df_clean["group_id"] = root.map(root_to_gid).astype(int)
df_clean.insert(0, "clean_idx", np.arange(len(df_clean)))        # position in the cached array (CELL 12)

grp = df_clean.groupby("group_id").agg(n=("image_id", "size"), n_labels=("label", "nunique"),
                                       splits=("orig_split", lambda s: set(s)))
print("\n" + "=" * 70)
print("CLEANED DATASET AND GROUPING")
print("=" * 70)
print(f"Cleaned images: {len(df_clean)} | groups: {len(grp)} | groups with >1 image: {int((grp['n'] > 1).sum())}")
print("Group size distribution:", grp["n"].value_counts().sort_index().to_dict())
print("Largest group size:", int(grp["n"].max()))
print("Near-duplicate groups with CONFLICTING labels (kept grouped, flagged):", int((grp["n_labels"] > 1).sum()))
print("Groups spanning original Training AND Testing:", int(grp["splits"].map(lambda s: len(s) > 1).sum()))
print("\nCleaned class counts:")
print(df_clean["class_name"].value_counts().reindex(CLASS_NAMES).to_string())
print("\nCleaned counts by original folder:")
print(pd.crosstab(df_clean["class_name"], df_clean["orig_split"]).reindex(CLASS_NAMES).to_string())

keep_cols = ["clean_idx", "image_id", "path", "rel_path", "filename", "label", "class_name", "orig_split",
             "group_id", "width", "height", "mode", "format", "int_mean", "int_std", "sha256", "pixel_sha"]
df_clean[keep_cols].to_csv(DIRS["data"] / "clean_index.csv", index=False)
grp.reset_index().assign(splits=lambda d: d["splits"].map(sorted)).to_csv(DIRS["data"] / "group_summary.csv", index=False)
np_clean.to_csv(DIRS["data"] / "near_duplicate_pairs_clean.csv", index=False)
print("\nSaved clean_index.csv, group_summary.csv, near_duplicate_pairs_clean.csv in", DIRS["data"])

assert df_clean["image_id"].is_unique and df_clean["clean_idx"].is_unique
assert df_clean["pixel_sha"].is_unique, "Exact duplicates remain in the cleaned set!"

In [ ]:
# ============================================================
# CELL 12: PREPROCESSING FUNCTION AND CACHED 224x224 ARRAY
# ============================================================

PREPROC_CONFIG = {
    "img_size": IMG_SIZE, "color": "grayscale_replicated_to_3_channels", "pad_to_square": True, "pad_value": 0,
    "resample": "BICUBIC", "dtype": "uint8", "normalisation": "applied inside each model (architecture-specific)",
    "class_names": CLASS_NAMES,
}
save_json(PREPROC_CONFIG, DIRS["logs"] / "preprocessing_config.json")

def preprocess_pil_image(im: Image.Image, size: int = IMG_SIZE) -> np.ndarray:
    """Single preprocessing routine shared by training, validation, CV, test and inference. Returns uint8 (size,size,3)."""
    g = im.convert("L")
    w, h = g.size
    s = max(w, h)
    if w != h:
        canvas = Image.new("L", (s, s), 0)
        canvas.paste(g, ((s - w) // 2, (s - h) // 2))
        g = canvas
    g = g.resize((size, size), Image.BICUBIC)
    a = np.asarray(g, dtype=np.uint8)
    return np.repeat(a[..., None], 3, axis=2)

def preprocess_path(path: str, size: int = IMG_SIZE) -> np.ndarray:
    with Image.open(path) as im:
        im.load()
        return preprocess_pil_image(im, size)

X_PATH, Y_PATH, ID_PATH = DIRS["data"] / "X_clean_224.npy", DIRS["data"] / "y_clean.npy", DIRS["data"] / "ids_clean.npy"

if X_PATH.exists() and Y_PATH.exists() and ID_PATH.exists() and len(np.load(Y_PATH)) == len(df_clean):
    print("Cache already exists and matches the cleaned index - loading from disk.")
    X_all = np.load(X_PATH); y_all = np.load(Y_PATH); ids_all = np.load(ID_PATH)
else:
    X_all = np.zeros((len(df_clean), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for i, p in enumerate(tqdm(df_clean["path"], desc="Preprocessing + caching")):
        X_all[i] = preprocess_path(p)
    y_all = df_clean["label"].values.astype(np.int64)
    ids_all = df_clean["image_id"].values.astype(np.int64)
    np.save(X_PATH, X_all); np.save(Y_PATH, y_all); np.save(ID_PATH, ids_all)

groups_all = df_clean["group_id"].values.astype(np.int64)
assert X_all.shape == (len(df_clean), IMG_SIZE, IMG_SIZE, 3) and X_all.dtype == np.uint8
assert np.array_equal(y_all, df_clean["label"].values) and np.array_equal(ids_all, df_clean["image_id"].values)

print("X_all:", X_all.shape, X_all.dtype, f"| {X_all.nbytes/1e6:.0f} MB | value range [{X_all.min()}, {X_all.max()}]")
print("y_all class counts:", {CLASS_DISPLAY[k]: int(v) for k, v in zip(*np.unique(y_all, return_counts=True))})
print("Number of groups:", len(np.unique(groups_all)))

# ---- Visual check: native image vs preprocessed ----
rng = np.random.RandomState(SEED)
pick = rng.choice(len(df_clean), 4, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8.4))
for c, i in enumerate(pick):
    with Image.open(df_clean.loc[i, "path"]) as im:
        axes[0, c].imshow(np.asarray(im.convert("L")), cmap="gray")
        axes[0, c].set_title(f"Original | {CLASS_DISPLAY[int(y_all[i])]} | {im.size[0]}x{im.size[1]}", fontsize=10)
    axes[1, c].imshow(X_all[i]); axes[1, c].set_title(f"Preprocessed {IMG_SIZE}x{IMG_SIZE}", fontsize=10)
    axes[0, c].axis("off"); axes[1, c].axis("off")
plt.suptitle("Preprocessing check: native vs pad-to-square 224x224", fontsize=14)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "05_preprocessing_check.png", dpi=130, bbox_inches="tight")
plt.show()

print("\nCached files:", X_PATH.name, Y_PATH.name, ID_PATH.name)

In [ ]:
# ============================================================
# CELL 13: SHORTCUT-RISK DIAGNOSTIC (METADATA-ONLY BASELINE)
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix

meta = df_clean.reset_index(drop=True).copy()
meta["aspect"] = meta["width"] / meta["height"]
meta["bytes_per_px"] = meta["file_kb"] * 1024.0 / (meta["width"] * meta["height"])

FEATS_A = ["width", "height", "aspect", "file_kb", "bytes_per_px"]   # file/geometry only, no pixel content
FEATS_B = FEATS_A + ["int_mean", "int_std"]                           # + global intensity statistics
y_m, g_m = meta["label"].values, meta["group_id"].values

def oof_predict(feats, n_splits=5):
    Xm = meta[feats].values.astype(np.float64)
    oof = np.full(len(meta), -1, dtype=int)
    skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for tr, te in skf.split(Xm, y_m, g_m):
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced",
                                     random_state=SEED, n_jobs=-1)
        clf.fit(Xm[tr], y_m[tr])
        oof[te] = clf.predict(Xm[te])
    assert (oof >= 0).all()
    return oof

def score_dict(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "balanced_acc": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0)}

majority = np.full_like(y_m, np.bincount(y_m).argmax())
res = {"Majority class": score_dict(y_m, majority)}
oof_A = oof_predict(FEATS_A); res["Metadata A (size/format)"] = score_dict(y_m, oof_A)
oof_B = oof_predict(FEATS_B); res["Metadata B (+ intensity stats)"] = score_dict(y_m, oof_B)
res_df = pd.DataFrame(res).T.round(3)

print("Class prediction from NON-PATHOLOGICAL metadata only (group-aware 5-fold, out-of-fold):")
print(res_df.to_string())
print("\n(Chance-level macro-F1 for 4 classes is roughly 0.25.)")
print("\nPer-class recall, metadata set A:")
cm_A = confusion_matrix(y_m, oof_A, labels=list(range(4)), normalize="true")
print(pd.DataFrame(cm_A.diagonal(), index=[CLASS_DISPLAY[i] for i in range(4)], columns=["recall"]).round(3).T.to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
res_df[["accuracy", "balanced_acc", "macro_f1"]].plot(kind="bar", ax=axes[0], edgecolor="black", rot=15)
axes[0].axhline(0.25, color="gray", ls="--", lw=1, label="~chance (macro-F1)")
axes[0].set_title("Predicting the class from metadata alone", fontsize=14)
axes[0].set_xlabel("Feature set", fontsize=12); axes[0].set_ylabel("Score", fontsize=12)
axes[0].set_ylim(0, 1); axes[0].legend(fontsize=10)
sns.heatmap(cm_A, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=axes[1],
            xticklabels=[CLASS_DISPLAY[i] for i in range(4)], yticklabels=[CLASS_DISPLAY[i] for i in range(4)])
axes[1].set_title("Row-normalised confusion matrix (metadata set A)", fontsize=14)
axes[1].set_xlabel("Predicted", fontsize=12); axes[1].set_ylabel("True", fontsize=12)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "06_shortcut_risk.png", dpi=150, bbox_inches="tight")
plt.show()

f1A = res_df.loc["Metadata A (size/format)", "macro_f1"]
level = "LOW" if f1A < 0.40 else ("MODERATE" if f1A < 0.60 else "HIGH")
print(f"\nShortcut-risk level from file/geometry metadata alone: {level} (macro-F1 = {f1A:.3f})")
print("Interpretation: this is a property of how the dataset was assembled, not something our preprocessing can remove.")
print("It will be reported as a limitation and checked with Grad-CAM (does attention fall on borders/background?).")
res_df.to_csv(DIRS["results"] / "shortcut_risk_metadata_baseline.csv")

In [ ]:
# ============================================================
# CELL 14: NESTED STRATIFIED GROUP SPLITS + CV FOLD ASSIGNMENTS
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold

N_POOL_FOLDS = 10     # 10 disjoint group-aware stratified folds -> nested test sets of ~10 / 20 / 30 %
VAL_SPLITS = 7        # 1 fold of 7 = 14.3 % validation (nominal 15 % in CELL 3)
N = len(y_all)
all_idx = np.arange(N)
SPLIT_FILE = DIRS["splits"] / "splits.json"
DATA_SIGNATURE = hashlib.sha256(y_all.tobytes() + groups_all.tobytes() + ids_all.tobytes()).hexdigest()

def sgk_folds(idx, n_splits, seed):
    """Group-aware, class-stratified folds over `idx` (positions into the cleaned arrays).
    Returns a list of sorted arrays; each is one held-out fold; together they partition `idx`."""
    idx = np.asarray(idx)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [np.sort(idx[te]) for _, te in sgkf.split(np.zeros(len(idx)), y_all[idx], groups_all[idx])]

def generate_splits():
    pool_folds = sgk_folds(all_idx, N_POOL_FOLDS, SEED)
    pool_fold_of = np.zeros(N, dtype=int)
    for k, f in enumerate(pool_folds): pool_fold_of[f] = k
    test_sets = {"90_10": pool_folds[0],
                 "80_20": np.sort(np.concatenate(pool_folds[:2])),
                 "70_30": np.sort(np.concatenate(pool_folds[:3]))}
    core = np.sort(np.concatenate(pool_folds[3:]))            # 70 % never used by any test set

    out = {"signature": DATA_SIGNATURE, "seed": SEED, "n_images": int(N), "val_splits": VAL_SPLITS,
           "n_pool_folds": N_POOL_FOLDS, "pool_fold_of_item": pool_fold_of.tolist(),
           "core_pool_idx": core.tolist(), "splits": {}}

    tune_val = sgk_folds(core, VAL_SPLITS, SEED)[0]
    out["tuning"] = {"fit_idx": np.setdiff1d(core, tune_val).tolist(), "val_idx": tune_val.tolist()}

    for name, cfg in SPLIT_CONFIGS.items():
        test_idx = test_sets[name]
        train_idx = np.setdiff1d(all_idx, test_idx)
        val_idx = sgk_folds(train_idx, VAL_SPLITS, SEED)[0]
        fit_idx = np.setdiff1d(train_idx, val_idx)
        cv = []
        for k, ho in enumerate(sgk_folds(train_idx, cfg["cv_folds"], SEED)):
            ftr = np.setdiff1d(train_idx, ho)
            ival = sgk_folds(ftr, VAL_SPLITS, SEED + k)[0]            # inner validation: early stopping / LR / checkpoint
            cv.append({"fold": k, "heldout_idx": ho.tolist(),
                       "fit_idx": np.setdiff1d(ftr, ival).tolist(), "inner_val_idx": ival.tolist()})
        out["splits"][name] = {"train_idx": train_idx.tolist(), "fit_idx": fit_idx.tolist(),
                               "val_idx": val_idx.tolist(), "test_idx": test_idx.tolist(), "cv": cv}
    return out

SPLITS_RAW = load_json(SPLIT_FILE)
if SPLITS_RAW is not None and SPLITS_RAW.get("signature") == DATA_SIGNATURE:
    print("Loaded existing splits from", SPLIT_FILE, "(signature matches the cleaned dataset).")
else:
    if SPLITS_RAW is not None:
        print("WARNING: stored splits do not match the current cleaned dataset - regenerating.")
    SPLITS_RAW = generate_splits()
    save_json(SPLITS_RAW, SPLIT_FILE)
    print("Generated and saved splits to", SPLIT_FILE)

def get_split(name):
    s = SPLITS_RAW["splits"][name]
    return {k: np.array(s[k], dtype=np.int64) for k in ["train_idx", "fit_idx", "val_idx", "test_idx"]}

def get_cv_folds(name):
    return [{"fold": f["fold"], **{k: np.array(f[k], dtype=np.int64) for k in ["heldout_idx", "fit_idx", "inner_val_idx"]}}
            for f in SPLITS_RAW["splits"][name]["cv"]]

def get_tuning_split():
    t = SPLITS_RAW["tuning"]
    return {"fit_idx": np.array(t["fit_idx"], dtype=np.int64), "val_idx": np.array(t["val_idx"], dtype=np.int64)}

# ---- Summary ----
print(f"\nCleaned images: {N} | groups: {len(np.unique(groups_all))}")
for name, cfg in SPLIT_CONFIGS.items():
    s = get_split(name)
    tf_real = len(s["test_idx"]) / N
    print(f"[{name}] train={len(s['train_idx'])} (fit={len(s['fit_idx'])}, val={len(s['val_idx'])}) | "
          f"test={len(s['test_idx'])} ({100*tf_real:.1f}% vs nominal {100*cfg['test_frac']:.0f}%) | CV folds={len(get_cv_folds(name))}")
    assert abs(tf_real - cfg["test_frac"]) < 0.025, "test fraction deviates too much from nominal"
print("Tuning split (core 70% pool): fit =", len(get_tuning_split()["fit_idx"]), "| val =", len(get_tuning_split()["val_idx"]))

# ---- Human-readable assignment table ----
assign = pd.DataFrame({"clean_idx": all_idx, "image_id": ids_all, "group_id": groups_all,
                       "label": y_all, "class_name": [CLASS_NAMES[i] for i in y_all],
                       "pool_fold": SPLITS_RAW["pool_fold_of_item"],
                       "in_core_pool": np.isin(all_idx, SPLITS_RAW["core_pool_idx"])})
for name in SPLIT_CONFIGS:
    s = get_split(name); lab = np.empty(N, dtype=object)
    lab[s["fit_idx"]] = "fit"; lab[s["val_idx"]] = "val"; lab[s["test_idx"]] = "test"
    assign[f"subset_{name}"] = lab
assign.to_csv(DIRS["splits"] / "split_assignments.csv", index=False)
print("Saved:", DIRS["splits"] / "split_assignments.csv")

In [ ]:
# ============================================================
# CELL 15: LEAKAGE VERIFICATION (HARD ASSERTIONS)
# ============================================================

id_to_pos = {int(i): p for p, i in enumerate(ids_all)}
near_a = np.array([id_to_pos[int(a)] for a in np_clean["id_a"]], dtype=int)
near_b = np.array([id_to_pos[int(b)] for b in np_clean["id_b"]], dtype=int)
print(f"Near-duplicate pairs inside the cleaned set to be checked against every partition: {len(near_a)}")

PASSED = []

def check_partition(parts: dict, universe, tag: str):
    """parts: name -> index array. Verifies exclusivity, exact coverage of `universe`,
    no group split across parts, and no near-duplicate pair split across parts."""
    v = np.full(N, -1, dtype=int)
    for code, (nm, idx) in enumerate(parts.items()):
        idx = np.asarray(idx)
        assert len(np.unique(idx)) == len(idx), f"[{tag}] repeated indices inside '{nm}'"
        assert (v[idx] == -1).all(), f"[{tag}] an image appears in more than one subset (involving '{nm}')"
        v[idx] = code
    assigned = np.where(v >= 0)[0]
    assert np.array_equal(assigned, np.sort(np.asarray(universe))), f"[{tag}] subsets do not exactly cover the universe"
    gdf = pd.DataFrame({"g": groups_all[assigned], "c": v[assigned]})
    assert gdf.groupby("g")["c"].nunique().max() == 1, f"[{tag}] a duplicate group is split across subsets"
    m = (v[near_a] >= 0) & (v[near_b] >= 0)
    assert (v[near_a][m] == v[near_b][m]).all(), f"[{tag}] a near-duplicate pair is split across subsets"
    PASSED.append(tag)

assert df_clean["pixel_sha"].is_unique, "Exact duplicates remain in the cleaned set"
PASSED.append("no exact duplicates in cleaned set")

T = {n: set(get_split(n)["test_idx"].tolist()) for n in SPLIT_CONFIGS}
assert T["90_10"] < T["80_20"] < T["70_30"], "test sets are not nested"
PASSED.append("nested test sets T10 < T20 < T30")

for name in SPLIT_CONFIGS:
    s = get_split(name)
    check_partition({"fit": s["fit_idx"], "val": s["val_idx"], "test": s["test_idx"]}, all_idx, f"{name}: fit/val/test")
    assert set(s["train_idx"].tolist()) == set(s["fit_idx"].tolist()) | set(s["val_idx"].tolist())
    assert not (set(s["train_idx"].tolist()) & T[name])

    folds = get_cv_folds(name)
    assert len(folds) == SPLIT_CONFIGS[name]["cv_folds"], "wrong number of CV folds"
    held = np.concatenate([f["heldout_idx"] for f in folds])
    assert len(held) == len(np.unique(held)) and set(held.tolist()) == set(s["train_idx"].tolist()), \
        f"{name}: held-out folds do not partition the training portion"
    for f in folds:
        check_partition({"fit": f["fit_idx"], "inner_val": f["inner_val_idx"], "heldout": f["heldout_idx"]},
                        s["train_idx"], f"{name}: CV fold {f['fold']}")
        assert not (set(f["heldout_idx"].tolist()) & T[name]), "held-out fold overlaps the test set"
        assert not (set(f["fit_idx"].tolist()) & T[name]) and not (set(f["inner_val_idx"].tolist()) & T[name])
    PASSED.append(f"{name}: {len(folds)} CV folds partition the training portion and never touch the test set")

core = np.array(SPLITS_RAW["core_pool_idx"], dtype=np.int64)
assert not (set(core.tolist()) & T["70_30"]), "core pool intersects the largest test set"
tu = get_tuning_split()
check_partition({"tune_fit": tu["fit_idx"], "tune_val": tu["val_idx"]}, core, "tuning split inside core pool")
PASSED.append("core pool is disjoint from every test set")

print("\nALL LEAKAGE CHECKS PASSED:")
for p in PASSED: print("   [OK]", p)

# ---- Informational: where did each subset's images come from in the ORIGINAL Kaggle folders? ----
orig = df_clean["orig_split"].values
prov = []
for name in SPLIT_CONFIGS:
    s = get_split(name)
    for sub in ["fit", "val", "test"]:
        idx = s[f"{sub}_idx"]
        prov.append({"split": name, "subset": sub, "n": len(idx),
                     "from_orig_Training_%": round(100 * np.mean(orig[idx] == "Training"), 1),
                     "from_orig_Testing_%": round(100 * np.mean(orig[idx] == "Testing"), 1)})
prov = pd.DataFrame(prov)
print("\nProvenance of each subset (share from the original Kaggle folders):")
print(prov.to_string(index=False))
print("NOTE: the original folders are deliberately mixed - the project does not use them as train/test sets.")

save_json({"checks_passed": PASSED, "n_checks": len(PASSED), "signature": DATA_SIGNATURE,
           "verified_at": time.strftime("%Y-%m-%d %H:%M:%S")}, DIRS["logs"] / "leakage_verification.json")
prov.to_csv(DIRS["splits"] / "subset_provenance.csv", index=False)

In [ ]:
# ============================================================
# CELL 16: SUBSET COUNTS, CLASS DISTRIBUTIONS, CV FOLD COMPOSITION
# ============================================================

overall_share = np.bincount(y_all, minlength=NUM_CLASSES) / N * 100
rows = []
for name in SPLIT_CONFIGS:
    s = get_split(name)
    for sub, idx in [("fit", s["fit_idx"]), ("val", s["val_idx"]), ("train (fit+val)", s["train_idx"]), ("test", s["test_idx"])]:
        c = np.bincount(y_all[idx], minlength=NUM_CLASSES)
        rows.append({"split": name, "subset": sub, "n": len(idx), "pct_of_all": round(100 * len(idx) / N, 1),
                     **{CLASS_NAMES[i]: int(c[i]) for i in range(NUM_CLASSES)},
                     **{f"{CLASS_NAMES[i]}_%": round(100 * c[i] / len(idx), 1) for i in range(NUM_CLASSES)}})
summary = pd.DataFrame(rows)
summary.to_csv(DIRS["splits"] / "split_summary.csv", index=False)

print("Overall class share (%):", {CLASS_NAMES[i]: round(overall_share[i], 1) for i in range(NUM_CLASSES)})
for name in SPLIT_CONFIGS:
    print(f"\n=== Experiment {name} ===")
    print(summary[summary["split"] == name].drop(columns="split").to_string(index=False))
    sub = summary[(summary["split"] == name) & (summary["subset"].isin(["fit", "val", "test"]))]
    dev = max(abs(r[f"{CLASS_NAMES[i]}_%"] - overall_share[i]) for _, r in sub.iterrows() for i in range(NUM_CLASSES))
    print(f"   max |class share - overall share| over fit/val/test: {dev:.2f} percentage points")
    nt = int(sub[sub["subset"] == "test"].iloc[0]["no_tumor"])
    print(f"   No-Tumor images in the TEST set: {nt}  -> per-class estimates for this class carry wide uncertainty")

# ---- Figure 1: shares and counts per subset ----
subs = ["fit", "val", "test"]
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
pal = sns.color_palette("viridis", NUM_CLASSES)
for j, name in enumerate(SPLIT_CONFIGS):
    sub = summary[(summary["split"] == name) & (summary["subset"].isin(subs))].set_index("subset").loc[subs]
    ax = axes[0, j]; x = np.arange(NUM_CLASSES); w = 0.26
    for k, sname in enumerate(subs):
        ax.bar(x + (k - 1) * w, [sub.loc[sname, f"{c}_%"] for c in CLASS_NAMES], w, edgecolor="black",
               label=f"{sname} (n={int(sub.loc[sname, 'n'])})")
    ax.plot(x, overall_share, "k--o", lw=1.2, label="overall share")
    ax.set_xticks(x); ax.set_xticklabels([CLASS_DISPLAY[i] for i in range(NUM_CLASSES)])
    ax.set_title(f"Experiment {name}: class share per subset", fontsize=14)
    ax.set_xlabel("Class", fontsize=12); ax.set_ylabel("Share of subset (%)", fontsize=12); ax.legend(fontsize=10)
    ax = axes[1, j]; bottom = np.zeros(len(subs))
    for i, c in enumerate(CLASS_NAMES):
        vals = np.array([sub.loc[sn, c] for sn in subs], dtype=float)
        ax.bar(subs, vals, bottom=bottom, color=pal[i], edgecolor="black", label=CLASS_DISPLAY[i]); bottom += vals
    for k, sn in enumerate(subs): ax.text(k, bottom[k] + 15, str(int(bottom[k])), ha="center", fontsize=11)
    ax.set_title(f"Experiment {name}: image counts", fontsize=14)
    ax.set_xlabel("Subset", fontsize=12); ax.set_ylabel("Number of images", fontsize=12); ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "07_split_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- CV fold composition ----
cv_rows = []
for name in SPLIT_CONFIGS:
    for f in get_cv_folds(name):
        c = np.bincount(y_all[f["heldout_idx"]], minlength=NUM_CLASSES)
        cv_rows.append({"split": name, "fold": f["fold"], "n_fit": len(f["fit_idx"]), "n_inner_val": len(f["inner_val_idx"]),
                        "n_heldout": len(f["heldout_idx"]), **{CLASS_NAMES[i]: int(c[i]) for i in range(NUM_CLASSES)}})
cv_summary = pd.DataFrame(cv_rows)
cv_summary.to_csv(DIRS["splits"] / "cv_fold_summary.csv", index=False)
for name in SPLIT_CONFIGS:
    d = cv_summary[cv_summary["split"] == name]
    print(f"\n=== CV folds for {name} ({len(d)}-fold, on the {len(get_split(name)['train_idx'])}-image training portion) ===")
    print(d.drop(columns="split").to_string(index=False))
    print(f"   smallest No-Tumor count in a held-out fold: {int(d['no_tumor'].min())}")

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for j, name in enumerate(SPLIT_CONFIGS):
    d = cv_summary[cv_summary["split"] == name]; bottom = np.zeros(len(d))
    for i, c in enumerate(CLASS_NAMES):
        axes[j].bar(d["fold"].astype(str), d[c], bottom=bottom, color=pal[i], edgecolor="black", label=CLASS_DISPLAY[i])
        bottom += d[c].values
    axes[j].set_title(f"{name}: held-out fold composition ({len(d)} folds)", fontsize=14)
    axes[j].set_xlabel("Fold", fontsize=12); axes[j].set_ylabel("Images in held-out fold", fontsize=12); axes[j].legend(fontsize=9)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "08_cv_fold_composition.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CELL 17: CLASS WEIGHTS, TRAIN-ONLY STATS, AUGMENTATION, tf.data PIPELINE
# ============================================================
from sklearn.utils.class_weight import compute_class_weight

DEFAULT_BATCH = 32

def class_weights_from_labels(y_sub):
    """Balanced class weights computed from the labels of ONE training subset only."""
    classes = np.arange(NUM_CLASSES)
    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y_sub))
    return {int(c): float(x) for c, x in zip(classes, w)}

def compute_pixel_stats(indices, chunk=256):
    """Mean/std of pixel intensities (scale 0-1) over the GIVEN indices only (call with training indices)."""
    indices = np.asarray(indices); s = ss = 0.0; n = 0
    for i in range(0, len(indices), chunk):
        a = X_all[indices[i:i + chunk], ..., 0].astype(np.float64) / 255.0
        s += a.sum(); ss += (a ** 2).sum(); n += a.size
    mean = s / n
    return float(mean), float(np.sqrt(max(ss / n - mean ** 2, 1e-12)))

# ---------------- Augmentation ----------------
AUG_PRESETS = {
    "none": None,
    "conservative": dict(rotation_deg=10, zoom=0.10, translate=0.05, brightness=0.10, contrast=0.10, hflip=False),
    "conservative_hflip": dict(rotation_deg=10, zoom=0.10, translate=0.05, brightness=0.10, contrast=0.10, hflip=True),
}
DEFAULT_AUG = "conservative"
save_json({"presets": AUG_PRESETS, "default": DEFAULT_AUG,
           "notes": "Applied on-the-fly to TRAINING data only; black fill; brightness/contrast are multiplicative; no vertical flip."},
          DIRS["logs"] / "augmentation_config.json")

def build_augmenter(preset: str):
    """Returns a function mapping a float32 batch (B,H,W,3) in [0,255] to an augmented batch, or None."""
    cfg = AUG_PRESETS[preset]
    if cfg is None:
        return None
    layers = []
    if cfg["hflip"]:
        layers.append(keras.layers.RandomFlip("horizontal", seed=SEED + 1, dtype="float32"))
    layers += [
        keras.layers.RandomRotation(cfg["rotation_deg"] / 360.0, fill_mode="constant", fill_value=0.0,
                                    interpolation="bilinear", seed=SEED + 2, dtype="float32"),
        keras.layers.RandomZoom(cfg["zoom"], fill_mode="constant", fill_value=0.0,
                                interpolation="bilinear", seed=SEED + 3, dtype="float32"),
        keras.layers.RandomTranslation(cfg["translate"], cfg["translate"], fill_mode="constant", fill_value=0.0,
                                       interpolation="bilinear", seed=SEED + 4, dtype="float32"),
    ]
    geo = keras.Sequential(layers, name="geometric_aug")
    b, c = cfg["brightness"], cfg["contrast"]

    def augment(x):
        x = tf.cast(geo(x, training=True), tf.float32)
        n = tf.shape(x)[0]
        x = x * tf.random.uniform((n, 1, 1, 1), 1.0 - b, 1.0 + b)                 # multiplicative brightness (keeps black background)
        m = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
        x = (x - m) * tf.random.uniform((n, 1, 1, 1), 1.0 - c, 1.0 + c) + m        # contrast about the image mean
        return tf.clip_by_value(x, 0.0, 255.0)
    return augment

def make_dataset(indices, batch_size=DEFAULT_BATCH, training=False, aug_preset="none",
                 class_weights=None, seed=SEED):
    """tf.data pipeline over positions `indices` of the cached arrays.
    - training=True : reshuffled every epoch, optional augmentation, optional per-sample class weights (x, y, w)
    - training=False: deterministic order, NO augmentation, NO weights (x, y)
    x is float32 in [0,255] (architecture-specific preprocessing lives inside each model); y is one-hot."""
    if (not training) and aug_preset != "none":
        raise ValueError("Augmentation requested for a non-training dataset - this is forbidden (leakage guard).")
    indices = np.asarray(indices)
    ds = tf.data.Dataset.from_tensor_slices((X_all[indices], y_all[indices]))
    if training:
        ds = ds.shuffle(len(indices), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    augment = build_augmenter(aug_preset) if training else None
    wt = tf.constant([class_weights[i] for i in range(NUM_CLASSES)], tf.float32) if (training and class_weights) else None

    def finish(x, y):
        if augment is not None:
            x = augment(x)
        y1 = tf.one_hot(y, NUM_CLASSES)
        return (x, y1, tf.gather(wt, y)) if wt is not None else (x, y1)

    ds = ds.map(finish, num_parallel_calls=1 if augment is not None else tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

# ---------------- Report class weights and normalisation stats per experiment (training data only) ----------------
print("Balanced class weights computed from each experiment's FIT subset only:")
stats_rows = []
for name in SPLIT_CONFIGS:
    s = get_split(name)
    cw = class_weights_from_labels(y_all[s["fit_idx"]])
    mean, std = compute_pixel_stats(s["fit_idx"])
    stats_rows.append({"split": name, **{CLASS_NAMES[i]: round(cw[i], 3) for i in range(NUM_CLASSES)},
                       "fit_pixel_mean": round(mean, 4), "fit_pixel_std": round(std, 4)})
print(pd.DataFrame(stats_rows).to_string(index=False))
print("\nPixel mean/std are fitted on the fit subset only and are used only by the from-scratch custom CNN.")
print("Pretrained models use their own preprocess_input (fixed, data-independent) inside the model.")

In [ ]:
# ============================================================
# CELL 18: AUGMENTATION DEMO AND PIPELINE SANITY TESTS
# ============================================================

s = get_split("80_20")
fit_idx, val_idx = s["fit_idx"], s["val_idx"]
cw = class_weights_from_labels(y_all[fit_idx])
ds_tr = make_dataset(fit_idx, DEFAULT_BATCH, training=True, aug_preset=DEFAULT_AUG, class_weights=cw)
ds_va = make_dataset(val_idx, DEFAULT_BATCH, training=False)

xb, yb, wb = next(iter(ds_tr))
xv, yv = next(iter(ds_va))
print("TRAIN batch:", xb.shape, xb.dtype, "| y:", yb.shape, "| w:", wb.shape,
      f"| pixel range [{float(tf.reduce_min(xb)):.1f}, {float(tf.reduce_max(xb)):.1f}]")
print("VAL   batch:", xv.shape, xv.dtype, "| y:", yv.shape)

# Test 1: validation pipeline is deterministic and un-augmented (identical to cached images, in order)
xv2, _ = next(iter(ds_va))
assert np.array_equal(xv.numpy(), xv2.numpy()), "validation batches differ between passes"
assert np.array_equal(xv.numpy(), X_all[val_idx[:DEFAULT_BATCH]].astype(np.float32)), "validation images were modified"
print("[OK] validation pipeline: deterministic and pixel-identical to the cached, un-augmented images")

# Test 2: leakage guard
raised = False
try:
    make_dataset(val_idx, DEFAULT_BATCH, training=False, aug_preset="conservative")
except ValueError:
    raised = True
assert raised, "augmentation on non-training data was NOT rejected"
print("[OK] augmentation on validation/test data is rejected with an error")

# Test 3: augmentation changes pixels, stays in range, and is random between calls
aug = build_augmenter(DEFAULT_AUG)
x0 = tf.cast(X_all[fit_idx[:16]], tf.float32)
a1, a2 = aug(x0).numpy(), aug(x0).numpy()
assert not np.array_equal(a1, a2), "augmentation is not random between calls"
assert a1.min() >= 0.0 and a1.max() <= 255.0
print(f"[OK] augmentation is random, range-safe; mean |change| = {np.abs(a1 - x0.numpy()).mean():.2f} intensity levels (0-255 scale)")

# Test 4: per-sample weights match class weights
pred_cls = yb.numpy().argmax(1)
assert np.allclose(wb.numpy(), [cw[int(c)] for c in pred_cls]), "sample weights do not match class weights"
print("[OK] per-sample weights equal the training-subset class weights:", {CLASS_DISPLAY[k]: round(v, 3) for k, v in cw.items()})

# Visual demonstration: original + 5 independent augmentations, one image per class
rng = np.random.RandomState(SEED)
fig, axes = plt.subplots(NUM_CLASSES, 6, figsize=(18, 12.5))
for r in range(NUM_CLASSES):
    i = int(rng.choice(fit_idx[y_all[fit_idx] == r]))
    x_one = X_all[i].astype(np.float32)[None]
    axes[r, 0].imshow(X_all[i]); axes[r, 0].set_title(f"Original | {CLASS_DISPLAY[r]}", fontsize=11)
    for c in range(1, 6):
        a = aug(tf.constant(x_one)).numpy()[0]
        axes[r, c].imshow(a.astype(np.uint8)); axes[r, c].set_title(f"Augmented {c}", fontsize=11)
    for c in range(6): axes[r, c].axis("off")
fig.suptitle(f"Original vs augmented training images (preset '{DEFAULT_AUG}': rot +/-10 deg, zoom +/-10%, shift +/-5%, brightness/contrast +/-10%)", fontsize=14, y=1.0)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "09_augmentation_examples.png", dpi=130, bbox_inches="tight")
plt.show()

del ds_tr, ds_va, xb, yb, wb, xv, xv2, x0, a1, a2
gc.collect()
print("\nStage D complete. Splits, verification, class weights, augmentation and data pipeline are ready.")

In [ ]:
# ============================================================
# CELL 20: MULTICLASS METRIC SUITE, CONFIDENCE INTERVALS, SELF-TEST
# ============================================================
from sklearn.metrics import (precision_recall_fscore_support, cohen_kappa_score, roc_auc_score,
                             average_precision_score, roc_curve, precision_recall_curve, classification_report)
# confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score were imported in CELL 13

PROB_EPS = 1e-7

def _safe_div(a, b):
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    out = np.full(a.shape, np.nan)
    np.divide(a, b, out=out, where=b > 0)
    return out

def expected_calibration_error(y_true, probs, n_bins=15):
    conf = probs.max(axis=1); correct = (probs.argmax(axis=1) == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def compute_metrics(y_true, probs):
    """All metrics for 4-class classification from true labels (n,) and class probabilities (n,4)."""
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs, dtype=np.float64)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-12, None)
    n, K = len(y_true), NUM_CLASSES
    y_pred = probs.argmax(axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(K)))
    tp = np.diag(cm).astype(float); fp = cm.sum(axis=0) - tp; fn = cm.sum(axis=1) - tp; tn = n - tp - fp - fn
    sens, spec = _safe_div(tp, tp + fn), _safe_div(tn, tn + fp)
    fpr, fnr = _safe_div(fp, fp + tn), _safe_div(fn, fn + tp)
    prec_c, rec_c, f1_c, support = precision_recall_fscore_support(y_true, y_pred, labels=list(range(K)), zero_division=0)
    w = support / max(support.sum(), 1)
    onehot = np.eye(K)[y_true]
    roc_c, pr_c = np.full(K, np.nan), np.full(K, np.nan)
    for c in range(K):
        pos = onehot[:, c].sum()
        if 0 < pos < n:
            roc_c[c] = roc_auc_score(onehot[:, c], probs[:, c])
            pr_c[c] = average_precision_score(onehot[:, c], probs[:, c])
    ok = ~np.isnan(roc_c)
    wsum = support[ok].sum()
    out = {
        "n": int(n),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(prec_c.mean()), "precision_weighted": float((prec_c * w).sum()),
        "recall_macro": float(rec_c.mean()),     "recall_weighted": float((rec_c * w).sum()),
        "f1_macro": float(f1_c.mean()),          "f1_weighted": float((f1_c * w).sum()),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "roc_auc_macro": float(np.nanmean(roc_c)) if ok.any() else np.nan,
        "roc_auc_weighted": float((roc_c[ok] * support[ok]).sum() / wsum) if ok.any() else np.nan,
        "pr_auc_macro": float(np.nanmean(pr_c)) if ok.any() else np.nan,
        "pr_auc_weighted": float((pr_c[ok] * support[ok]).sum() / wsum) if ok.any() else np.nan,
        "loss": float(-np.mean(np.log(np.clip(probs[np.arange(n), y_true], PROB_EPS, 1.0)))),
        "ece": expected_calibration_error(y_true, probs),
        "confusion_matrix": cm.tolist(),
        "per_class": {CLASS_NAMES[c]: {"precision": float(prec_c[c]), "sensitivity": float(sens[c]),
                                       "specificity": float(spec[c]), "fpr": float(fpr[c]), "fnr": float(fnr[c]),
                                       "f1": float(f1_c[c]), "roc_auc": float(roc_c[c]), "pr_auc": float(pr_c[c]),
                                       "support": int(support[c])} for c in range(K)},
    }
    return out

def flatten_metrics(m, prefix=""):
    """Scalar-only dict (one table row), including per-class values as '<metric>_<class>'."""
    flat = {f"{prefix}{k}": v for k, v in m.items() if k not in ("per_class", "confusion_matrix")}
    for cname, d in m["per_class"].items():
        for k in ("precision", "sensitivity", "specificity", "fpr", "fnr", "f1", "roc_auc", "pr_auc"):
            flat[f"{prefix}{k}_{cname}"] = d[k]
    return flat

def classification_report_text(y_true, probs):
    return classification_report(y_true, np.asarray(probs).argmax(1), labels=list(range(NUM_CLASSES)),
                                 target_names=[CLASS_DISPLAY[i] for i in range(NUM_CLASSES)], digits=4, zero_division=0)

def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z ** 2 / n; c = (p + z ** 2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return float(c - h), float(c + h)

def cluster_bootstrap_ci(y_true, y_pred, groups=None, metrics=("accuracy", "f1_macro", "kappa"),
                         n_boot=1000, seed=SEED, alpha=0.05):
    """Percentile bootstrap CI that resamples near-duplicate GROUPS (not single images), so it is not over-confident."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred); n = len(y_true)
    groups = np.arange(n) if groups is None else np.asarray(groups)
    uniq, inv = np.unique(groups, return_inverse=True)
    members = [np.where(inv == g)[0] for g in range(len(uniq))]
    fns = {"accuracy": accuracy_score, "kappa": cohen_kappa_score,
           "f1_macro": lambda a, b: f1_score(a, b, average="macro", zero_division=0)}
    rng = np.random.RandomState(seed); draws = {m: [] for m in metrics}
    for _ in range(n_boot):
        pick = rng.randint(0, len(uniq), len(uniq))
        idx = np.concatenate([members[g] for g in pick])
        for m in metrics: draws[m].append(fns[m](y_true[idx], y_pred[idx]))
    return {m: (float(np.nanpercentile(v, 100 * alpha / 2)), float(np.nanpercentile(v, 100 * (1 - alpha / 2))))
            for m, v in draws.items()}

# ---------------- SELF-TEST against scikit-learn (synthetic data) ----------------
_rs = np.random.RandomState(0)
y_syn = _rs.choice(4, size=1000, p=[0.30, 0.29, 0.29, 0.12])
_lg = _rs.normal(0, 1, (1000, 4)) + 1.8 * np.eye(4)[y_syn]
p_syn = np.exp(_lg) / np.exp(_lg).sum(1, keepdims=True)
m_syn = compute_metrics(y_syn, p_syn); yp = p_syn.argmax(1)

assert abs(m_syn["accuracy"] - accuracy_score(y_syn, yp)) < 1e-12
for avg in ("macro", "weighted"):
    assert abs(m_syn[f"f1_{avg}"] - f1_score(y_syn, yp, average=avg)) < 1e-12
    assert abs(m_syn[f"precision_{avg}"] - precision_recall_fscore_support(y_syn, yp, average=avg, zero_division=0)[0]) < 1e-12
    assert abs(m_syn[f"recall_{avg}"] - precision_recall_fscore_support(y_syn, yp, average=avg, zero_division=0)[1]) < 1e-12
    assert abs(m_syn[f"roc_auc_{avg}"] - roc_auc_score(y_syn, p_syn, multi_class="ovr", average=avg)) < 1e-9
assert abs(m_syn["kappa"] - cohen_kappa_score(y_syn, yp)) < 1e-12
for c in range(4):                                                      # one-vs-rest binary cross-check
    tn_, fp_, fn_, tp_ = confusion_matrix(y_syn == c, yp == c).ravel()
    pc = m_syn["per_class"][CLASS_NAMES[c]]
    assert abs(pc["specificity"] - tn_ / (tn_ + fp_)) < 1e-12 and abs(pc["sensitivity"] - tp_ / (tp_ + fn_)) < 1e-12
    assert abs(pc["fpr"] - fp_ / (fp_ + tn_)) < 1e-12 and abs(pc["fnr"] - fn_ / (fn_ + tp_)) < 1e-12
    assert abs(pc["pr_auc"] - average_precision_score(y_syn == c, p_syn[:, c])) < 1e-12
perfect = compute_metrics(y_syn, np.eye(4)[y_syn])
assert perfect["accuracy"] == 1.0 and perfect["kappa"] == 1.0 and perfect["roc_auc_macro"] == 1.0 and perfect["ece"] < 1e-9
assert wilson_ci(50, 100)[0] < 0.5 < wilson_ci(50, 100)[1]
ci_syn = cluster_bootstrap_ci(y_syn, yp, groups=np.arange(1000) // 3, n_boot=200)
assert ci_syn["accuracy"][0] <= m_syn["accuracy"] <= ci_syn["accuracy"][1]

print("[OK] metric suite matches scikit-learn (accuracy, P/R/F1 macro+weighted, kappa, OvR ROC-AUC, PR-AUC, sens/spec/FPR/FNR)")
print(f"     synthetic example: acc={m_syn['accuracy']:.3f}  macroF1={m_syn['f1_macro']:.3f}  "
      f"ROC-AUC(macro)={m_syn['roc_auc_macro']:.3f}  kappa={m_syn['kappa']:.3f}  ECE={m_syn['ece']:.3f}  "
      f"accuracy bootstrap CI={tuple(round(v, 3) for v in ci_syn['accuracy'])}")

In [ ]:
# ============================================================
# CELL 21: PLOTTING UTILITIES (HISTORY, CONFUSION, ROC/PR, COMPARISON BARS)
# ============================================================

def _finish(fig, path, show):
    if path is not None:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=150, bbox_inches="tight")
    if show: plt.show()
    else: plt.close(fig)

def plot_history(hist, title, path=None, show=True):
    """hist: DataFrame with columns stage, accuracy, val_accuracy, loss, val_loss (one row per epoch)."""
    h = hist.reset_index(drop=True); ep = np.arange(1, len(h) + 1)
    best = int(h["val_loss"].values.argmin()) + 1
    s2 = np.where(h["stage"].values == 2)[0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    for ax, (tr, va, yl) in zip(axes, [("accuracy", "val_accuracy", "Accuracy"), ("loss", "val_loss", "Loss")]):
        ax.plot(ep, h[tr], "-o", ms=3, lw=1.8, label="Training (augmented, class-weighted)")
        ax.plot(ep, h[va], "-s", ms=3, lw=1.8, label="Validation")
        if len(s2): ax.axvline(s2[0] + 0.5, color="gray", ls=":", lw=1.5, label="Start of fine-tuning (stage 2)")
        ax.axvline(best, color="green", ls="--", lw=1.2, label=f"Best val. loss (epoch {best})")
        ax.set_title(f"{title}: {yl}", fontsize=13); ax.set_xlabel("Epoch", fontsize=12); ax.set_ylabel(yl, fontsize=12)
        ax.legend(fontsize=9)
    plt.tight_layout(); _finish(fig, path, show)

def plot_confusion(cm, title, path=None, show=True):
    cm = np.asarray(cm); rown = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    labels = [CLASS_DISPLAY[i] for i in range(NUM_CLASSES)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0], xticklabels=labels, yticklabels=labels, cbar=False)
    axes[0].set_title(f"{title}: counts", fontsize=13)
    sns.heatmap(rown, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=axes[1], xticklabels=labels, yticklabels=labels)
    axes[1].set_title(f"{title}: row-normalised (recall)", fontsize=13)
    for ax in axes: ax.set_xlabel("Predicted class", fontsize=12); ax.set_ylabel("True class", fontsize=12)
    plt.tight_layout(); _finish(fig, path, show)

def plot_roc_pr(y_true, probs, title, path=None, show=True):
    y_true, probs = np.asarray(y_true), np.asarray(probs); oh = np.eye(NUM_CLASSES)[y_true]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
    for c in range(NUM_CLASSES):
        f, t, _ = roc_curve(oh[:, c], probs[:, c])
        axes[0].plot(f, t, lw=2, label=f"{CLASS_DISPLAY[c]} (AUC={roc_auc_score(oh[:, c], probs[:, c]):.3f})")
        pr, rc, _ = precision_recall_curve(oh[:, c], probs[:, c])
        axes[1].plot(rc, pr, lw=2, label=f"{CLASS_DISPLAY[c]} (AP={average_precision_score(oh[:, c], probs[:, c]):.3f})")
    axes[0].plot([0, 1], [0, 1], "k--", lw=1)
    axes[0].set_title(f"{title}: ROC (one-vs-rest)", fontsize=13)
    axes[0].set_xlabel("False positive rate", fontsize=12); axes[0].set_ylabel("True positive rate", fontsize=12)
    axes[1].set_title(f"{title}: precision-recall (one-vs-rest)", fontsize=13)
    axes[1].set_xlabel("Recall", fontsize=12); axes[1].set_ylabel("Precision", fontsize=12)
    for ax in axes: ax.legend(fontsize=10, loc="best"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    plt.tight_layout(); _finish(fig, path, show)

def plot_bar_compare(table, title, ylabel, path=None, err=None, ylim=None, show=True, figsize=(12, 5.5), xlabel=""):
    """table: DataFrame (index = categories on x-axis, columns = series). err: optional DataFrame of same shape."""
    cats, series = list(table.index), list(table.columns)
    n = len(series); w = 0.8 / n; x = np.arange(len(cats))
    fig, ax = plt.subplots(figsize=figsize)
    for k, s in enumerate(series):
        yerr = None if err is None else err[s].values
        bars = ax.bar(x + (k - (n - 1) / 2) * w, table[s].values, w, yerr=yerr, capsize=3, edgecolor="black", label=str(s))
        for b, v in zip(bars, table[s].values):
            if np.isfinite(v):
                ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{v:.3g}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([str(c) for c in cats], fontsize=11)
    ax.set_title(title, fontsize=14); ax.set_ylabel(ylabel, fontsize=12); ax.set_xlabel(xlabel, fontsize=12)
    if ylim is not None: ax.set_ylim(*ylim)
    ax.legend(fontsize=10); plt.tight_layout(); _finish(fig, path, show)

# ---------------- quick self-test with synthetic data (saved to a temp folder, then deleted) ----------------
_tmp = Path("/kaggle/working/_plot_selftest"); _tmp.mkdir(exist_ok=True)
_h = pd.DataFrame({"stage": [1] * 4 + [2] * 4, "accuracy": np.linspace(.6, .95, 8), "val_accuracy": np.linspace(.55, .9, 8),
                   "loss": np.linspace(1, .2, 8), "val_loss": [1, .8, .6, .5, .45, .4, .42, .44]})
plot_history(_h, "synthetic", _tmp / "h.png", show=False)
plot_confusion(m_syn["confusion_matrix"], "synthetic", _tmp / "c.png", show=False)
plot_roc_pr(y_syn, p_syn, "synthetic", _tmp / "r.png", show=False)
plot_bar_compare(pd.DataFrame({"A": [.9, .8], "B": [.85, .88]}, index=["x", "y"]), "synthetic", "score", _tmp / "b.png", show=False)
assert all((_tmp / f).exists() for f in ["h.png", "c.png", "r.png", "b.png"])
shutil.rmtree(_tmp)
print("[OK] plotting utilities defined and self-tested (history, confusion, ROC/PR, comparison bars)")

In [ ]:
# ============================================================
# CELL 22: MODEL BUILDERS (CUSTOM CNN + 4 PRETRAINED BACKBONES), RELU/GELU HEADS, TWO-STAGE FREEZING
# ============================================================

def _register_serializable(cls):
    try:
        return keras.saving.register_keras_serializable(package="btc")(cls)
    except AttributeError:
        return keras.utils.register_keras_serializable(package="btc")(cls)

@_register_serializable
class InputPreprocess(keras.layers.Layer):
    """Architecture-specific input scaling. Input: float pixels in [0,255].
    No learned or data-dependent state: 'standardize' receives mean/std computed from TRAINING data only (constants)."""
    def __init__(self, mode="builtin", mean=0.0, std=1.0, **kwargs):
        kwargs.setdefault("dtype", "float32")
        super().__init__(**kwargs)
        assert mode in ("caffe", "torch", "standardize", "builtin")
        self.mode, self.mean, self.std = mode, float(mean), float(std)

    def call(self, x):
        x = tf.cast(x, tf.float32)
        if self.mode == "caffe":        # ResNet50 / VGG19: RGB->BGR, subtract ImageNet channel means
            x = tf.reverse(x, axis=[-1]) - tf.constant([103.939, 116.779, 123.68], tf.float32)
        elif self.mode == "torch":      # DenseNet121: scale to [0,1], ImageNet mean/std
            x = (x / 255.0 - tf.constant([0.485, 0.456, 0.406], tf.float32)) / tf.constant([0.229, 0.224, 0.225], tf.float32)
        elif self.mode == "standardize":  # custom CNN: training-set mean/std
            x = (x / 255.0 - self.mean) / self.std
        return x                         # 'builtin': EfficientNet rescales/normalises inside the backbone

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        cfg = super().get_config(); cfg.update({"mode": self.mode, "mean": self.mean, "std": self.std}); return cfg

ARCH_SPECS = {
    "custom_cnn":     {"kind": "scratch",    "preprocess": "standardize", "fn": None,
                       "display": "Custom 4-block CNN", "in_paper": True},
    "resnet50":       {"kind": "pretrained", "preprocess": "caffe",   "fn": keras.applications.ResNet50,
                       "display": "ResNet50", "in_paper": True},
    "vgg19":          {"kind": "pretrained", "preprocess": "caffe",   "fn": keras.applications.VGG19,
                       "display": "VGG19", "in_paper": True},
    "densenet121":    {"kind": "pretrained", "preprocess": "torch",   "fn": keras.applications.DenseNet121,
                       "display": "DenseNet121", "in_paper": True},
    "efficientnetb0": {"kind": "pretrained", "preprocess": "builtin", "fn": keras.applications.EfficientNetB0,
                       "display": "EfficientNetB0", "in_paper": False},
}
assert set(ARCH_SPECS) == set(ARCHITECTURES)

HP_PRETRAINED = dict(batch_size=32, dropout=0.3, head_units=256, weight_decay=0.01,
                     lr_head=1e-3, lr_ft=3e-5, unfreeze_frac=0.30,
                     epochs_head=10, epochs_ft=30, patience_head=4, patience=6,
                     rlrop_factor=0.5, rlrop_patience=3, min_lr=1e-7, aug=DEFAULT_AUG, use_class_weights=True)
HP_CUSTOM = dict(batch_size=32, dropout=0.5, conv_dropout=0.1, head_units=128, weight_decay=0.01,
                 lr=1e-3, epochs=60, patience=10, rlrop_factor=0.5, rlrop_patience=4, min_lr=1e-7,
                 aug=DEFAULT_AUG, use_class_weights=True)

def get_hp(arch, **overrides):
    base = dict(HP_CUSTOM if arch == "custom_cnn" else HP_PRETRAINED)
    bad = set(overrides) - set(base)
    if bad: raise KeyError(f"Unknown hyperparameter(s) for {arch}: {sorted(bad)}")
    base.update(overrides); return base

def reset_session(seed=SEED):
    keras.backend.clear_session(); gc.collect()
    keras.mixed_precision.set_global_policy("mixed_float16" if MIXED_PRECISION else "float32")
    set_global_seed(seed)

def build_model(arch, activation, hp, pixel_stats=None, weights="imagenet"):
    """activation in {'relu','gelu'}. For pretrained models only the NEW head activation changes; the backbone keeps
    its native activations so pretrained weights stay valid. For the custom CNN the activation applies everywhere."""
    assert arch in ARCH_SPECS and activation in ("relu", "gelu")
    spec = ARCH_SPECS[arch]
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image_0_255")
    if spec["kind"] == "scratch":
        assert pixel_stats is not None, "custom CNN needs training-set (mean, std)"
        x = InputPreprocess("standardize", pixel_stats[0], pixel_stats[1], name="preprocess")(inp)
        for i, f in enumerate([32, 64, 128, 256]):
            x = keras.layers.Conv2D(f, 3, padding="same", kernel_initializer="he_normal", name=f"conv{i+1}")(x)
            x = keras.layers.Activation(activation, name=f"act{i+1}")(x)
            x = keras.layers.BatchNormalization(name=f"bn{i+1}")(x)
            x = keras.layers.MaxPooling2D(2, name=f"pool{i+1}")(x)
            if i < 2 and hp["conv_dropout"] > 0:
                x = keras.layers.Dropout(hp["conv_dropout"], name=f"drop{i+1}")(x)
        x = keras.layers.Flatten(name="flatten")(x)
        x = keras.layers.Dense(hp["head_units"], kernel_initializer="he_normal", name="head_dense")(x)
        x = keras.layers.Activation(activation, name="head_act")(x)
        x = keras.layers.Dropout(hp["dropout"], name="head_drop")(x)
    else:
        x = InputPreprocess(spec["preprocess"], name="preprocess")(inp)
        base = spec["fn"](include_top=False, weights=weights, input_shape=(IMG_SIZE, IMG_SIZE, 3))
        x = base(x, training=False)          # backbone BatchNorm always in inference mode
        x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
        x = keras.layers.Dropout(hp["dropout"], name="head_drop1")(x)
        x = keras.layers.Dense(hp["head_units"], kernel_initializer="he_normal", name="head_dense")(x)
        x = keras.layers.Activation(activation, name="head_act")(x)
        x = keras.layers.Dropout(hp["dropout"], name="head_drop2")(x)
    out = keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="predictions")(x)
    return keras.Model(inp, out, name=f"{arch}_{activation}")

def get_backbone(model):
    for l in model.layers:
        if isinstance(l, keras.Model): return l
    return None

def set_stage(model, stage, unfreeze_frac):
    """stage 1: backbone fully frozen. stage 2: top `unfreeze_frac` of backbone layers trainable, BatchNorm stays frozen.
    Returns (n_backbone_layers_unfrozen_with_weights, n_backbone_layers). Custom CNN (no backbone): no-op."""
    bb = get_backbone(model)
    if bb is None: return 0, 0
    layers = bb.layers
    if stage == 1:
        bb.trainable = False
        return 0, len(layers)
    bb.trainable = True
    cutoff = len(layers) - int(round(len(layers) * unfreeze_frac)); n_unf = 0
    for i, l in enumerate(layers):
        want = (i >= cutoff) and not isinstance(l, keras.layers.BatchNormalization)
        l.trainable = want
        n_unf += int(want and len(l.weights) > 0)
    return n_unf, len(layers)

def count_params(model):
    total = int(model.count_params())
    trainable = int(sum(int(np.prod(w.shape)) for w in model.trainable_weights))
    return {"total": total, "trainable": trainable, "non_trainable": total - trainable}

def compile_model(model, lr, weight_decay):
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay),
                  loss=keras.losses.CategoricalCrossentropy(),
                  metrics=[keras.metrics.CategoricalAccuracy(name="accuracy")],
                  jit_compile=False)

print("Architectures:", {a: (s["display"], s["preprocess"], "from paper" if s["in_paper"] else "added by specification")
                         for a, s in ARCH_SPECS.items()})
print("Pretrained defaults:", HP_PRETRAINED)
print("Custom CNN defaults:", HP_CUSTOM)

In [ ]:
# ============================================================
# CELL 23: BUILD + VERIFY ALL 10 VARIANTS, PARAMETER TABLE, PREPROCESSING CHECKS
# ============================================================

px_stats_80 = compute_pixel_stats(get_split("80_20")["fit_idx"])
probe = X_all[:4].astype(np.float32)
rows, effnet_has_rescaling = [], None

for arch in ARCHITECTURES:
    for act in ACTIVATIONS:
        reset_session()
        hp = get_hp(arch); t0 = time.time()
        try:
            model = build_model(arch, act, hp, pixel_stats=px_stats_80)
        except Exception as e:
            raise RuntimeError(f"Could not build {arch}/{act}. If this is a weight-download error, turn Internet ON "
                               f"(Settings -> Internet). Original error: {e!r}") from e
        build_s = time.time() - t0
        frac = hp.get("unfreeze_frac", 0.0)
        set_stage(model, 1, frac); p1 = count_params(model)
        n_unf, n_bb = set_stage(model, 2, frac); p2 = count_params(model)
        out = model.predict(probe, verbose=0)
        assert out.shape == (4, NUM_CLASSES) and out.dtype == np.float32 and np.allclose(out.sum(1), 1.0, atol=1e-3), \
            f"{arch}/{act}: invalid output {out.shape} {out.dtype}"
        if arch == "efficientnetb0" and act == "relu":
            effnet_has_rescaling = any(isinstance(l, keras.layers.Rescaling) for l in get_backbone(model).layers)
        rows.append({"arch": arch, "activation": act, "preprocess": ARCH_SPECS[arch]["preprocess"],
                     "total_params": p1["total"], "trainable_stage1": p1["trainable"], "trainable_stage2": p2["trainable"],
                     "non_trainable_stage1": p1["non_trainable"], "non_trainable_stage2": p2["non_trainable"],
                     "backbone_layers": n_bb, "backbone_layers_unfrozen_stage2": n_unf, "build_s": round(build_s, 1)})
        print(f"[built] {arch:15s} {act:5s} total={p1['total']/1e6:6.2f}M | trainable stage1={p1['trainable']/1e6:6.2f}M "
              f"stage2={p2['trainable']/1e6:6.2f}M | {build_s:.1f}s")
        del model

ptab = pd.DataFrame(rows)
# ReLU and GELU variants must differ only by activation, never by parameter count
for arch in ARCHITECTURES:
    d = ptab[ptab.arch == arch]
    assert d["total_params"].nunique() == 1 and d["trainable_stage2"].nunique() == 1, f"{arch}: ReLU/GELU parameter mismatch"
print("\n[OK] every variant outputs valid float32 softmax probabilities")
print("[OK] ReLU and GELU variants have identical parameter counts for every architecture")

# ---- preprocessing layer vs Keras reference implementations ----
xs = X_all[:8].astype(np.float32)
for mode, ref in [("caffe", keras.applications.resnet50.preprocess_input),
                  ("torch", keras.applications.densenet.preprocess_input)]:
    mine = InputPreprocess(mode)(tf.constant(xs)).numpy()
    assert np.allclose(mine, ref(xs.copy()), atol=1e-3), f"InputPreprocess('{mode}') differs from Keras preprocess_input"
print("[OK] InputPreprocess('caffe') and ('torch') match Keras preprocess_input")
assert effnet_has_rescaling, "EfficientNetB0 backbone has no built-in Rescaling layer - input scaling would be wrong"
print("[OK] EfficientNetB0 contains its built-in rescaling, so it correctly receives raw 0-255 pixels")

reset_session()
print("\nPARAMETER TABLE (ReLU rows shown; GELU variants are identical):")
show = ptab[ptab.activation == "relu"].drop(columns=["activation", "build_s"]).copy()
for c in ["total_params", "trainable_stage1", "trainable_stage2", "non_trainable_stage1", "non_trainable_stage2"]:
    show[c] = show[c].map(lambda v: f"{v:,}")
print(show.to_string(index=False))
print("\nNOTE: custom_cnn is single-stage (all weights trainable from the start); stage-2 numbers equal stage-1 there.")
print("NOTE: pretrained backbones keep native activations; only the new Dense(256) head differs between ReLU and GELU.")
ptab.to_csv(DIRS["results"] / "model_parameter_table.csv", index=False)

pt = ptab[ptab.activation == "relu"].set_index("arch")
tab = pd.DataFrame({"Total": pt["total_params"] / 1e6, "Trainable (stage 1)": pt["trainable_stage1"] / 1e6,
                    "Trainable (stage 2)": pt["trainable_stage2"] / 1e6})
tab.index = [ARCH_SPECS[a]["display"] for a in tab.index]
plot_bar_compare(tab, "Model parameter comparison (millions)", "Parameters (millions)",
                 DIRS["figures"] / "10_parameter_comparison.png", xlabel="Architecture")

In [ ]:
# ============================================================
# CELL 24: TRAINING + EVALUATION ENGINE (REGISTRY-AWARE, TEST-LOCKED)
# ============================================================

TEST_UNLOCKED = False        # only the Stage L test-evaluation cell may set this to True

def run_paths(run_id, root=None):
    root = Path(root) if root else BASE
    d = {"runs": root / "results" / "runs", "probs": root / "results" / "probs",
         "hist": root / "histories", "ckpt": root / "checkpoints"}
    for p in d.values(): p.mkdir(parents=True, exist_ok=True)
    safe = run_id.replace("|", "__").replace("/", "_")
    return {"result": d["runs"] / f"{safe}.json", "probs": d["probs"] / f"{safe}.npz",
            "history": d["hist"] / f"{safe}.csv", "ckpt": d["ckpt"] / f"{safe}.weights.h5"}

def make_run_id(kind, split, arch, act, fold=None, trial=None):
    parts = [kind, split, arch, act]
    if trial is not None: parts.append(f"trial{trial}")
    if fold is not None: parts.append(f"fold{fold}")
    return "__".join(parts)

def test_indices_for(split_name):
    return get_split("70_30" if split_name == "tuning" else split_name)["test_idx"]   # core pool never touches any test set

def assert_no_test_overlap(split_name, idx, what):
    ov = np.intersect1d(np.asarray(idx), test_indices_for(split_name))
    if len(ov):
        raise RuntimeError(f"LEAKAGE GUARD: '{what}' for split '{split_name}' contains {len(ov)} TEST images.")

class EpochTimer(keras.callbacks.Callback):
    def __init__(self): super().__init__(); self.times = []
    def on_epoch_begin(self, epoch, logs=None): self._t = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.perf_counter() - self._t)

def predict_probs(model, indices, batch_size=64):
    return np.asarray(model.predict(make_dataset(indices, batch_size, training=False), verbose=0), dtype=np.float32)

def evaluate_subset(model, split_name, name, idx):
    idx = np.asarray(idx)
    if name == "test":
        if not TEST_UNLOCKED:
            raise RuntimeError("TEST SET IS LOCKED. It is released only by the Stage L cell after model selection is saved.")
    else:
        assert_no_test_overlap(split_name, idx, name)
    probs = predict_probs(model, idx)
    return probs, compute_metrics(y_all[idx], probs)

def measure_inference(model, sample_idx, n_single=50, bs=32, reps=5):
    """GPU inference timing on images that are NOT used for any decision (labels are not touched)."""
    x1 = tf.constant(X_all[np.asarray(sample_idx)[:1]].astype(np.float32))
    xb = tf.constant(X_all[np.asarray(sample_idx)[:bs]].astype(np.float32))
    @tf.function
    def f(x): return model(x, training=False)
    for _ in range(5): f(x1).numpy()
    single = []
    for _ in range(n_single):
        t0 = time.perf_counter(); f(x1).numpy(); single.append(time.perf_counter() - t0)
    f(xb).numpy(); batch = []
    for _ in range(reps):
        t0 = time.perf_counter(); f(xb).numpy(); batch.append(time.perf_counter() - t0)
    return {"latency_ms_bs1": float(np.median(single) * 1000), "throughput_img_s_bs32": float(len(xb) / np.median(batch))}

def train_and_evaluate(run_id, arch, activation, split_name, fit_idx, val_idx, hp_overrides=None, extra_eval=None,
                       seed=SEED, register=True, out_root=None, verbose=0, eval_train=True, weights="imagenet"):
    """Two-stage (pretrained) or single-stage (custom CNN) training with validation-based early stopping.
    Returns the result dict (also saved as JSON). Skips runs already in the registry."""
    t_run0 = time.perf_counter()
    paths = run_paths(run_id, out_root)
    if register and run_is_done(run_id) and paths["result"].exists():
        print(f"[skip] {run_id} (already completed)"); return load_json(paths["result"])
    fit_idx, val_idx = np.asarray(fit_idx), np.asarray(val_idx)
    extra_eval = {k: np.asarray(v) for k, v in (extra_eval or {}).items()}
    assert_no_test_overlap(split_name, fit_idx, "fit"); assert_no_test_overlap(split_name, val_idx, "val")
    assert len(np.intersect1d(fit_idx, val_idx)) == 0, "fit and val overlap"
    for nm, ix in extra_eval.items():
        if nm != "test": assert_no_test_overlap(split_name, ix, nm)
        assert len(np.intersect1d(ix, fit_idx)) == 0 and len(np.intersect1d(ix, val_idx)) == 0, f"'{nm}' overlaps training data"

    hp = get_hp(arch, **(hp_overrides or {})); scratch = ARCH_SPECS[arch]["kind"] == "scratch"
    reset_session(seed)
    cw = class_weights_from_labels(y_all[fit_idx])
    px = compute_pixel_stats(fit_idx) if scratch else None
    ds_tr = make_dataset(fit_idx, hp["batch_size"], training=True, aug_preset=hp["aug"],
                         class_weights=cw if hp["use_class_weights"] else None, seed=seed)
    ds_va = make_dataset(val_idx, hp["batch_size"], training=False)
    model = build_model(arch, activation, hp, pixel_stats=px, weights=weights)

    stages = [(1, hp["epochs"], hp["lr"], hp["patience"])] if scratch else \
             [(1, hp["epochs_head"], hp["lr_head"], hp["patience_head"]), (2, hp["epochs_ft"], hp["lr_ft"], hp["patience"])]
    ckpt = str(paths["ckpt"]); best_val = np.inf; hist_parts, epoch_times, params_by_stage = [], {}, {}
    t_fit0 = time.perf_counter()
    for stage, epochs, lr, pat in stages:
        set_stage(model, stage, hp.get("unfreeze_frac", 0.0))
        params_by_stage[stage] = count_params(model)
        compile_model(model, lr, hp["weight_decay"])
        timer = EpochTimer()
        cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=pat, restore_best_weights=False),
               keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=hp["rlrop_factor"],
                                                 patience=hp["rlrop_patience"], min_lr=hp["min_lr"]),
               keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, save_weights_only=True,
                                               initial_value_threshold=(best_val if np.isfinite(best_val) else None)),
               timer]
        h = model.fit(ds_tr, validation_data=ds_va, epochs=epochs, callbacks=cbs, verbose=verbose)
        hdf = pd.DataFrame(h.history); hdf["stage"] = stage; hist_parts.append(hdf)
        epoch_times[f"stage{stage}"] = [float(t) for t in timer.times]
        best_val = min(best_val, float(hdf["val_loss"].min()))
        model.load_weights(ckpt)                                  # explicit: global best-so-far weights
    train_time = time.perf_counter() - t_fit0

    hist = pd.concat(hist_parts, ignore_index=True); hist.insert(0, "epoch", np.arange(1, len(hist) + 1))
    hist.to_csv(paths["history"], index=False)
    best_ep = int(hist["val_loss"].values.argmin()) + 1
    conv_ep = int(np.argmax(hist["val_accuracy"].values >= hist["val_accuracy"].max() - 0.01)) + 1

    val_probs, val_m = evaluate_subset(model, split_name, "val", val_idx)
    out_probs, eval_metrics = {"val_idx": val_idx, "val_probs": val_probs}, {"val": val_m}
    for nm, ix in extra_eval.items():
        pr, mm = evaluate_subset(model, split_name, nm, ix)
        out_probs[f"{nm}_idx"], out_probs[f"{nm}_probs"], eval_metrics[nm] = ix, pr, mm
    train_m = evaluate_subset(model, split_name, "fit", fit_idx)[1] if eval_train else None
    inf = measure_inference(model, fit_idx)
    np.savez_compressed(paths["probs"], **out_probs)

    n1 = len(epoch_times["stage1"]); n2 = len(epoch_times.get("stage2", []))
    summary = {"val_loss": val_m["loss"], "val_accuracy": val_m["accuracy"], "val_f1_macro": val_m["f1_macro"],
               "val_roc_auc_macro": val_m["roc_auc_macro"], "val_kappa": val_m["kappa"],
               "train_loss": train_m["loss"] if train_m else np.nan, "train_accuracy": train_m["accuracy"] if train_m else np.nan,
               "train_f1_macro": train_m["f1_macro"] if train_m else np.nan,
               "train_time_s": train_time, "best_epoch": best_ep, "conv_epoch": conv_ep, "epochs_run": n1 + n2,
               "params_total": params_by_stage[1]["total"], **inf}
    summary["gap_accuracy"] = summary["train_accuracy"] - summary["val_accuracy"]
    summary["gap_loss"] = summary["val_loss"] - summary["train_loss"]
    for nm, mm in eval_metrics.items():
        if nm != "val": summary[f"{nm}_accuracy"], summary[f"{nm}_f1_macro"], summary[f"{nm}_loss"] = mm["accuracy"], mm["f1_macro"], mm["loss"]

    res = {"run_id": run_id, "arch": arch, "activation": activation, "split": split_name, "seed": seed, "hp": hp,
           "n_fit": int(len(fit_idx)), "n_val": int(len(val_idx)), "class_weights": cw,
           "params_by_stage": {str(k): v for k, v in params_by_stage.items()},
           "epochs_stage1": n1, "epochs_stage2": n2, "best_epoch_stage": int(hist["stage"].iloc[best_ep - 1]),
           "epoch_times_s": epoch_times, "metrics": eval_metrics, "train_clean_metrics": train_m, "summary": summary,
           "paths": {k: str(v) for k, v in paths.items()}, "run_wall_s": time.perf_counter() - t_run0}
    save_json(res, paths["result"])
    if register:
        register_run(run_id, {"arch": arch, "activation": activation, "split": split_name,
                              "val_f1_macro": val_m["f1_macro"], "val_loss": val_m["loss"], "result_path": str(paths["result"])})
    print(f"[done] {run_id} | val acc {val_m['accuracy']:.4f} F1 {val_m['f1_macro']:.4f} loss {val_m['loss']:.4f} | "
          f"train acc {summary['train_accuracy']:.4f} | epochs {n1}+{n2} | {train_time/60:.1f} min")
    del model, ds_tr, ds_va; reset_session(seed)
    return res

def load_result(run_id, out_root=None):
    return load_json(run_paths(run_id, out_root)["result"])

def runs_table(prefix="", root=None):
    """DataFrame of all completed runs (from the registry) whose id starts with `prefix`."""
    rows = []
    for rid in list_runs(prefix):
        p = run_paths(rid, root)["result"]
        if p.exists():
            r = load_json(p); rows.append({"run_id": rid, "arch": r["arch"], "activation": r["activation"], "split": r["split"], **r["summary"]})
    return pd.DataFrame(rows)

print("Training engine ready. TEST_UNLOCKED =", TEST_UNLOCKED)
print("Run-id convention example:", make_run_id("init", "70_30", "densenet121", "relu"),
      "|", make_run_id("cv", "80_20", "efficientnetb0", "relu", fold=3))

In [ ]:
# ============================================================
# CELL 25: SMOKE TEST, MEASURED THROUGHPUT, RECALIBRATED COMPUTE PLAN
# ============================================================
from sklearn.model_selection import train_test_split

SMOKE_DIR = Path("/kaggle/working/_smoke")
tu = get_tuning_split()
sub_fit, _ = train_test_split(tu["fit_idx"], train_size=640, stratify=y_all[tu["fit_idx"]], random_state=SEED)
sub_val, _ = train_test_split(tu["val_idx"], train_size=160, stratify=y_all[tu["val_idx"]], random_state=SEED)
assert len(np.intersect1d(sub_fit, sub_val)) == 0
print(f"Smoke subsets: fit={len(sub_fit)}, val={len(sub_val)} (from the tuning pool; the test sets are never touched)")

# ---- raw data-pipeline throughput (augmentation included) ----
cw_s = class_weights_from_labels(y_all[sub_fit])
it = iter(make_dataset(sub_fit, 32, training=True, aug_preset=DEFAULT_AUG, class_weights=cw_s).repeat())
next(it); t0 = time.perf_counter()
for _ in range(15): next(it)
pipe_img_s = 15 * 32 / (time.perf_counter() - t0)
print(f"tf.data pipeline with augmentation: {pipe_img_s:,.0f} images/s (CPU side)")

# ---- smoke runs ----
smoke_res = {}
for arch in ARCHITECTURES:
    ov = {"epochs": 3, "patience": 5} if arch == "custom_cnn" else {"epochs_head": 2, "epochs_ft": 2, "patience_head": 5, "patience": 5}
    smoke_res[arch] = train_and_evaluate(make_run_id("smoke", "tuning", arch, "relu"), arch, "relu", "tuning",
                                         sub_fit, sub_val, hp_overrides=ov, register=False, out_root=SMOKE_DIR, verbose=0)

# ---- steady-state timing (last epoch of each stage; includes the validation pass, so slightly conservative) ----
timing, rows = {}, []
for arch, r in smoke_res.items():
    et, n = r["epoch_times_s"], r["n_fit"]
    s1 = et["stage1"][-1] / n; s2 = et["stage2"][-1] / n if "stage2" in et else np.nan
    overhead = r["run_wall_s"] - sum(sum(v) for v in et.values())
    timing[arch] = {"s1": s1, "s2": s2, "overhead": overhead}
    rows.append({"arch": arch, "stage1_img_per_s": round(1 / s1), "stage2_img_per_s": (round(1 / s2) if np.isfinite(s2) else None),
                 "fixed_overhead_s": round(overhead), "latency_ms_bs1": round(r["summary"]["latency_ms_bs1"], 1),
                 "infer_img_per_s": round(r["summary"]["throughput_img_s_bs32"]), "params_M": round(r["summary"]["params_total"] / 1e6, 2)})
print("\nMeasured speed (steady state):")
print(pd.DataFrame(rows).to_string(index=False))
slowest_model_rate = min(1 / timing[a]["s1"] for a in ARCHITECTURES)
fastest_model_rate = max(1 / timing[a]["s1"] for a in ARCHITECTURES)
print(f"\nData pipeline {pipe_img_s:,.0f} img/s vs fastest model stage-1 rate {fastest_model_rate:,.0f} img/s -> "
      + ("pipeline is probably the BOTTLENECK for the fastest models." if pipe_img_s < fastest_model_rate * 1.2
         else "pipeline is not the bottleneck."))

# ---- recalibrated compute plan (assumed realised epochs are stated explicitly) ----
EXP_EPOCHS  = {"head": 8, "ft": 18, "custom": 35}      # ASSUMPTION: typical epochs actually run before early stopping
TUNE_EPOCHS = {"head": 4, "ft": 6,  "custom": 12}      # ASSUMPTION: short tuning trials

def est_run_s(arch, n_fit, ep):
    t = timing[arch]
    core = ep["custom"] * t["s1"] * n_fit if arch == "custom_cnn" else (ep["head"] * t["s1"] + ep["ft"] * t["s2"]) * n_fit
    return core + t["overhead"]

def cv_cost(arch, split):
    return sum(est_run_s(arch, len(f["fit_idx"]), EXP_EPOCHS) for f in get_cv_folds(split))

init_s = sum(est_run_s(a, len(get_split(sp)["fit_idx"]), EXP_EPOCHS) for sp in SPLIT_CONFIGS for a in ARCHITECTURES for _ in ACTIVATIONS)
tune_s = sum(6 * est_run_s(a, len(tu["fit_idx"]), TUNE_EPOCHS) for a in ARCHITECTURES)
mand_s = sum(cv_cost(a, sp) for sp in SPLIT_CONFIGS for a in MANDATORY_CV_ARCHS)
others = [a for a in ARCHITECTURES if a not in MANDATORY_CV_ARCHS]
extra_worst_s = sum(sorted([cv_cost(a, sp) for a in others], reverse=True)[:2][0] + sorted([cv_cost(a, sp) for a in others], reverse=True)[:2][1]
                    for sp in SPLIT_CONFIGS)
final_s = 2 * max(est_run_s(a, len(get_split("90_10")["train_idx"]), EXP_EPOCHS) for a in ARCHITECTURES)
n_cv = sum(len(get_cv_folds(sp)) for sp in SPLIT_CONFIGS)

plan = pd.DataFrame([
    {"stage": "F  Hyperparameter search (5 archs x 6 short trials)", "runs": 30, "est_hours": tune_s / 3600},
    {"stage": "G  Initial comparison (5 archs x ReLU/GELU x 3 splits)", "runs": 30, "est_hours": init_s / 3600},
    {"stage": "I  MANDATORY CV: EffNetB0 + DenseNet121 (1 activation)", "runs": 2 * n_cv, "est_hours": mand_s / 3600},
    {"stage": "N  Final retrain (upper bound, 2 runs)", "runs": 2, "est_hours": final_s / 3600},
    {"stage": "(optional) second activation for the mandatory CV", "runs": 2 * n_cv, "est_hours": mand_s / 3600},
    {"stage": "(worst case) J  best-two extra CV for other archs", "runs": 2 * n_cv, "est_hours": extra_worst_s / 3600},
])
plan["est_hours"] = plan["est_hours"].round(2)
print("\nRECALIBRATED PLAN (GPU hours, one T4; assumptions: head<=%d / fine-tune<=%d / custom<=%d realised epochs):"
      % (EXP_EPOCHS["head"], EXP_EPOCHS["ft"], EXP_EPOCHS["custom"]))
print(plan.to_string(index=False))
core_h = plan["est_hours"].iloc[:4].sum()
print(f"\nCore (mandatory) total: ~{core_h:.1f} h | with optional second-activation CV: ~{core_h + plan['est_hours'].iloc[4]:.1f} h | "
      f"plus worst-case extra CV: ~{plan['est_hours'].sum():.1f} h")
print("Check your weekly GPU quota on your Kaggle account page; the plan is organised in resumable stages either way.")
plan.to_csv(DIRS["results"] / "compute_plan_recalibrated.csv", index=False)

# ---- visual check of one real history plot (meaningless accuracy; checks the plotting path and stage marker) ----
hist_s = pd.read_csv(smoke_res["densenet121"]["paths"]["history"])
plot_history(hist_s, "SMOKE TEST densenet121 (4 epochs, 640 images: not a result)", path=None, show=True)
print("\nSmoke outputs are in", SMOKE_DIR, "- safe to delete later; they are not part of the project folder.")

In [ ]:
# ============================================================
# CELL 26: GPU-SIDE AUGMENTATION LAYER + LEANER DATA PIPELINE (OVERRIDES make_dataset AND build_model)
# ============================================================
assert hasattr(keras.ops, "image") and hasattr(keras.ops.image, "affine_transform"), \
    "This cell needs Keras 3 (keras.ops.image.affine_transform). Check the versions printed in CELL 1."

def make_affine_transforms(theta, scale, tx, ty, h=IMG_SIZE, w=IMG_SIZE):
    """theta (radians), scale (zoom factor; >1 = zoom in), tx/ty (pixels): float32 tensors of shape (n,).
    Returns (n,8) projective transforms mapping each OUTPUT pixel (x=column, y=row) to its INPUT pixel,
    about the image centre. Content therefore moves by (+tx, +ty), is scaled by `scale` and rotated by theta."""
    cx, cy = (w - 1) / 2.0, (h - 1) / 2.0
    c, s = tf.cos(theta), tf.sin(theta)
    inv = 1.0 / scale
    a00, a01, a10, a11 = inv * c, inv * s, -inv * s, inv * c
    a2 = cx - (a00 * (cx + tx) + a01 * (cy + ty))
    b2 = cy - (a10 * (cx + tx) + a11 * (cy + ty))
    zeros = tf.zeros_like(theta)
    return tf.stack([a00, a01, a2, a10, a11, b2, zeros, zeros], axis=1)

def apply_affine(x, transforms):
    return keras.ops.image.affine_transform(x, transforms, interpolation="bilinear", fill_mode="constant",
                                            fill_value=0.0, data_format="channels_last")

def _uniform(n, lo, hi):
    return lo + (hi - lo) * tf.random.uniform(tf.stack([n]), 0.0, 1.0)

@_register_serializable
class RandomAffineAugment(keras.layers.Layer):
    """Conservative MRI augmentation on the GPU: ONE fused resampling (rotation + zoom + translation, black fill),
    then multiplicative brightness and contrast (+ optional horizontal flip). Active ONLY when training=True;
    identity otherwise. Has no weights. Input/output: float pixels in [0,255]."""
    def __init__(self, rotation_deg=10.0, zoom=0.10, translate=0.05, brightness=0.10, contrast=0.10, hflip=False, **kwargs):
        kwargs.setdefault("dtype", "float32")
        super().__init__(**kwargs)
        self.rotation_deg, self.zoom, self.translate = float(rotation_deg), float(zoom), float(translate)
        self.brightness, self.contrast, self.hflip = float(brightness), float(contrast), bool(hflip)

    def call(self, x, training=None):
        if not training:
            return x
        x = tf.cast(x, tf.float32)
        n = tf.shape(x)[0]
        h, w = int(x.shape[1] or IMG_SIZE), int(x.shape[2] or IMG_SIZE)
        theta = _uniform(n, -self.rotation_deg, self.rotation_deg) * (np.pi / 180.0)
        scale = _uniform(n, 1.0 - self.zoom, 1.0 + self.zoom)
        tx = _uniform(n, -self.translate, self.translate) * float(w)
        ty = _uniform(n, -self.translate, self.translate) * float(h)
        x = apply_affine(x, make_affine_transforms(theta, scale, tx, ty, h, w))
        if self.hflip:
            flip = tf.random.uniform(tf.stack([n, 1, 1, 1])) < 0.5
            x = tf.where(flip, tf.reverse(x, axis=[2]), x)
        x = x * _uniform(n, 1.0 - self.brightness, 1.0 + self.brightness)[:, None, None, None]
        m = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
        x = (x - m) * _uniform(n, 1.0 - self.contrast, 1.0 + self.contrast)[:, None, None, None] + m
        return tf.clip_by_value(x, 0.0, 255.0)

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"rotation_deg": self.rotation_deg, "zoom": self.zoom, "translate": self.translate,
                    "brightness": self.brightness, "contrast": self.contrast, "hflip": self.hflip})
        return cfg

# ---------------- OVERRIDE: tf.data pipeline without CPU augmentation ----------------
def make_dataset(indices, batch_size=DEFAULT_BATCH, training=False, aug_preset="none",
                 class_weights=None, seed=SEED):
    """Overrides CELL 17. Augmentation is now applied INSIDE the model (RandomAffineAugment, training=True only),
    so this pipeline only shuffles, batches, casts and (for training) attaches per-sample class weights.
    The leakage guard is kept: asking for augmentation on non-training data still raises an error."""
    if aug_preset not in AUG_PRESETS:
        raise KeyError(f"Unknown augmentation preset '{aug_preset}'")
    if (not training) and aug_preset != "none":
        raise ValueError("Augmentation requested for a non-training dataset - this is forbidden (leakage guard).")
    indices = np.asarray(indices)
    ds = tf.data.Dataset.from_tensor_slices((X_all[indices], y_all[indices]))
    if training:
        ds = ds.shuffle(len(indices), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    wt = tf.constant([class_weights[i] for i in range(NUM_CLASSES)], tf.float32) if (training and class_weights) else None

    def finish(x, y):
        x = tf.cast(x, tf.float32)
        y1 = tf.one_hot(y, NUM_CLASSES)
        return (x, y1, tf.gather(wt, y)) if wt is not None else (x, y1)

    return ds.map(finish, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# ---------------- OVERRIDE: model builder with the augmentation layer right after the input ----------------
def _aug_layer_from_hp(hp):
    cfg = AUG_PRESETS[hp.get("aug", "none")]
    return None if cfg is None else RandomAffineAugment(name="augment", **cfg)

def build_model(arch, activation, hp, pixel_stats=None, weights="imagenet"):
    """Overrides CELL 22. Identical architecture, plus an optional 'augment' layer (no weights, training-only)."""
    assert arch in ARCH_SPECS and activation in ("relu", "gelu")
    spec = ARCH_SPECS[arch]
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image_0_255")
    x = inp
    aug = _aug_layer_from_hp(hp)
    if aug is not None:
        x = aug(x)
    if spec["kind"] == "scratch":
        assert pixel_stats is not None, "custom CNN needs training-set (mean, std)"
        x = InputPreprocess("standardize", pixel_stats[0], pixel_stats[1], name="preprocess")(x)
        for i, f in enumerate([32, 64, 128, 256]):
            x = keras.layers.Conv2D(f, 3, padding="same", kernel_initializer="he_normal", name=f"conv{i+1}")(x)
            x = keras.layers.Activation(activation, name=f"act{i+1}")(x)
            x = keras.layers.BatchNormalization(name=f"bn{i+1}")(x)
            x = keras.layers.MaxPooling2D(2, name=f"pool{i+1}")(x)
            if i < 2 and hp["conv_dropout"] > 0:
                x = keras.layers.Dropout(hp["conv_dropout"], name=f"drop{i+1}")(x)
        x = keras.layers.Flatten(name="flatten")(x)
        x = keras.layers.Dense(hp["head_units"], kernel_initializer="he_normal", name="head_dense")(x)
        x = keras.layers.Activation(activation, name="head_act")(x)
        x = keras.layers.Dropout(hp["dropout"], name="head_drop")(x)
    else:
        x = InputPreprocess(spec["preprocess"], name="preprocess")(x)
        base = spec["fn"](include_top=False, weights=weights, input_shape=(IMG_SIZE, IMG_SIZE, 3))
        x = base(x, training=False)
        x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
        x = keras.layers.Dropout(hp["dropout"], name="head_drop1")(x)
        x = keras.layers.Dense(hp["head_units"], kernel_initializer="he_normal", name="head_dense")(x)
        x = keras.layers.Activation(activation, name="head_act")(x)
        x = keras.layers.Dropout(hp["dropout"], name="head_drop2")(x)
    out = keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="predictions")(x)
    return keras.Model(inp, out, name=f"{arch}_{activation}")

# ---------------- SELF-TESTS ----------------
xs = tf.constant(X_all[:8].astype(np.float32)); n8 = 8
z, one = tf.zeros((n8,)), tf.ones((n8,))

# 1. identity outside training
lay = RandomAffineAugment(**AUG_PRESETS[DEFAULT_AUG])
assert np.array_equal(lay(xs, training=False).numpy(), xs.numpy()); assert np.array_equal(lay(xs).numpy(), xs.numpy())
print("[OK] layer is the exact identity when training=False / None (validation, test, CV held-out, inference)")

# 2. zero ranges -> identity even in training mode
zero_lay = RandomAffineAugment(0, 0, 0, 0, 0, False)
assert np.abs(zero_lay(xs, training=True).numpy() - xs.numpy()).max() < 1e-3
print("[OK] zero-range augmentation reproduces the input")

# 3. translation: content moves RIGHT by exactly 10 px, left strip is black
out = apply_affine(xs, make_affine_transforms(z, one, z + 10.0, z)).numpy()
assert np.abs(out[:, :, 10:, :] - xs.numpy()[:, :, :-10, :]).max() < 0.5 and out[:, :, :10, :].max() <= 0.5
print("[OK] +10 px translation moves content exactly 10 px; exposed strip is black")

# 4. 90-degree rotation equals np.rot90 (either direction, since only the magnitude of the angle range matters)
out = apply_affine(xs, make_affine_transforms(z + np.pi / 2, one, z, z)).numpy()
d = min(np.abs(out - np.rot90(xs.numpy(), k, axes=(1, 2))).mean() for k in (1, 3))
assert d < 1.0, f"rotation check failed (mean abs diff {d:.3f})"
print(f"[OK] 90-degree rotation matches np.rot90 (mean abs diff {d:.4f})")

# 5. zoom-out fills with black, zoom-in stays inside the image
ones_img = tf.fill((2, IMG_SIZE, IMG_SIZE, 3), 255.0); z2, o2 = tf.zeros((2,)), tf.ones((2,))
zo = apply_affine(ones_img, make_affine_transforms(z2, o2 * 0.5, z2, z2)).numpy()
zi = apply_affine(ones_img, make_affine_transforms(z2, o2 * 2.0, z2, z2)).numpy()
assert zo[:, 0, 0, :].max() == 0.0 and abs(zo[:, IMG_SIZE // 2, IMG_SIZE // 2, :].mean() - 255.0) < 1.0 and abs(zi.mean() - 255.0) < 1.0
print("[OK] zoom-out exposes black fill at the corners; zoom-in stays fully inside the image")

# 4b. randomness and range
a1, a2 = lay(xs, training=True).numpy(), lay(xs, training=True).numpy()
assert not np.array_equal(a1, a2) and a1.min() >= 0.0 and a1.max() <= 255.0
print(f"[OK] training-mode augmentation is random and range-safe; mean |change| = {np.abs(a1 - xs.numpy()).mean():.2f} levels")

# 5b. leakage guard on the data side is still active
raised = False
try: make_dataset(get_split("80_20")["val_idx"], 32, training=False, aug_preset="conservative")
except ValueError: raised = True
assert raised; print("[OK] make_dataset still rejects augmentation on non-training data")

# 6. model-level checks
reset_session()
px_t = (0.2, 0.2)
m = build_model("custom_cnn", "relu", get_hp("custom_cnn"), pixel_stats=px_t)
ref_params = int(ptab[(ptab.arch == "custom_cnn") & (ptab.activation == "relu")]["total_params"].iloc[0])
assert any(l.name == "augment" for l in m.layers) and m.count_params() == ref_params
p1, p2 = m.predict(xs.numpy(), verbose=0), m.predict(xs.numpy(), verbose=0)
assert np.allclose(p1, p2), "inference is not deterministic - augmentation leaked into predict()"
m_none = build_model("custom_cnn", "relu", get_hp("custom_cnn", aug="none"), pixel_stats=px_t)
assert not any(l.name == "augment" for l in m_none.layers)
reset_session()
md = build_model("densenet121", "relu", get_hp("densenet121"), weights=None)
bb = get_backbone(md); assert bb is not None and bb.name.startswith("densenet")
n_unf, n_bb = set_stage(md, 2, 0.30); assert n_unf > 0
reset_session()
print(f"[OK] model-level: 'augment' layer present, parameter count unchanged ({ref_params:,}), predict() deterministic, "
      "backbone/freezing logic still finds the DenseNet backbone")

# ---------------- record + visual demonstration ----------------
cfg_aug = load_json(DIRS["logs"] / "augmentation_config.json", default={})
cfg_aug["implementation"] = ("v2: GPU layer 'RandomAffineAugment' inside the model, active only when training=True; "
                             "one fused resampling for rotation+zoom+translation, then multiplicative brightness/contrast")
save_json(cfg_aug, DIRS["logs"] / "augmentation_config.json")

s80 = get_split("80_20"); rng = np.random.RandomState(SEED)
fig, axes = plt.subplots(NUM_CLASSES, 6, figsize=(18, 12.5))
for r in range(NUM_CLASSES):
    i = int(rng.choice(s80["fit_idx"][y_all[s80["fit_idx"]] == r]))
    x_one = tf.constant(X_all[i].astype(np.float32)[None])
    axes[r, 0].imshow(X_all[i]); axes[r, 0].set_title(f"Original | {CLASS_DISPLAY[r]}", fontsize=11)
    for c in range(1, 6):
        axes[r, c].imshow(lay(x_one, training=True).numpy()[0].astype(np.uint8)); axes[r, c].set_title(f"Augmented {c}", fontsize=11)
    for c in range(6): axes[r, c].axis("off")
fig.suptitle("GPU augmentation layer (v2): rot +/-10 deg, zoom +/-10%, shift +/-5%, brightness/contrast +/-10%", fontsize=14, y=1.0)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "11_augmentation_examples_gpu.png", dpi=130, bbox_inches="tight"); plt.show()
print("\nCELL 26 complete: make_dataset and build_model are now the GPU-augmentation versions.")

In [ ]:
# ============================================================
# CELL 27: RE-BENCHMARK, CUSTOM-CNN LEARNING CHECK, RECALIBRATED COMPUTE PLAN
# ============================================================
from sklearn.model_selection import train_test_split

SMOKE2_DIR = Path("/kaggle/working/_smoke2")
PREV_STAGE1_IMG_S = {"custom_cnn": 84, "resnet50": 81, "vgg19": 85, "densenet121": 75, "efficientnetb0": 89}   # CELL 25, CPU augmentation
PREV_CORE_HOURS = 14.0                                                                                          # CELL 25 plan

tu = get_tuning_split()
sub_fit, _ = train_test_split(tu["fit_idx"], train_size=640, stratify=y_all[tu["fit_idx"]], random_state=SEED)
sub_val, _ = train_test_split(tu["val_idx"], train_size=160, stratify=y_all[tu["val_idx"]], random_state=SEED)
assert len(np.intersect1d(sub_fit, sub_val)) == 0

# ---- component throughputs ----
cw_s = class_weights_from_labels(y_all[sub_fit])
it = iter(make_dataset(sub_fit, 32, training=True, aug_preset=DEFAULT_AUG, class_weights=cw_s).repeat())
next(it); t0 = time.perf_counter()
for _ in range(40): next(it)
pipe_img_s = 40 * 32 / (time.perf_counter() - t0)

aug_layer = RandomAffineAugment(**AUG_PRESETS[DEFAULT_AUG], name="bench_aug")
xb = tf.constant(X_all[sub_fit[:32]].astype(np.float32))
@tf.function
def _aug(x): return aug_layer(x, training=True)
_aug(xb).numpy(); t0 = time.perf_counter()
for _ in range(20): _aug(xb).numpy()
aug_img_s = 20 * 32 / (time.perf_counter() - t0)
print(f"tf.data pipeline (CPU, no augmentation): {pipe_img_s:,.0f} img/s   (was 97 img/s with CPU augmentation)")
print(f"GPU augmentation layer alone           : {aug_img_s:,.0f} img/s")

# ---- smoke runs (3 + 3 epochs; custom CNN 3) ----
smoke2 = {}
for arch in ARCHITECTURES:
    ov = {"epochs": 3, "patience": 5} if arch == "custom_cnn" else {"epochs_head": 3, "epochs_ft": 3, "patience_head": 5, "patience": 5}
    smoke2[arch] = train_and_evaluate(make_run_id("smoke2", "tuning", arch, "relu"), arch, "relu", "tuning",
                                      sub_fit, sub_val, hp_overrides=ov, register=False, out_root=SMOKE2_DIR, verbose=0)

timing, rows = {}, []
for arch, r in smoke2.items():
    et, n = r["epoch_times_s"], r["n_fit"]
    s1 = et["stage1"][-1] / n; s2 = et["stage2"][-1] / n if "stage2" in et else np.nan
    overhead = r["run_wall_s"] - sum(sum(v) for v in et.values())
    timing[arch] = {"s1": s1, "s2": s2, "overhead": overhead}
    rows.append({"arch": arch, "stage1_img_s": round(1 / s1), "previous_stage1": PREV_STAGE1_IMG_S[arch],
                 "speedup_x": round((1 / s1) / PREV_STAGE1_IMG_S[arch], 1),
                 "stage2_img_s": (round(1 / s2) if np.isfinite(s2) else None),
                 "inference_img_s": round(r["summary"]["throughput_img_s_bs32"]), "overhead_s": round(overhead)})
print("\nMeasured speed with GPU augmentation (steady state; includes the validation pass, so slightly conservative):")
print(pd.DataFrame(rows).to_string(index=False))
print("Reading guide: for frozen-backbone stage 1, training speed should now track the 'inference_img_s' column "
      "(custom CNN trains all layers, so it will sit well below its inference rate).")

# ---- custom-CNN learning check (criterion fixed in advance: val accuracy > 0.50 after 15 epochs) ----
chk = train_and_evaluate(make_run_id("smoke2_learn", "tuning", "custom_cnn", "relu"), "custom_cnn", "relu", "tuning",
                         sub_fit, sub_val, hp_overrides={"epochs": 15, "patience": 20}, register=False,
                         out_root=SMOKE2_DIR, verbose=0)
va = chk["summary"]["val_accuracy"]
print(f"\nCustom CNN, 15 epochs on 640 images: val accuracy {va:.3f}, val macro-F1 {chk['summary']['val_f1_macro']:.3f}")
print("LEARNING CHECK:", "PASSED (> 0.50): the 3-epoch result was batch-norm warm-up, not a defect." if va > 0.50
      else "NOT PASSED (<= 0.50): report this to me; I will investigate before Stage F.")
plot_history(pd.read_csv(chk["paths"]["history"]), "Custom CNN learning check (640 images: not a result)", path=None, show=True)

# ---- recalibrated compute plan (same assumptions as CELL 25) ----
EXP_EPOCHS  = {"head": 8, "ft": 18, "custom": 35}      # ASSUMPTION: realised epochs before early stopping
TUNE_EPOCHS = {"head": 4, "ft": 6,  "custom": 12}      # ASSUMPTION: short tuning trials

def est_run_s(arch, n_fit, ep):
    t = timing[arch]
    core = ep["custom"] * t["s1"] * n_fit if arch == "custom_cnn" else (ep["head"] * t["s1"] + ep["ft"] * t["s2"]) * n_fit
    return core + t["overhead"]

def cv_cost(arch, split):
    return sum(est_run_s(arch, len(f["fit_idx"]), EXP_EPOCHS) for f in get_cv_folds(split))

init_s = sum(est_run_s(a, len(get_split(sp)["fit_idx"]), EXP_EPOCHS) for sp in SPLIT_CONFIGS for a in ARCHITECTURES for _ in ACTIVATIONS)
tune_s = sum(6 * est_run_s(a, len(tu["fit_idx"]), TUNE_EPOCHS) for a in ARCHITECTURES)
mand_s = sum(cv_cost(a, sp) for sp in SPLIT_CONFIGS for a in MANDATORY_CV_ARCHS)
others = [a for a in ARCHITECTURES if a not in MANDATORY_CV_ARCHS]
extra_worst_s = sum(sum(sorted([cv_cost(a, sp) for a in others], reverse=True)[:2]) for sp in SPLIT_CONFIGS)
final_s = 2 * max(est_run_s(a, len(get_split("90_10")["train_idx"]), EXP_EPOCHS) for a in ARCHITECTURES)
n_cv = sum(len(get_cv_folds(sp)) for sp in SPLIT_CONFIGS)

plan = pd.DataFrame([
    {"stage": "F  Hyperparameter search (5 archs x 6 short trials)", "runs": 30, "est_hours": tune_s / 3600},
    {"stage": "G  Initial comparison (5 archs x ReLU/GELU x 3 splits)", "runs": 30, "est_hours": init_s / 3600},
    {"stage": "I  MANDATORY CV: EffNetB0 + DenseNet121 (1 activation)", "runs": 2 * n_cv, "est_hours": mand_s / 3600},
    {"stage": "N  Final retrain (upper bound, 2 runs)", "runs": 2, "est_hours": final_s / 3600},
    {"stage": "(optional) second activation for the mandatory CV", "runs": 2 * n_cv, "est_hours": mand_s / 3600},
    {"stage": "(worst case) J  best-two extra CV for other archs", "runs": 2 * n_cv, "est_hours": extra_worst_s / 3600},
])
plan["est_hours"] = plan["est_hours"].round(2)
print(f"\nRECALIBRATED PLAN v2 (GPU hours, one T4; assumed realised epochs: head<={EXP_EPOCHS['head']} / "
      f"fine-tune<={EXP_EPOCHS['ft']} / custom<={EXP_EPOCHS['custom']}):")
print(plan.to_string(index=False))
core_h = plan["est_hours"].iloc[:4].sum()
print(f"\nCore (mandatory) total: ~{core_h:.1f} h  (CELL 25 estimate with CPU augmentation: {PREV_CORE_HOURS} h) | "
      f"with optional second-activation CV: ~{core_h + plan['est_hours'].iloc[4]:.1f} h | "
      f"plus worst-case extra CV: ~{plan['est_hours'].sum():.1f} h")
print("The epoch assumptions are guesses; the real totals come from the early-stopping behaviour in Stage F/G.")
plan.to_csv(DIRS["results"] / "compute_plan_recalibrated_v2.csv", index=False)
print("Smoke outputs are in", SMOKE2_DIR, "- safe to delete later.")

In [ ]:
# ============================================================
# CELL 28: CUSTOM-CNN LEARNING DIAGNOSTIC (SIX CONTROLLED ARMS ON THE TUNING SPLIT)
# ============================================================

# ---- 1. Make BN momentum and the pooling choice configurable (defaults reproduce CELL 26 exactly) ----
HP_CUSTOM.setdefault("bn_momentum", 0.99)
HP_CUSTOM.setdefault("head_pool", "flatten")      # 'flatten' = paper topology, 'gap' = diagnostic deviation

def build_model(arch, activation, hp, pixel_stats=None, weights="imagenet"):
    """Overrides CELL 26. Identical unless hp['bn_momentum'] / hp['head_pool'] are changed (custom CNN only)."""
    assert arch in ARCH_SPECS and activation in ("relu", "gelu")
    spec = ARCH_SPECS[arch]
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image_0_255")
    x = inp
    aug = _aug_layer_from_hp(hp)
    if aug is not None:
        x = aug(x)
    if spec["kind"] == "scratch":
        assert pixel_stats is not None, "custom CNN needs training-set (mean, std)"
        x = InputPreprocess("standardize", pixel_stats[0], pixel_stats[1], name="preprocess")(x)
        for i, f in enumerate([32, 64, 128, 256]):
            x = keras.layers.Conv2D(f, 3, padding="same", kernel_initializer="he_normal", name=f"conv{i+1}")(x)
            x = keras.layers.Activation(activation, name=f"act{i+1}")(x)
            x = keras.layers.BatchNormalization(momentum=hp.get("bn_momentum", 0.99), name=f"bn{i+1}")(x)
            x = keras.layers.MaxPooling2D(2, name=f"pool{i+1}")(x)
            if i < 2 and hp["conv_dropout"] > 0:
                x = keras.layers.Dropout(hp["conv_dropout"], name=f"drop{i+1}")(x)
        if hp.get("head_pool", "flatten") == "gap":
            x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
        else:
            x = keras.layers.Flatten(name="flatten")(x)
        x = keras.layers.Dense(hp["head_units"], kernel_initializer="he_normal", name="head_dense")(x)
        x = keras.layers.Activation(activation, name="head_act")(x)
        x = keras.layers.Dropout(hp["dropout"], name="head_drop")(x)
    else:
        x = InputPreprocess(spec["preprocess"], name="preprocess")(x)
        base = spec["fn"](include_top=False, weights=weights, input_shape=(IMG_SIZE, IMG_SIZE, 3))
        x = base(x, training=False)
        x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
        x = keras.layers.Dropout(hp["dropout"], name="head_drop1")(x)
        x = keras.layers.Dense(hp["head_units"], kernel_initializer="he_normal", name="head_dense")(x)
        x = keras.layers.Activation(activation, name="head_act")(x)
        x = keras.layers.Dropout(hp["dropout"], name="head_drop2")(x)
    out = keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="predictions")(x)
    return keras.Model(inp, out, name=f"{arch}_{activation}")

# ---- 2. Sanity checks: defaults unchanged, new options take effect ----
reset_session()
ref_params = int(ptab[(ptab.arch == "custom_cnn") & (ptab.activation == "relu")]["total_params"].iloc[0])
m0 = build_model("custom_cnn", "relu", get_hp("custom_cnn"), pixel_stats=(0.2, 0.2))
assert m0.count_params() == ref_params and abs(m0.get_layer("bn1").momentum - 0.99) < 1e-9
assert any(l.name == "flatten" for l in m0.layers)
reset_session()
m1 = build_model("custom_cnn", "relu", get_hp("custom_cnn", bn_momentum=0.9, head_pool="gap"), pixel_stats=(0.2, 0.2))
assert abs(m1.get_layer("bn1").momentum - 0.9) < 1e-9 and any(l.name == "gap" for l in m1.layers)
gap_params = m1.count_params()
reset_session()
print(f"[OK] default custom CNN unchanged ({ref_params:,} params, Flatten, BN momentum 0.99)")
print(f"[OK] options work; GAP variant has {gap_params:,} params (topology deviation, diagnostic only)")

# ---- 3. Reference points ----
tu = get_tuning_split()
val_share = np.bincount(y_all[tu["val_idx"]], minlength=NUM_CLASSES) / len(tu["val_idx"])
print(f"\nTuning split: fit={len(tu['fit_idx'])}, val={len(tu['val_idx'])} (test sets untouched)")
print(f"Reference: majority-class accuracy on val = {val_share.max():.3f} | uniform-prediction loss = ln4 = {np.log(4):.3f}")
PASS_ACC, PASS_F1 = 0.60, 0.55
print(f"Pre-registered 'learns' criterion: val accuracy > {PASS_ACC} AND val macro-F1 > {PASS_F1}")

# ---- 4. The six arms (one change at a time relative to B) ----
COMMON = {"epochs": 30, "patience": 10}
ARMS = {
    "A_baseline_lr1e-3": {},
    "B_lr3e-4":          {"lr": 3e-4},
    "C_lr1e-4":          {"lr": 1e-4},
    "D_lr3e-4_bn09":     {"lr": 3e-4, "bn_momentum": 0.9},
    "E_lr3e-4_noaug":    {"lr": 3e-4, "aug": "none"},
    "F_lr3e-4_GAP":      {"lr": 3e-4, "head_pool": "gap"},
}
TIER = {"A": 1, "B": 1, "C": 1, "D": 1, "E": 2, "F": 3}     # 1 = paper topology + our augmentation; 2 = no augmentation; 3 = topology change

results = {}
for arm, ov in ARMS.items():
    rid = make_run_id("diag", "tuning", "custom_cnn", "relu", trial=arm)
    results[arm] = train_and_evaluate(rid, "custom_cnn", "relu", "tuning", tu["fit_idx"], tu["val_idx"],
                                      hp_overrides={**COMMON, **ov}, verbose=0)

# ---- 5. Results table ----
rows, hists = [], {}
for arm, r in results.items():
    s = r["summary"]; h = pd.read_csv(r["paths"]["history"]); hists[arm] = h
    pc = r["metrics"]["val"]["per_class"]
    rows.append({"arm": arm, "tier": TIER[arm[0]], "val_acc": s["val_accuracy"], "val_f1": s["val_f1_macro"],
                 "val_loss": s["val_loss"], "train_acc_clean": s["train_accuracy"], "train_loss_clean": s["train_loss"],
                 "loss_ep1_train": float(h["loss"].iloc[0]), "loss_ep1_val": float(h["val_loss"].iloc[0]),
                 "best_epoch": s["best_epoch"], "epochs": s["epochs_run"], "minutes": round(s["train_time_s"] / 60, 1),
                 **{f"recall_{c}": pc[c]["sensitivity"] for c in CLASS_NAMES}})
tab = pd.DataFrame(rows)
print("\n" + "=" * 100)
print("DIAGNOSTIC RESULTS (validation = 270 tuning-split images; one seed per arm; differences of a few points are noise)")
print("=" * 100)
print(tab.round(3).to_string(index=False))

# ---- 6. Pre-registered verdict ----
passed = tab[(tab["val_acc"] > PASS_ACC) & (tab["val_f1"] > PASS_F1)]
winner = None
for tier in (1, 2, 3):
    cand = passed[passed["tier"] == tier]
    if len(cand):
        winner = cand.sort_values("val_loss").iloc[0]; break
print(f"\nArms meeting the 'learns' criterion: {list(passed['arm']) if len(passed) else 'NONE'}")
if winner is None:
    print("VERDICT: NO ARM LEARNS. Do not start Stage F; paste this output and I will investigate further.")
else:
    t = int(winner["tier"])
    kind = {1: "paper topology + our augmentation", 2: "NO augmentation (would need discussion in the report)",
            3: "GAP topology - a deviation from the paper (must be reported as such)"}[t]
    print(f"VERDICT: selected arm = {winner['arm']} (tier {t}: {kind}); val loss {winner['val_loss']:.3f}, "
          f"acc {winner['val_acc']:.3f}, macro-F1 {winner['val_f1']:.3f}")

tab.to_csv(DIRS["results"] / "custom_cnn_diagnostic.csv", index=False)
save_json({"criterion": {"val_acc_gt": PASS_ACC, "val_f1_macro_gt": PASS_F1}, "arms": ARMS, "common": COMMON,
           "selected_arm": None if winner is None else str(winner["arm"]),
           "selected_overrides": None if winner is None else ARMS[str(winner["arm"])],
           "note": "single seed per arm, 270-image validation set, tuning split only; test sets untouched"},
          DIRS["logs"] / "custom_cnn_diagnostic.json")

# ---- 7. Curves for all arms ----
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for arm, h in hists.items():
    ep = np.arange(1, len(h) + 1)
    axes[0].plot(ep, h["val_accuracy"], lw=1.8, label=arm)
    axes[1].plot(ep, np.clip(h["val_loss"], 0, 4), lw=1.8, label=arm)
axes[0].axhline(val_share.max(), color="gray", ls=":", label="majority-class accuracy")
axes[1].axhline(np.log(4), color="gray", ls=":", label="uniform guess (ln 4)")
axes[0].set_title("Custom CNN diagnostic: validation accuracy", fontsize=14)
axes[1].set_title("Custom CNN diagnostic: validation loss (clipped at 4)", fontsize=14)
for ax, yl in zip(axes, ["Validation accuracy", "Validation loss"]):
    ax.set_xlabel("Epoch", fontsize=12); ax.set_ylabel(yl, fontsize=12); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "12_custom_cnn_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nThis diagnostic used only the tuning split; the test sets were never touched.")

In [ ]:
# ============================================================
# CELL 59: MEMORY GUARD V3 (HOST RAM + GPU) AND RESTART STATE CHECK
# ============================================================
import inspect, ctypes, functools, psutil

def _read_int(path):
    try:
        s = Path(path).read_text().strip()
        return None if s in ("max", "") else int(s)
    except Exception:
        return None

def container_memory():
    """(used_bytes, limit_bytes, source). Uses the container's cgroup limit when visible; psutil's host view otherwise."""
    for used_p, lim_p in [("/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory.max"),
                          ("/sys/fs/cgroup/memory/memory.usage_in_bytes", "/sys/fs/cgroup/memory/memory.limit_in_bytes")]:
        u, l = _read_int(used_p), _read_int(lim_p)
        if u is not None and l is not None and l < 2**50:
            return u, l, "cgroup"
    vm = psutil.virtual_memory()
    return vm.total - vm.available, vm.total, "psutil host view (may overstate the real limit)"

def host_rss_gib():
    return psutil.Process(os.getpid()).memory_info().rss / 2**30

def trim_host_memory():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)      # return freed heap pages to the OS
    except Exception:
        pass

def mem_line(tag=""):
    used, lim, _ = container_memory()
    try:
        g = f"{tf.config.experimental.get_memory_info('GPU:0')['current'] / 2**20:7.0f} MiB"
    except Exception:
        g = "n/a"
    return (f"[mem] {tag:<30} process RSS {host_rss_gib():5.2f} GiB | container {used/2**30:5.1f} of {lim/2**30:4.1f} GiB "
            f"(includes file cache) | GPU {g}")

class MemoryBudgetExceeded(RuntimeError):
    pass

MEM_STOP_FRAC = 0.75       # my choice: stop starting new runs when process RSS exceeds 75% of the container limit

# ---- install: unwrap any earlier guard (CELL 53, 51A, 51B) so wrappers can never call themselves ----
_raw_make_dataset = inspect.unwrap(make_dataset)
_raw_train_and_evaluate = inspect.unwrap(train_and_evaluate)

@functools.wraps(_raw_make_dataset)
def make_dataset(*args, **kwargs):
    with tf.device("/CPU:0"):                  # keep the image tensors off the GPU
        return _raw_make_dataset(*args, **kwargs)

@functools.wraps(_raw_train_and_evaluate)
def train_and_evaluate(*args, **kwargs):
    rid = args[0] if args else kwargs.get("run_id", "?")
    if run_is_done(rid):                        # already finished: nothing will be trained, no memory check needed
        return _raw_train_and_evaluate(*args, **kwargs)
    trim_host_memory()
    used, lim, _ = container_memory()
    if host_rss_gib() * 2**30 > MEM_STOP_FRAC * lim:
        raise MemoryBudgetExceeded(
            f"Process RSS {host_rss_gib():.1f} GiB exceeds {100*MEM_STOP_FRAC:.0f}% of the {lim/2**30:.1f} GiB container limit even after "
            "cleanup, so a leak is accumulating. Stopping BEFORE the kernel is killed. Finished runs are saved: restart the session, "
            "re-run the setup cells (not 53/51A/51B), CELL 59, then this cell again.")
    rss0 = host_rss_gib(); t0 = time.time()
    try:
        return _raw_train_and_evaluate(*args, **kwargs)
    finally:
        keras.backend.clear_session(); trim_host_memory()
        keras.mixed_precision.set_global_policy("mixed_float16" if MIXED_PRECISION else "float32")
        print(mem_line(f"after run ({host_rss_gib() - rss0:+.2f} GiB RSS)") + f" | {(time.time()-t0)/60:.1f} min")

# ---- self-tests ----
assert inspect.unwrap(make_dataset) is _raw_make_dataset and inspect.unwrap(train_and_evaluate) is _raw_train_and_evaluate
assert make_dataset.__wrapped__ is _raw_make_dataset, "wrapper chain is not a single level (recursion risk)"
_ds = make_dataset(np.arange(64), 32, training=False); _xb, _yb = next(iter(_ds))
assert _xb.shape == (32, 224, 224, 3) and _yb.shape == (32, 4)
del _ds, _xb, _yb; trim_host_memory()
print("[OK] guard installed as a single wrapper level (no recursion), and make_dataset still works")
_, lim, src = container_memory()
print(f"Container memory limit: {lim/2**30:.1f} GiB (source: {src}); runs stop starting when process RSS > {MEM_STOP_FRAC*lim/2**30:.1f} GiB")
print(mem_line("now"))

# ---- restart state check ----
required = ["X_all", "y_all", "groups_all", "make_dataset", "train_and_evaluate", "run_cv", "cv_activation", "final_overrides",
            "make_run_id", "load_result", "SEL", "AUG_CHOICE", "disp", "get_cv_folds"]
missing = [n for n in required if n not in globals()]
print("\nMissing objects:", missing if missing else "none - all definitions are loaded")
if missing:
    print("Run the setup cells in the listed order (1-18, 20-24, 26, 28, 38, 45, 46) before CELL 52.")
else:
    print(f"X_all: {X_all.shape} {X_all.dtype} | {X_all.nbytes/2**20:.0f} MiB | TEST_UNLOCKED = {TEST_UNLOCKED}")
    print("\nSaved CV fold results on disk:")
    for sp in SPLIT_CONFIGS:
        for arch in MANDATORY_CV_ARCHS:
            act, K = cv_activation(sp, arch), SPLIT_CONFIGS[sp]["cv_folds"]
            done = [f for f in range(K) if load_result(make_run_id("cv", sp, arch, act, fold=f)) is not None]
            print(f"   {sp:<6}{disp(arch):<16}{act.upper():<6} {len(done):>2}/{K} folds saved {done}")

In [ ]:
# ============================================================
# CELL 60: MEMORY GUARD V4 (HOST RAM + GPU, CHECKED BEFORE EVERY NEW RUN)
# ============================================================
GPU_STOP_MIB = 8000     # my choice: refuse to start a run if GPU memory still in use after cleanup exceeds this

class GpuMemoryBudgetExceeded(RuntimeError):
    pass

def gpu_current_mib():
    try:
        return tf.config.experimental.get_memory_info("GPU:0")["current"] / 2**20
    except Exception:
        return float("nan")

# Rebuild the wrappers from the raw functions, so CELL 59's wrapper is replaced and never stacked.
_raw_make_dataset = inspect.unwrap(make_dataset)
_raw_train_and_evaluate = inspect.unwrap(train_and_evaluate)

@functools.wraps(_raw_make_dataset)
def make_dataset(*args, **kwargs):
    with tf.device("/CPU:0"):
        return _raw_make_dataset(*args, **kwargs)

@functools.wraps(_raw_train_and_evaluate)
def train_and_evaluate(*args, **kwargs):
    rid = args[0] if args else kwargs.get("run_id", "?")
    if run_is_done(rid):                                  # finished runs are skipped inside; no check needed
        return _raw_train_and_evaluate(*args, **kwargs)
    drop_ref = globals().get("drop_traceback_refs")
    if drop_ref: drop_ref()
    keras.backend.clear_session(); trim_host_memory()
    _, lim, _ = container_memory()
    gpu0, rss0 = gpu_current_mib(), host_rss_gib()
    if rss0 * 2**30 > MEM_STOP_FRAC * lim:
        raise MemoryBudgetExceeded(f"Host RSS {rss0:.1f} GiB exceeds {100*MEM_STOP_FRAC:.0f}% of {lim/2**30:.1f} GiB after cleanup. "
                                   "Restart the session (not Stop), re-run the setup cells, CELL 59, CELL 60, then this cell again.")
    if gpu0 > GPU_STOP_MIB:
        raise GpuMemoryBudgetExceeded(f"GPU memory in use after cleanup is {gpu0:.0f} MiB (> {GPU_STOP_MIB}). Stopping BEFORE an "
                                      "out-of-memory failure. Restart the session (not Stop), re-run the setup cells, CELL 59, CELL 60, "
                                      "then this cell again. Finished runs are kept.")
    t0 = time.time()
    try:
        return _raw_train_and_evaluate(*args, **kwargs)
    finally:
        keras.backend.clear_session(); trim_host_memory()
        keras.mixed_precision.set_global_policy("mixed_float16" if MIXED_PRECISION else "float32")
        print(mem_line(f"after run (RSS {host_rss_gib() - rss0:+.2f}, GPU {gpu_current_mib() - gpu0:+.0f} MiB)") + f" | {(time.time()-t0)/60:.1f} min")

assert make_dataset.__wrapped__ is _raw_make_dataset and train_and_evaluate.__wrapped__ is _raw_train_and_evaluate
_ds = make_dataset(np.arange(64), 32, training=False); _xb, _yb = next(iter(_ds))
assert _xb.shape == (32, 224, 224, 3); del _ds, _xb, _yb; trim_host_memory()
print("[OK] guard v4 installed as a single wrapper level")
print(f"Stops before a new run if: host RSS > {MEM_STOP_FRAC*container_memory()[1]/2**30:.1f} GiB, or GPU memory in use > {GPU_STOP_MIB} MiB")
print(mem_line("now"))

In [ ]:
# ============================================================
# CELL 30: STAGE F SETUP - FREEZE SPLITS, SEARCH SPACES, PRE-REGISTERED RULES, TRIAL CONFIGS
# ============================================================

# ---- 1. Freeze the splits (any later change would invalidate every training run) ----
FREEZE_PATH = DIRS["splits"] / "splits_frozen.json"
assert SPLITS_RAW["signature"] == DATA_SIGNATURE, "splits do not match the cleaned dataset"
lv = load_json(DIRS["logs"] / "leakage_verification.json")
assert lv is not None and lv["signature"] == DATA_SIGNATURE and lv["n_checks"] > 0, "run CELL 15 (leakage verification) first"
fr = load_json(FREEZE_PATH)
if fr is None:
    save_json({"signature": DATA_SIGNATURE, "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"),
               "sizes": {n: {k: int(len(v)) for k, v in get_split(n).items()} for n in SPLIT_CONFIGS},
               "leakage_checks_passed": lv["n_checks"]}, FREEZE_PATH)
    print("Splits FROZEN:", FREEZE_PATH)
else:
    assert fr["signature"] == DATA_SIGNATURE, "SPLITS CHANGED AFTER FREEZING - training results would not be comparable"
    print("Splits already frozen at", fr["frozen_at"], "- signature matches.")

# ---- 2. Search spaces, budgets, defaults ----
tu = get_tuning_split()
SEARCH_SPACE = {"custom_cnn": {"lr": [5e-5, 1e-4, 2e-4, 4e-4], "dropout": [0.3, 0.5], "conv_dropout": [0.0, 0.1, 0.2],
                               "weight_decay": [1e-4, 1e-3, 1e-2], "batch_size": [16, 32, 64]}}
_pre = {"lr_head": [3e-4, 1e-3, 3e-3], "lr_ft": [1e-5, 3e-5, 1e-4], "dropout": [0.2, 0.3, 0.5],
        "weight_decay": [1e-4, 1e-3, 1e-2], "batch_size": [16, 32, 64], "unfreeze_frac": [0.1, 0.2, 0.3, 0.5]}
for a in ARCHITECTURES:
    if a != "custom_cnn":
        SEARCH_SPACE[a] = {k: list(v) for k, v in _pre.items()}

TUNE_BUDGET = {a: ({"epochs": 25, "patience": 8} if a == "custom_cnn" else
                   {"epochs_head": 5, "epochs_ft": 8, "patience_head": 3, "patience": 4, "rlrop_patience": 2})
               for a in ARCHITECTURES}
DEFAULT_OVERRIDES = {a: ({"lr": 1e-4} if a == "custom_cnn" else {}) for a in ARCHITECTURES}   # custom CNN: arm C of CELL 28

def base_hp(arch):
    b = dict(HP_CUSTOM if arch == "custom_cnn" else HP_PRETRAINED)
    b.update(DEFAULT_OVERRIDES[arch]); return b

def sample_configs(arch, n_random=5):
    """Trial 0 = defaults; trials 1..n = distinct random grid points (seeded per architecture)."""
    space, base = SEARCH_SPACE[arch], base_hp(arch)
    for k in space: assert base[k] in space[k], f"default {k}={base[k]} not in its search list"
    rng = np.random.RandomState(SEED + 100 * ARCHITECTURES.index(arch))
    full = lambda ov: tuple((k, ov.get(k, base[k])) for k in space)
    cfgs = [dict(DEFAULT_OVERRIDES[arch])]; seen = {full(cfgs[0])}
    while len(cfgs) < 1 + n_random:
        ov = {k: space[k][rng.randint(len(space[k]))] for k in space}
        if full(ov) in seen: continue
        seen.add(full(ov)); cfgs.append(ov)
    return cfgs

TUNE_CONFIGS = {a: sample_configs(a) for a in ARCHITECTURES}

cfg_path = DIRS["tuning"] / "trial_configs.json"
canon = json.loads(json.dumps(TUNE_CONFIGS, cls=NumpyEncoder))
old = load_json(cfg_path)
if old is not None:
    assert old == canon, "regenerated trial configs differ from the saved ones - do not mix old and new trial results"
else:
    save_json(canon, cfg_path)

RULES = {"data": "tuning split only (core 70% pool); test sets never touched", "search": "trial 0 = defaults + 5 seeded random trials per architecture",
         "budget": TUNE_BUDGET, "activation": "tuned with ReLU; reused unchanged for GELU",
         "selection": ["highest validation macro-F1", "trials within 0.01 macro-F1 of the best: lowest validation loss",
                       "a non-default trial must beat the default (trial 0) by >= 0.02 macro-F1, otherwise the default is kept"],
         "caveats": ["6 trials on 270 validation images: noisy", "short budgets favour fast-learning (higher-lr) configurations",
                     "selected validation scores are optimistically biased (selection on the same images)"]}
save_json(RULES, DIRS["logs"] / "stage_f_rules.json")

# ---- 3. Helpers ----
_SHARE_CACHE = {}
def unfrozen_share(arch, frac):
    """Share of the BACKBONE's weights that are trainable in stage 2 (BatchNorm stays frozen). NaN for the custom CNN."""
    if ARCH_SPECS[arch]["kind"] == "scratch": return float("nan")
    key = (arch, round(float(frac), 4))
    if key not in _SHARE_CACHE:
        reset_session()
        m = build_model(arch, "relu", get_hp(arch), weights=None)        # random init: only the structure matters here
        set_stage(m, 2, frac); bb = get_backbone(m)
        tot = sum(int(np.prod(w.shape)) for w in bb.weights)
        trn = sum(int(np.prod(w.shape)) for w in bb.trainable_weights)
        _SHARE_CACHE[key] = trn / tot
        del m; reset_session()
    return _SHARE_CACHE[key]

tune_results = {}
def run_tuning(arch):
    """Runs (or skips, if already registered) the 6 trials of one architecture on the tuning split with ReLU."""
    rows, t0 = [], time.time()
    for t, ov in enumerate(TUNE_CONFIGS[arch]):
        rid = make_run_id("tune", "tuning", arch, "relu", trial=t)
        r = train_and_evaluate(rid, arch, "relu", "tuning", tu["fit_idx"], tu["val_idx"],
                               hp_overrides={**TUNE_BUDGET[arch], **ov}, verbose=0)
        s = r["summary"]
        rows.append({"trial": t, "config": "defaults" if (t == 0 and not ov) else str({k: v for k, v in ov.items()}),
                     "val_f1": round(s["val_f1_macro"], 4), "val_acc": round(s["val_accuracy"], 4), "val_loss": round(s["val_loss"], 4),
                     "train_acc": round(s["train_accuracy"], 4), "best_epoch": s["best_epoch"], "min": round(s["train_time_s"] / 60, 1)})
    out = pd.DataFrame(rows)
    print(f"\n{arch}: 6 trials finished in {(time.time()-t0)/60:.1f} min (skipped trials count as ~0)")
    print(out.to_string(index=False))
    return out

# ---- 4. Show what will be run ----
print("\nTrial configurations (trial 0 = defaults):")
for a in ARCHITECTURES:
    print(f"\n  {a}  (budget: {TUNE_BUDGET[a]})")
    for t, ov in enumerate(TUNE_CONFIGS[a]):
        print(f"    trial {t}: {ov if ov else 'defaults'}")
print("\nRules saved to logs/stage_f_rules.json; configurations to tuning/trial_configs.json.")

In [ ]:
# ============================================================
# CELL 31: STAGE F HYPERPARAMETER SEARCH - CUSTOM CNN (6 TRIALS, RESUMABLE)
# ============================================================
tune_results["custom_cnn"] = run_tuning("custom_cnn")

In [ ]:
# ============================================================
# CELL 32: STAGE F HYPERPARAMETER SEARCH - RESNET50 (6 TRIALS, RESUMABLE)
# ============================================================
tune_results["resnet50"] = run_tuning("resnet50")

In [ ]:
# ============================================================
# CELL 33: STAGE F HYPERPARAMETER SEARCH - VGG19 (6 TRIALS, RESUMABLE)
# ============================================================
tune_results["vgg19"] = run_tuning("vgg19")

In [ ]:
# ============================================================
# CELL 34: STAGE F HYPERPARAMETER SEARCH - DENSENET121 (6 TRIALS, RESUMABLE)
# ============================================================
tune_results["densenet121"] = run_tuning("densenet121")

In [ ]:
# ============================================================
# CELL 35: STAGE F HYPERPARAMETER SEARCH - EFFICIENTNETB0 (6 TRIALS, RESUMABLE)
# ============================================================
tune_results["efficientnetb0"] = run_tuning("efficientnetb0")

In [ ]:
# ============================================================
# CELL 36: APPLY PRE-REGISTERED SELECTION, SAVE TUNED HYPERPARAMETERS
# ============================================================
DELETE_TUNING_CHECKPOINTS = True

rows = []
for arch in ARCHITECTURES:
    base = base_hp(arch)
    for t, ov in enumerate(TUNE_CONFIGS[arch]):
        r = load_result(make_run_id("tune", "tuning", arch, "relu", trial=t))
        if r is None:
            raise RuntimeError(f"Missing trial result for {arch} trial {t}: run its search cell (CELLs 31-35) first.")
        s = r["summary"]
        row = {"arch": arch, "trial": t, **{k: ov.get(k, base[k]) for k in SEARCH_SPACE[arch]},
               "unfrozen_weight_share": unfrozen_share(arch, ov.get("unfreeze_frac", base.get("unfreeze_frac", 0.0))),
               "val_f1": s["val_f1_macro"], "val_acc": s["val_accuracy"], "val_loss": s["val_loss"],
               "train_acc": s["train_accuracy"], "gap_acc": s["gap_accuracy"], "best_epoch": s["best_epoch"],
               "minutes": s["train_time_s"] / 60}
        rows.append(row)
T = pd.DataFrame(rows)
T.to_csv(DIRS["tuning"] / "trial_results.csv", index=False)

# ---- Pre-registered rule ----
TUNED_HP, lines = {}, []
for arch in ARCHITECTURES:
    g = T[T.arch == arch]
    top = g[g.val_f1 >= g.val_f1.max() - 0.01].sort_values("val_loss").iloc[0]
    dflt = g[g.trial == 0].iloc[0]
    if int(top.trial) != 0 and (top.val_f1 - dflt.val_f1) < 0.02:
        chosen, reason = dflt, (f"default kept: best trial {int(top.trial)} beats it by only {top.val_f1 - dflt.val_f1:+.3f} F1 (< 0.02 margin)")
    elif int(top.trial) == 0:
        chosen, reason = dflt, "default was the best trial"
    else:
        chosen, reason = top, f"trial {int(top.trial)} beats the default by {top.val_f1 - dflt.val_f1:+.3f} F1 (>= 0.02 margin)"
    ov = TUNE_CONFIGS[arch][int(chosen.trial)]
    TUNED_HP[arch] = {"trial": int(chosen.trial), "overrides": json.loads(json.dumps(ov, cls=NumpyEncoder)), "reason": reason,
                      "val_f1": float(chosen.val_f1), "val_loss": float(chosen.val_loss), "val_acc": float(chosen.val_acc),
                      "f1_range_over_trials": [float(g.val_f1.min()), float(g.val_f1.max())],
                      "full_hp": {k: (v if not isinstance(v, np.generic) else v.item()) for k, v in get_hp(arch, **ov).items()}}
    lines.append(f"  {arch:15s} -> trial {int(chosen.trial)} | val F1 {chosen.val_f1:.3f}, loss {chosen.val_loss:.3f} | {reason}")

print("=" * 110); print("ALL 30 TRIALS (validation = 270 tuning-split images; ReLU)"); print("=" * 110)
show = T.copy(); show["unfrozen_weight_share"] = show["unfrozen_weight_share"].round(2)
print(show.round(3).to_string(index=False))
print("\nSELECTION (rule fixed before the search: top F1, ties within 0.01 -> lowest loss; challenger needs >= 0.02 F1 over defaults):")
print("\n".join(lines))
print("\nSpread of validation macro-F1 across the 6 trials per architecture (a rough noise indicator):")
sp = T.groupby("arch")["val_f1"].agg(["min", "max"]); sp["range"] = sp["max"] - sp["min"]
print(sp.round(3).to_string())
print("\nCAUTION: the selected validation scores are optimistically biased (chosen on the same 270 images). "
      "Do not compare them with later test results as if they were independent.")

save_json(TUNED_HP, DIRS["tuning"] / "best_hyperparameters.json")
save_json(TUNED_HP, DIRS["logs"] / "best_hyperparameters.json")
print("Saved tuning/best_hyperparameters.json (Stage G loads these via get_hp(arch, **overrides)).")

# ---- Figures ----
disp = lambda a: ARCH_SPECS[a]["display"]
piv_f1 = T.pivot(index="arch", columns="trial", values="val_f1").loc[ARCHITECTURES]
piv_ls = T.pivot(index="arch", columns="trial", values="val_loss").loc[ARCHITECTURES]
for piv in (piv_f1, piv_ls):
    piv.index = [disp(a) for a in piv.index]; piv.columns = [f"trial {c}" + (" (default)" if c == 0 else "") for c in piv.columns]
plot_bar_compare(piv_f1, "Stage F: validation macro-F1 per trial (tuning split, ReLU)", "Validation macro-F1",
                 DIRS["figures"] / "13_tuning_val_f1.png", ylim=(0, 1.05), figsize=(15, 5.5), xlabel="Architecture")
plot_bar_compare(piv_ls, "Stage F: validation loss per trial (tuning split, ReLU)", "Validation loss",
                 DIRS["figures"] / "14_tuning_val_loss.png", figsize=(15, 5.5), xlabel="Architecture")

# ---- Disk hygiene: tuning checkpoints are never reused ----
if DELETE_TUNING_CHECKPOINTS:
    freed = n = 0
    for p in list(DIRS["checkpoints"].glob("tune__*")) + list(DIRS["checkpoints"].glob("diag__*")):
        freed += p.stat().st_size; p.unlink(); n += 1
    print(f"\nDeleted {n} tuning/diagnostic checkpoint files ({freed/1e6:.0f} MB). Histories, probabilities and results are kept.")
print("\nStage F search complete. The test sets were never touched.")

In [ ]:
# ============================================================
# CELL 37: AUGMENTATION ABLATION (3 PRESETS x 2 ARCHITECTURES x 2 SEEDS, TUNING SPLIT)
# ============================================================

TUNED_HP = load_json(DIRS["tuning"] / "best_hyperparameters.json")
assert TUNED_HP is not None and set(TUNED_HP) == set(ARCHITECTURES), "run CELL 36 first (best_hyperparameters.json missing)"
tu = get_tuning_split()

ABL_ARCHS   = ["efficientnetb0", "densenet121"]
ABL_PRESETS = ["none", "conservative", "conservative_hflip"]
ABL_SEEDS   = [42, 43]
ABL_BUDGET  = {"epochs_head": 5, "epochs_ft": 15, "patience_head": 3, "patience": 5, "rlrop_patience": 2}
RULE = {"data": "tuning split only; test sets locked", "design": "2 archs x 3 presets x 2 seeds, tuned hyperparameters, ReLU head",
        "budget": ABL_BUDGET,
        "adopt_rule": "a preset replaces 'conservative' only if mean paired val macro-F1 gain >= 0.02 AND it wins >= 3 of 4 pairs; "
                      "if both qualify the larger mean gain wins; otherwise 'conservative' is kept",
        "caveat": "4 paired comparisons on 270 validation images; short budget"}
save_json(RULE, DIRS["logs"] / "augmentation_ablation_rule.json")
print("Pre-registered rule:", RULE["adopt_rule"])

for arch in ABL_ARCHS:
    for preset in ABL_PRESETS:
        for seed in ABL_SEEDS:
            rid = make_run_id("abl", "tuning", arch, "relu", trial=f"{preset}_s{seed}")
            train_and_evaluate(rid, arch, "relu", "tuning", tu["fit_idx"], tu["val_idx"],
                               hp_overrides={**TUNED_HP[arch]["overrides"], **ABL_BUDGET, "aug": preset},
                               seed=seed, verbose=0)

# ---- Results ----
rows = []
for arch in ABL_ARCHS:
    for preset in ABL_PRESETS:
        for seed in ABL_SEEDS:
            r = load_result(make_run_id("abl", "tuning", arch, "relu", trial=f"{preset}_s{seed}")); s = r["summary"]
            rows.append({"arch": arch, "preset": preset, "seed": seed, "val_f1": s["val_f1_macro"], "val_acc": s["val_accuracy"],
                         "val_loss": s["val_loss"], "train_acc": s["train_accuracy"], "gap_acc": s["gap_accuracy"],
                         "best_epoch": s["best_epoch"]})
A = pd.DataFrame(rows)
A.to_csv(DIRS["results"] / "augmentation_ablation.csv", index=False)
print("\nALL 12 RUNS:"); print(A.round(4).to_string(index=False))

piv = A.pivot_table(index=["arch", "seed"], columns="preset", values="val_f1")[ABL_PRESETS]
diff = piv.sub(piv["conservative"], axis=0)
print("\nPaired difference in val macro-F1 vs 'conservative' (rows = architecture, seed):")
print(diff.round(4).to_string())

pv = A.pivot_table(index=["arch", "preset"], columns="seed", values="val_f1")
seed_gap = (pv[ABL_SEEDS[0]] - pv[ABL_SEEDS[1]]).abs()
print(f"\nNOISE ESTIMATE: |F1(seed {ABL_SEEDS[0]}) - F1(seed {ABL_SEEDS[1]})| for identical configurations: "
      f"mean {seed_gap.mean():.4f}, max {seed_gap.max():.4f} over {len(seed_gap)} configs")

det, cands = {}, []
for p in ["none", "conservative_hflip"]:
    d = diff[p]; mean_d, wins = float(d.mean()), int((d > 0).sum())
    ok = (mean_d >= 0.02) and (wins >= 3)
    det[p] = {"mean_gain_vs_conservative": mean_d, "wins_of_4": wins, "qualifies": bool(ok)}
    if ok: cands.append((mean_d, p))
    print(f"  {p:20s}: mean gain {mean_d:+.4f} | wins {wins}/4 | qualifies: {ok}")
AUG_CHOICE = max(cands)[1] if cands else "conservative"
print(f"\nVERDICT (pre-registered rule): augmentation preset for all later experiments = '{AUG_CHOICE}'")
if AUG_CHOICE == "conservative_hflip":
    print("NOTE: horizontal flip was adopted; the report must state the mixed-orientation caveat.")
save_json({"choice": AUG_CHOICE, "details": det, "seed_noise_mean_abs_f1": float(seed_gap.mean())},
          DIRS["logs"] / "augmentation_choice.json")

tab = A.groupby(["arch", "preset"])["val_f1"].mean().unstack()[ABL_PRESETS]
err = A.groupby(["arch", "preset"])["val_f1"].agg(lambda v: (v.max() - v.min()) / 2).unstack()[ABL_PRESETS]
tab.index = err.index = [ARCH_SPECS[a]["display"] for a in tab.index]
plot_bar_compare(tab, "Augmentation ablation: validation macro-F1 (mean of 2 seeds; bars = half-range)", "Validation macro-F1",
                 DIRS["figures"] / "15_augmentation_ablation.png", err=err, ylim=(0.5, 1.0), xlabel="Architecture")

freed = 0
for p in list(DIRS["checkpoints"].glob("abl__*")):
    freed += p.stat().st_size; p.unlink()
print(f"\nDeleted ablation checkpoints ({freed/1e6:.0f} MB). The test sets were never touched.")

In [ ]:
# ============================================================
# CELL 29: PROJECT BACKUP V2 (SUPERSEDES CELL 19; EXCLUDES MODEL-WEIGHT FILES)
# ============================================================

BACKUP_EXCLUDE_NAMES = {"X_clean_224.npy"}                 # re-created by CELL 12
BACKUP_EXCLUDE_SUFFIXES = (".weights.h5", ".keras", ".tmp")  # model weights are large; histories/results/probs are kept

def backup_project_zip(zip_name: str = "project_backup.zip", include_weights: bool = False):
    """Zips the project folder into /kaggle/working/<zip_name>. By default model-weight files and the image cache are skipped.
    Use include_weights=True only when you specifically want the weights (e.g. the final model in Stage N)."""
    if not REGISTRY_PATH.exists():
        save_json({}, REGISTRY_PATH)
    zpath = Path("/kaggle/working") / zip_name
    if zpath.exists():
        zpath.unlink()
    n_files = raw = n_skipped = skipped_bytes = 0
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(BASE.rglob("*")):
            if not p.is_file():
                continue
            skip = (p.name in BACKUP_EXCLUDE_NAMES or p.name.endswith(".tmp")
                    or ((not include_weights) and p.name.endswith(BACKUP_EXCLUDE_SUFFIXES)))
            if skip:
                n_skipped += 1; skipped_bytes += p.stat().st_size; continue
            zf.write(p, p.relative_to(BASE.parent)); n_files += 1; raw += p.stat().st_size
    print(f"Backup written : {zpath}")
    print(f"Files stored   : {n_files} | uncompressed {raw/1e6:.1f} MB | zip {zpath.stat().st_size/1e6:.1f} MB")
    print(f"Skipped        : {n_skipped} files ({skipped_bytes/1e6:.0f} MB: image cache and, unless include_weights=True, model weights)")
    return zpath

zpath = backup_project_zip()

with zipfile.ZipFile(zpath) as zf:
    assert zf.testzip() is None, "zip is corrupt"
    names = set(zf.namelist())
expected = ["logs/config_base.json", "logs/run_registry.json", "logs/leakage_verification.json", "logs/augmentation_config.json",
            "logs/custom_cnn_diagnostic.json", "splits/splits.json", "data/clean_index.csv", "data/removed_images.csv",
            "results/model_parameter_table.csv", "results/custom_cnn_diagnostic.csv"]
print("\nContents check:")
for e in expected:
    print(f"   [{'OK     ' if 'brain_tumor_project/' + e in names else 'MISSING'}] {e}")
n_runs = sum(n.startswith("brain_tumor_project/results/runs/") for n in names)
n_figs = sum(n.startswith("brain_tumor_project/figures/") for n in names)
print(f"\nRun-result files stored: {n_runs} | figures stored: {n_figs} | runs in registry: {len(list_runs())}")
print("\nDownload: right-hand panel -> Data -> Output -> /kaggle/working -> project_backup.zip -> three dots -> Download.")
print("From now on use THIS cell for backups (re-running CELL 19 would restore the old behaviour that includes weights).")

In [ ]:
# ============================================================
# CELL 38: STAGE G SETUP - FINAL HYPERPARAMETERS, run_stage_g(), DISK MONITOR
# ============================================================

TUNED_HP = load_json(DIRS["tuning"] / "best_hyperparameters.json")
_ac = load_json(DIRS["logs"] / "augmentation_choice.json")
assert TUNED_HP is not None and _ac is not None, "run CELLs 36 and 37 first"
AUG_CHOICE = _ac["choice"]
_fr = load_json(DIRS["splits"] / "splits_frozen.json")
assert _fr is not None and _fr["signature"] == DATA_SIGNATURE, "splits are not frozen or have changed"

def final_overrides(arch):
    """Tuned hyperparameters (Stage F) + the augmentation preset chosen in CELL 37. Training budgets stay at the full defaults."""
    return {**TUNED_HP[arch]["overrides"], "aug": AUG_CHOICE}

FINAL_HP = {a: get_hp(a, **final_overrides(a)) for a in ARCHITECTURES}
save_json(FINAL_HP, DIRS["logs"] / "final_hyperparameters.json")

print(f"Augmentation preset: {AUG_CHOICE}")
print("Final hyperparameters per architecture (tuned values shown; all other settings are the CELL 22 defaults):")
for a in ARCHITECTURES:
    hp = FINAL_HP[a]
    budget = f"up to {hp['epochs']} epochs (patience {hp['patience']})" if a == "custom_cnn" else \
             f"up to {hp['epochs_head']}+{hp['epochs_ft']} epochs (patience {hp['patience_head']}/{hp['patience']})"
    print(f"  {a:15s} tuned trial {TUNED_HP[a]['trial']} | overrides {TUNED_HP[a]['overrides'] or 'defaults'} | {budget}")

def disk_report(tag=""):
    du = shutil.disk_usage("/kaggle/working")
    proj = sum(p.stat().st_size for p in BASE.rglob("*") if p.is_file())
    ck = sum(p.stat().st_size for p in DIRS["checkpoints"].glob("*") if p.is_file())
    print(f"[disk{tag}] /kaggle/working used {du.used/1e9:.1f} of {du.total/1e9:.1f} GB | "
          f"project {proj/1e9:.2f} GB (checkpoints {ck/1e9:.2f} GB)")

def run_stage_g(split, activation):
    """Trains (or skips, if already registered) all 5 architectures for one split and one activation. Validation only."""
    assert split in SPLIT_CONFIGS and activation in ACTIVATIONS
    s = get_split(split); t0 = time.time()
    for arch in ARCHITECTURES:
        train_and_evaluate(make_run_id("init", split, arch, activation), arch, activation, split,
                           s["fit_idx"], s["val_idx"], hp_overrides=final_overrides(arch), verbose=0)
    rows = []
    for arch in ARCHITECTURES:
        r = load_result(make_run_id("init", split, arch, activation)); sm = r["summary"]; hp = r["hp"]
        hit = (r["epochs_stage1"] >= hp["epochs"]) if arch == "custom_cnn" else (r["epochs_stage2"] >= hp["epochs_ft"])
        rows.append({"arch": arch, "val_acc": sm["val_accuracy"], "val_f1": sm["val_f1_macro"], "val_loss": sm["val_loss"],
                     "val_auc": sm["val_roc_auc_macro"], "train_acc": sm["train_accuracy"], "gap_acc": sm["gap_accuracy"],
                     "epochs": f"{r['epochs_stage1']}+{r['epochs_stage2']}", "best_ep": sm["best_epoch"],
                     "hit_epoch_cap": bool(hit), "min": round(sm["train_time_s"] / 60, 1)})
    out = pd.DataFrame(rows)
    print(f"\n=== Stage G | split {split} | activation {activation.upper()} | validation set n={len(s['val_idx'])} ===")
    print(out.round(4).to_string(index=False))
    print(f"(cell wall time {(time.time()-t0)/60:.1f} min; skipped runs count as ~0)")
    disk_report(f" after {split}/{activation}")
    return out

disk_report(" now")
print("\nStage G ready. Run CELLs 39-41 (ReLU, Phase A), then CELLs 42-44 (GELU, Phase B), one at a time.")

In [ ]:
# ============================================================
# CELL 39: STAGE G PHASE A - RELU - SPLIT 70:30 (5 ARCHITECTURES)
# ============================================================
G_70_30_relu = run_stage_g("70_30", "relu")

In [ ]:
# ============================================================
# CELL 40: STAGE G PHASE A - RELU - SPLIT 80:20 (5 ARCHITECTURES)
# ============================================================
G_80_20_relu = run_stage_g("80_20", "relu")

In [ ]:
# ============================================================
# CELL 41: STAGE G PHASE A - RELU - SPLIT 90:10 (5 ARCHITECTURES)
# ============================================================
G_90_10_relu = run_stage_g("90_10", "relu")

In [ ]:
# ============================================================
# CELL 42: STAGE G PHASE B - GELU - SPLIT 70:30 (5 ARCHITECTURES)
# ============================================================
G_70_30_gelu = run_stage_g("70_30", "gelu")

In [ ]:
# ============================================================
# CELL 43: STAGE G PHASE B - GELU - SPLIT 80:20 (5 ARCHITECTURES)
# ============================================================
G_80_20_gelu = run_stage_g("80_20", "gelu")

In [ ]:
# ============================================================
# CELL 44: STAGE G PHASE B - GELU - SPLIT 90:10 (5 ARCHITECTURES)
# ============================================================
G_90_10_gelu = run_stage_g("90_10", "gelu")

In [ ]:
# ============================================================
# CELL 45: STAGE H - VALIDATION COMPARISON, RELU vs GELU, BEST-TWO SELECTION (NO TEST DATA)
# ============================================================
import itertools
disp = lambda a: ARCH_SPECS[a]["display"]

# ---- 0. Is the 70:30 validation set the same one Stage F tuned on? ----
tu, s70 = get_tuning_split(), get_split("70_30")
same_val = np.array_equal(np.sort(s70["val_idx"]), np.sort(tu["val_idx"]))
same_fit = np.array_equal(np.sort(s70["fit_idx"]), np.sort(tu["fit_idx"]))
print(f"70:30 validation set identical to the Stage F tuning validation set: {same_val} (fit sets identical: {same_fit})")
if same_val:
    print("   -> 70:30 validation scores of tuned architectures (VGG19, DenseNet121, EfficientNetB0) are optimistically biased.")

# ---- 1. Collect all 30 Stage G runs ----
rows = []
for sp in SPLIT_CONFIGS:
    for act in ACTIVATIONS:
        for arch in ARCHITECTURES:
            r = load_result(make_run_id("init", sp, arch, act))
            if r is None: raise RuntimeError(f"Missing run init {sp} {arch} {act}: run CELLs 39-44 first.")
            v, s = r["metrics"]["val"], r["summary"]
            rows.append({"split": sp, "activation": act, "arch": arch, "val_acc": v["accuracy"],
                         "val_prec_macro": v["precision_macro"], "val_rec_macro": v["recall_macro"], "val_f1": v["f1_macro"],
                         "val_f1_weighted": v["f1_weighted"], "val_loss": v["loss"], "val_auc": v["roc_auc_macro"],
                         "val_kappa": v["kappa"], "train_acc": s["train_accuracy"], "train_loss": s["train_loss"],
                         "gap_acc": s["gap_accuracy"], "gap_loss": s["gap_loss"], "epochs": s["epochs_run"],
                         "best_epoch": s["best_epoch"], "conv_epoch": s["conv_epoch"], "params_M": s["params_total"] / 1e6,
                         "train_min": s["train_time_s"] / 60, "latency_ms": s["latency_ms_bs1"],
                         "throughput": s["throughput_img_s_bs32"]})
S = pd.DataFrame(rows); assert len(S) == 30
S["f1_r3"] = S["val_f1"].round(3)
S.to_csv(DIRS["results"] / "stage_g_validation_table.csv", index=False)

cols = ["activation", "arch", "val_acc", "val_prec_macro", "val_rec_macro", "val_f1", "val_loss", "val_auc",
        "train_acc", "gap_acc", "epochs", "best_epoch", "train_min"]
for sp in SPLIT_CONFIGS:
    print(f"\n=== Stage G validation results | split {sp} | validation n={len(get_split(sp)['val_idx'])} ===")
    print(S[S.split == sp][cols].round(4).to_string(index=False))

# ---- 2. Overfitting / underfitting (descriptive thresholds I chose; not statistical tests) ----
def fit_label(r):
    if r.train_acc < 0.85: return "underfit-leaning (clean train acc < 0.85)"
    if r.gap_acc > 0.08:   return "overfit-leaning (train-val gap > 0.08)"
    return "balanced"
S["fit_status"] = S.apply(fit_label, axis=1)
print("\nFit status per architecture (count of the 6 runs each):")
print(S.groupby(["arch", "fit_status"]).size().unstack(fill_value=0).to_string())

# ---- 3. ReLU vs GELU: paired differences (GELU - ReLU) ----
P = S.pivot_table(index=["split", "arch"], columns="activation", values=["val_f1", "val_loss", "val_auc", "best_epoch"])
D = pd.DataFrame({"d_f1": P["val_f1"]["gelu"] - P["val_f1"]["relu"], "d_loss": P["val_loss"]["gelu"] - P["val_loss"]["relu"],
                  "d_auc": P["val_auc"]["gelu"] - P["val_auc"]["relu"], "d_best_epoch": P["best_epoch"]["gelu"] - P["best_epoch"]["relu"]})
print("\nPaired differences GELU - ReLU (positive F1/AUC = GELU better; negative loss = GELU better):")
print(D.round(4).to_string())
print("\nMean difference per architecture (val macro-F1):")
print(D.groupby(level="arch")["d_f1"].mean().loc[ARCHITECTURES].round(4).to_string())
v = D["d_f1"].values; obs = v.mean()
perm = (np.array(list(itertools.product([-1, 1], repeat=len(v)))) * v).mean(axis=1)
p_perm = float((np.abs(perm) >= abs(obs) - 1e-12).mean())
noise = (load_json(DIRS["logs"] / "augmentation_choice.json") or {}).get("seed_noise_mean_abs_f1", float("nan"))
print(f"\nOverall mean F1 difference (GELU - ReLU): {obs:+.4f} | GELU better in {(v>0).sum()}, ReLU better in {(v<0).sum()}, ties {(v==0).sum()} of {len(v)}")
print(f"Exact sign-flip permutation test on the 15 pairs: p = {p_perm:.3f}  (EXPLORATORY: pairs are not independent)")
print(f"Reference: mean seed-to-seed |F1 difference| for identical configurations (CELL 37) = {noise:.4f}")
print("No activation winner is declared from single-seed validation runs; cross-validation evidence will be added in Stage K.")
D.to_csv(DIRS["results"] / "relu_vs_gelu_paired.csv")

# ---- 4. Best-two selection per split (rule fixed in the design: better activation variant per architecture,
#         top two DISTINCT architectures by validation macro-F1; F1 equal to 3 decimals counts as a tie -> lower validation loss) ----
SEL = {"rule": "per split: each architecture represented by its better activation variant; rank by validation macro-F1 "
               "(equal to 3 decimals = tie -> lower validation loss); top two architectures",
       "data": "validation sets only; test sets locked", "splits": {}}
print("\n" + "=" * 100); print("BEST-TWO SELECTION (validation evidence only)"); print("=" * 100)
for sp in SPLIT_CONFIGS:
    d = S[S.split == sp].sort_values(["f1_r3", "val_loss"], ascending=[False, True])
    R = d.groupby("arch", sort=False).head(1).sort_values(["f1_r3", "val_loss"], ascending=[False, True]).reset_index(drop=True)
    top2 = list(R["arch"][:2]); extra = [a for a in top2 if a not in MANDATORY_CV_ARCHS]
    gap3 = float(R["val_f1"][1] - R["val_f1"][2])
    print(f"\nSplit {sp}:")
    print(R[["arch", "activation", "val_f1", "val_loss", "val_acc"]].round(4).to_string(index=True))
    print(f"   best two: {top2} | gap between 2nd and 3rd place: {gap3:.4f}"
          + ("  <- within run-to-run noise" if gap3 < 0.02 else ""))
    print(f"   extra CV needed beyond the mandatory EfficientNetB0 + DenseNet121: {extra if extra else 'none (Stage J not needed for this split)'}")
    SEL["splits"][sp] = {"ranking": R[["arch", "activation", "val_f1", "val_loss"]].to_dict("records"), "top_two": top2,
                         "extra_cv_needed": extra, "third_place_gap_to_second": gap3,
                         "best_activation": {r.arch: r.activation for r in R.itertuples()}}
save_json(SEL, DIRS["logs"] / "stage_h_selection.json")
print("\nCV activation per mandatory architecture (validation winner; weakly supported given the noise above):")
for sp in SPLIT_CONFIGS:
    print(f"   {sp}: " + ", ".join(f"{disp(a)} -> {SEL['splits'][sp]['best_activation'][a].upper()}" for a in MANDATORY_CV_ARCHS))
print("Saved logs/stage_h_selection.json (used by the CV cells).")

# ---- 5. Figures ----
S["cfg"] = S["split"].str.replace("_", ":") + " " + S["activation"].str.upper()
cfgs = [f"{sp.replace('_', ':')} {a.upper()}" for sp in SPLIT_CONFIGS for a in ACTIVATIONS]
fig, axes = plt.subplots(2, 2, figsize=(19, 11))
for ax, (col, lab) in zip(axes.ravel(), [("val_acc", "Accuracy"), ("val_prec_macro", "Macro precision"),
                                         ("val_rec_macro", "Macro recall"), ("val_f1", "Macro F1-score")]):
    t = S.pivot(index="arch", columns="cfg", values=col).loc[ARCHITECTURES, cfgs]; t.index = [disp(a) for a in t.index]
    t.plot(kind="bar", ax=ax, edgecolor="black", rot=0, width=0.85)
    ax.set_title(f"Validation {lab.lower()} by model (split and activation)", fontsize=13)
    ax.set_xlabel("Architecture", fontsize=12); ax.set_ylabel(lab, fontsize=12); ax.set_ylim(0.5, 1.0); ax.legend(fontsize=8, ncol=3)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "16_stage_g_model_metrics.png", dpi=150, bbox_inches="tight"); plt.show()

dt = D["d_f1"].unstack("split").loc[ARCHITECTURES]; dt.index = [disp(a) for a in dt.index]; dt.columns = [c.replace("_", ":") for c in dt.columns]
plot_bar_compare(dt, "ReLU vs GELU: validation macro-F1 difference (GELU - ReLU)", "F1 difference",
                 DIRS["figures"] / "17_relu_vs_gelu_difference.png", xlabel="Architecture")
bs = S.groupby(["arch", "split"])["val_f1"].max().unstack().loc[ARCHITECTURES]; bs.index = [disp(a) for a in bs.index]; bs.columns = [c.replace("_", ":") for c in bs.columns]
plot_bar_compare(bs, "Best-variant validation macro-F1 per split (different validation sets: do not rank splits from this)",
                 "Validation macro-F1", DIRS["figures"] / "18_performance_across_splits.png", ylim=(0.5, 1.0), xlabel="Architecture")

eff = S.groupby("arch").agg(params_M=("params_M", "first"), train_min=("train_min", "mean"), latency_ms=("latency_ms", "mean")).loc[ARCHITECTURES]
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, (col, lab, c) in zip(axes, [("params_M", "Parameters (millions)", "#4c72b0"), ("train_min", "Mean training time (min)", "#dd8452"),
                                    ("latency_ms", "GPU inference latency, batch 1 (ms)", "#55a868")]):
    ax.bar([disp(a) for a in eff.index], eff[col], color=c, edgecolor="black")
    for i, val in enumerate(eff[col]): ax.text(i, val, f"{val:.2f}", ha="center", va="bottom", fontsize=10)
    ax.set_title(lab, fontsize=13); ax.set_xlabel("Architecture", fontsize=12); ax.set_ylabel(lab, fontsize=12); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "19_efficiency_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

hdir = DIRS["figures"] / "history"; hdir.mkdir(exist_ok=True)
for r in S.itertuples():
    rid = make_run_id("init", r.split, r.arch, r.activation)
    plot_history(pd.read_csv(run_paths(rid)["history"]), f"{disp(r.arch)} {r.activation.upper()} {r.split}", hdir / f"{rid}.png", show=False)
print(f"\nSaved 30 training/validation history plots to {hdir}")
plot_history(pd.read_csv(run_paths(make_run_id("init", "90_10", "efficientnetb0", "relu"))["history"]),
             "EfficientNetB0 ReLU 90:10 (the one run that hit the epoch cap)", show=True)
print("\nStage H complete. The test sets were never touched.")

In [ ]:
# ============================================================
# CELL 29: PROJECT BACKUP V2 (SUPERSEDES CELL 19; EXCLUDES MODEL-WEIGHT FILES)
# ============================================================

BACKUP_EXCLUDE_NAMES = {"X_clean_224.npy"}                 # re-created by CELL 12
BACKUP_EXCLUDE_SUFFIXES = (".weights.h5", ".keras", ".tmp")  # model weights are large; histories/results/probs are kept

def backup_project_zip(zip_name: str = "project_backup.zip", include_weights: bool = False):
    """Zips the project folder into /kaggle/working/<zip_name>. By default model-weight files and the image cache are skipped.
    Use include_weights=True only when you specifically want the weights (e.g. the final model in Stage N)."""
    if not REGISTRY_PATH.exists():
        save_json({}, REGISTRY_PATH)
    zpath = Path("/kaggle/working") / zip_name
    if zpath.exists():
        zpath.unlink()
    n_files = raw = n_skipped = skipped_bytes = 0
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(BASE.rglob("*")):
            if not p.is_file():
                continue
            skip = (p.name in BACKUP_EXCLUDE_NAMES or p.name.endswith(".tmp")
                    or ((not include_weights) and p.name.endswith(BACKUP_EXCLUDE_SUFFIXES)))
            if skip:
                n_skipped += 1; skipped_bytes += p.stat().st_size; continue
            zf.write(p, p.relative_to(BASE.parent)); n_files += 1; raw += p.stat().st_size
    print(f"Backup written : {zpath}")
    print(f"Files stored   : {n_files} | uncompressed {raw/1e6:.1f} MB | zip {zpath.stat().st_size/1e6:.1f} MB")
    print(f"Skipped        : {n_skipped} files ({skipped_bytes/1e6:.0f} MB: image cache and, unless include_weights=True, model weights)")
    return zpath

zpath = backup_project_zip()

with zipfile.ZipFile(zpath) as zf:
    assert zf.testzip() is None, "zip is corrupt"
    names = set(zf.namelist())
expected = ["logs/config_base.json", "logs/run_registry.json", "logs/leakage_verification.json", "logs/augmentation_config.json",
            "logs/custom_cnn_diagnostic.json", "splits/splits.json", "data/clean_index.csv", "data/removed_images.csv",
            "results/model_parameter_table.csv", "results/custom_cnn_diagnostic.csv"]
print("\nContents check:")
for e in expected:
    print(f"   [{'OK     ' if 'brain_tumor_project/' + e in names else 'MISSING'}] {e}")
n_runs = sum(n.startswith("brain_tumor_project/results/runs/") for n in names)
n_figs = sum(n.startswith("brain_tumor_project/figures/") for n in names)
print(f"\nRun-result files stored: {n_runs} | figures stored: {n_figs} | runs in registry: {len(list_runs())}")
print("\nDownload: right-hand panel -> Data -> Output -> /kaggle/working -> project_backup.zip -> three dots -> Download.")
print("From now on use THIS cell for backups (re-running CELL 19 would restore the old behaviour that includes weights).")

In [ ]:
# ============================================================
# CELL 46: STAGE I SETUP - CROSS-VALIDATION ENGINE (PRINTS EVERY FOLD AND A FULL SUMMARY)
# ============================================================
SEL = load_json(DIRS["logs"] / "stage_h_selection.json")
assert SEL is not None, "run CELL 45 first"
KEEP_CV_CHECKPOINTS = False       # fold checkpoints are not needed afterwards (held-out predictions are saved) and would use ~GBs
CV_DIR = DIRS["cv"]; CV_SUMMARIES = {}

def cv_activation(split, arch):
    return SEL["splits"][split]["best_activation"][arch]

def summarize_cv(split, arch, activation):
    K = len(get_cv_folds(split)); rows, pcs, ys, ps, gs = [], [], [], [], []
    for k in range(K):
        rid = make_run_id("cv", split, arch, activation, fold=k); r = load_result(rid)
        if r is None: raise RuntimeError(f"missing CV result {rid}")
        m, s, hp = r["metrics"]["heldout"], r["summary"], r["hp"]
        hit = (r["epochs_stage1"] >= hp["epochs"]) if ARCH_SPECS[arch]["kind"] == "scratch" else (r["epochs_stage2"] >= hp["epochs_ft"])
        rows.append({"fold": k, "n": m["n"], "acc": m["accuracy"], "prec_macro": m["precision_macro"], "prec_w": m["precision_weighted"],
                     "rec_macro": m["recall_macro"], "rec_w": m["recall_weighted"], "f1_macro": m["f1_macro"], "f1_w": m["f1_weighted"],
                     "loss": m["loss"], "kappa": m["kappa"], "auc_macro": m["roc_auc_macro"], "train_acc_clean": s["train_accuracy"],
                     "epochs": s["epochs_run"], "hit_cap": bool(hit), "train_min": s["train_time_s"] / 60})
        for c, d in m["per_class"].items():
            pcs.append({"fold": k, "class": c, "sensitivity": d["sensitivity"], "specificity": d["specificity"], "f1": d["f1"]})
        z = np.load(run_paths(rid)["probs"]); idx = z["heldout_idx"]
        ys.append(y_all[idx]); ps.append(z["heldout_probs"]); gs.append(groups_all[idx])
    F, PC = pd.DataFrame(rows), pd.DataFrame(pcs)
    y, Pr, g = np.concatenate(ys), np.concatenate(ps), np.concatenate(gs)
    pm = compute_metrics(y, Pr); ci = cluster_bootstrap_ci(y, Pr.argmax(1), groups=g, n_boot=1000)
    tag = f"{split}__{arch}__{activation}"; title = f"{disp(arch)} {activation.upper()} {split} ({K}-fold CV)"

    print(f"\n{'='*100}\nCV RESULTS: {title}\n{'='*100}")
    print("Fold-wise HELD-OUT metrics (each fold scored once by its own model; never used for early stopping):")
    print(F.round(4).to_string(index=False))
    mcols = ["acc", "prec_macro", "rec_macro", "f1_macro", "f1_w", "loss", "kappa", "auc_macro", "train_min"]
    ms = F[mcols].agg(["mean", "std"]).T
    print("\nMean +/- SD over folds (SD = sample standard deviation):")
    print(pd.DataFrame({"mean +/- sd": [f"{a:.4f} +/- {b:.4f}" for a, b in zip(ms["mean"], ms["std"])]}, index=mcols).to_string())
    print(f"folds that hit the epoch cap: {int(F['hit_cap'].sum())} of {K}")
    pcm = PC.groupby("class")[["sensitivity", "specificity", "f1"]].mean().loc[CLASS_NAMES]
    print("\nPer-class metrics (mean over folds):"); print(pcm.round(4).to_string())
    cmdf = pd.DataFrame(pm["confusion_matrix"], index=[f"true {CLASS_DISPLAY[i]}" for i in range(4)],
                        columns=[f"pred {CLASS_DISPLAY[i]}" for i in range(4)])
    print(f"\nPOOLED out-of-fold confusion matrix (all {len(y)} held-out images, each predicted once):"); print(cmdf.to_string())
    print(f"Pooled: acc {pm['accuracy']:.4f} [95% CI {ci['accuracy'][0]:.3f}-{ci['accuracy'][1]:.3f}] | macro-F1 {pm['f1_macro']:.4f} "
          f"[{ci['f1_macro'][0]:.3f}-{ci['f1_macro'][1]:.3f}] | kappa {pm['kappa']:.4f} | macro ROC-AUC {pm['roc_auc_macro']:.4f} | loss {pm['loss']:.4f}")
    print("(CIs resample duplicate groups, not single images.)")

    F.to_csv(CV_DIR / f"{tag}_folds.csv", index=False); PC.to_csv(CV_DIR / f"{tag}_perclass.csv", index=False)
    cmdf.to_csv(CV_DIR / f"{tag}_pooled_confusion.csv")
    ms.to_csv(CV_DIR / f"{tag}_mean_sd.csv")
    save_json({"split": split, "arch": arch, "activation": activation, "folds": K, "pooled": {k: v for k, v in pm.items()},
               "pooled_ci": ci, "mean": ms["mean"].to_dict(), "sd": ms["std"].to_dict()}, CV_DIR / f"{tag}_summary.json")

    fig, ax = plt.subplots(figsize=(max(8, K * 0.9), 4.6)); x = np.arange(K)
    ax.bar(x - 0.2, F["acc"], 0.4, label="Accuracy", edgecolor="black"); ax.bar(x + 0.2, F["f1_macro"], 0.4, label="Macro-F1", edgecolor="black")
    ax.axhline(F["acc"].mean(), color="C0", ls="--", lw=1); ax.axhline(F["f1_macro"].mean(), color="C1", ls="--", lw=1)
    ax.set_xticks(x); ax.set_xticklabels([str(k + 1) for k in range(K)]); ax.set_ylim(max(0.0, F[["acc", "f1_macro"]].min().min() - 0.1), 1.0)
    ax.set_title(f"{title}: held-out performance per fold (dashed = mean)", fontsize=12); ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel("Score", fontsize=12); ax.legend()
    plt.tight_layout(); plt.savefig(CV_DIR / f"{tag}_folds.png", dpi=150, bbox_inches="tight"); plt.show()
    plot_confusion(pm["confusion_matrix"], f"{title} pooled", CV_DIR / f"{tag}_pooled_confusion.png", show=True)
    return {"folds": F, "perclass": PC, "pooled": pm, "pooled_ci": ci, "mean_sd": ms, "activation": activation}

def run_cv(split, arch, activation=None):
    activation = activation or cv_activation(split, arch)
    folds = get_cv_folds(split); K = len(folds); t0 = time.time()
    print(f"CV: {disp(arch)} | split {split} | {K}-fold on the {len(get_split(split)['train_idx'])}-image training portion | "
          f"activation {activation.upper()} (Stage H validation winner) | tuned hyperparameters | augmentation '{AUG_CHOICE}'")
    print("(Each '[done]' line shows the INNER validation score used for early stopping; the 'HELD-OUT' line is the fold result.)")
    for f in folds:
        rid = make_run_id("cv", split, arch, activation, fold=f["fold"])
        r = train_and_evaluate(rid, arch, activation, split, f["fit_idx"], f["inner_val_idx"], hp_overrides=final_overrides(arch),
                               extra_eval={"heldout": f["heldout_idx"]}, verbose=0)
        m = r["metrics"]["heldout"]
        print(f"   fold {f['fold']+1}/{K}: HELD-OUT acc {m['accuracy']:.4f} | macro-F1 {m['f1_macro']:.4f} | loss {m['loss']:.4f} | "
              f"n={m['n']} | epochs {r['summary']['epochs_run']} | {r['summary']['train_time_s']/60:.1f} min")
        if not KEEP_CV_CHECKPOINTS:
            pth = run_paths(rid)["ckpt"]
            if pth.exists(): pth.unlink()
    out = summarize_cv(split, arch, activation)
    CV_SUMMARIES[(split, arch)] = out
    print(f"\n(CV cell wall time {(time.time()-t0)/60:.1f} min; skipped folds count as ~0)")
    disk_report(f" after CV {split}/{arch}")
    return out

print("CV engine ready. Activations that will be used:")
for sp in SPLIT_CONFIGS:
    print(f"   {sp} ({SPLIT_CONFIGS[sp]['cv_folds']}-fold): " + ", ".join(f"{disp(a)} {cv_activation(sp, a).upper()}" for a in MANDATORY_CV_ARCHS))

In [ ]:
# ============================================================
# CELL 47: STAGE I - 3-FOLD STRATIFIED CV - EFFICIENTNETB0 - SPLIT 70:30
# ============================================================
cv_70_30_eff = run_cv("70_30", "efficientnetb0")

In [ ]:
# ============================================================
# CELL 48: STAGE I - 3-FOLD STRATIFIED CV - DENSENET121 - SPLIT 70:30
# ============================================================
cv_70_30_dense = run_cv("70_30", "densenet121")

In [ ]:
# ============================================================
# CELL 49: STAGE I - 5-FOLD STRATIFIED CV - EFFICIENTNETB0 - SPLIT 80:20
# ============================================================
cv_80_20_eff = run_cv("80_20", "efficientnetb0")

In [ ]:
# ============================================================
# CELL 50: STAGE I - 5-FOLD STRATIFIED CV - DENSENET121 - SPLIT 80:20
# ============================================================
cv_80_20_dense = run_cv("80_20", "densenet121")

In [ ]:
# ============================================================
# CELL 51: STAGE I - 10-FOLD STRATIFIED CV - EFFICIENTNETB0 - SPLIT 90:10
# ============================================================
cv_90_10_eff = run_cv("90_10", "efficientnetb0")

In [ ]:
# ============================================================
# CELL 52: STAGE I - 10-FOLD STRATIFIED CV - DENSENET121 - SPLIT 90:10
# ============================================================
cv_90_10_dense = run_cv("90_10", "densenet121")

In [ ]:
# ============================================================
# CELL 29: PROJECT BACKUP V2 (SUPERSEDES CELL 19; EXCLUDES MODEL-WEIGHT FILES)
# ============================================================

BACKUP_EXCLUDE_NAMES = {"X_clean_224.npy"}                 # re-created by CELL 12
BACKUP_EXCLUDE_SUFFIXES = (".weights.h5", ".keras", ".tmp")  # model weights are large; histories/results/probs are kept

def backup_project_zip(zip_name: str = "project_backup.zip", include_weights: bool = False):
    """Zips the project folder into /kaggle/working/<zip_name>. By default model-weight files and the image cache are skipped.
    Use include_weights=True only when you specifically want the weights (e.g. the final model in Stage N)."""
    if not REGISTRY_PATH.exists():
        save_json({}, REGISTRY_PATH)
    zpath = Path("/kaggle/working") / zip_name
    if zpath.exists():
        zpath.unlink()
    n_files = raw = n_skipped = skipped_bytes = 0
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(BASE.rglob("*")):
            if not p.is_file():
                continue
            skip = (p.name in BACKUP_EXCLUDE_NAMES or p.name.endswith(".tmp")
                    or ((not include_weights) and p.name.endswith(BACKUP_EXCLUDE_SUFFIXES)))
            if skip:
                n_skipped += 1; skipped_bytes += p.stat().st_size; continue
            zf.write(p, p.relative_to(BASE.parent)); n_files += 1; raw += p.stat().st_size
    print(f"Backup written : {zpath}")
    print(f"Files stored   : {n_files} | uncompressed {raw/1e6:.1f} MB | zip {zpath.stat().st_size/1e6:.1f} MB")
    print(f"Skipped        : {n_skipped} files ({skipped_bytes/1e6:.0f} MB: image cache and, unless include_weights=True, model weights)")
    return zpath

zpath = backup_project_zip()

with zipfile.ZipFile(zpath) as zf:
    assert zf.testzip() is None, "zip is corrupt"
    names = set(zf.namelist())
expected = ["logs/config_base.json", "logs/run_registry.json", "logs/leakage_verification.json", "logs/augmentation_config.json",
            "logs/custom_cnn_diagnostic.json", "splits/splits.json", "data/clean_index.csv", "data/removed_images.csv",
            "results/model_parameter_table.csv", "results/custom_cnn_diagnostic.csv"]
print("\nContents check:")
for e in expected:
    print(f"   [{'OK     ' if 'brain_tumor_project/' + e in names else 'MISSING'}] {e}")
n_runs = sum(n.startswith("brain_tumor_project/results/runs/") for n in names)
n_figs = sum(n.startswith("brain_tumor_project/figures/") for n in names)
print(f"\nRun-result files stored: {n_runs} | figures stored: {n_figs} | runs in registry: {len(list_runs())}")
print("\nDownload: right-hand panel -> Data -> Output -> /kaggle/working -> project_backup.zip -> three dots -> Download.")
print("From now on use THIS cell for backups (re-running CELL 19 would restore the old behaviour that includes weights).")

In [ ]:
# ============================================================
# CELL 54: STAGE K1 - AGGREGATE ALL CROSS-VALIDATION RESULTS (6 MANDATORY JOBS)
# ============================================================
SEL = load_json(DIRS["logs"] / "stage_h_selection.json")
assert SEL is not None, "run CELL 45 first (logs/stage_h_selection.json is missing)"
CV_DIR = DIRS["cv"]
dname = lambda a: ARCH_SPECS[a]["display"]
ARCH_COL = {"efficientnetb0": "#4c72b0", "densenet121": "#dd8452"}
CELL_OF = {("70_30", "efficientnetb0"): 47, ("70_30", "densenet121"): 48, ("80_20", "efficientnetb0"): 49,
           ("80_20", "densenet121"): 50, ("90_10", "efficientnetb0"): 51, ("90_10", "densenet121"): 52}
SPLIT_LABEL = {sp: f"{sp.replace('_', ':')} ({SPLIT_CONFIGS[sp]['cv_folds']}-fold)" for sp in SPLIT_CONFIGS}

# ---- 1. Load every CV job ----
fold_parts, pc_parts, pooled_rows, missing = [], [], [], []
for sp in SPLIT_CONFIGS:
    for arch in MANDATORY_CV_ARCHS:
        act = SEL["splits"][sp]["best_activation"][arch]
        tag = f"{sp}__{arch}__{act}"
        p_f, p_pc, p_js = CV_DIR / f"{tag}_folds.csv", CV_DIR / f"{tag}_perclass.csv", CV_DIR / f"{tag}_summary.json"
        if not (p_f.exists() and p_pc.exists() and p_js.exists()):
            missing.append(f"{dname(arch)} on {sp} (CELL {CELL_OF[(sp, arch)]})"); continue
        f = pd.read_csv(p_f)
        assert len(f) == SPLIT_CONFIGS[sp]["cv_folds"], f"{tag}: expected {SPLIT_CONFIGS[sp]['cv_folds']} folds, found {len(f)}"
        for df_, parts in ((f, fold_parts), (pd.read_csv(p_pc), pc_parts)):
            df_.insert(0, "activation", act); df_.insert(0, "arch", arch); df_.insert(0, "split", sp); parts.append(df_)
        js = load_json(p_js); pm, ci = js["pooled"], js["pooled_ci"]
        pooled_rows.append({"split": sp, "arch": arch, "activation": act, "K": js["folds"], "n": pm["n"],
                            "pooled_acc": pm["accuracy"], "acc_ci_lo": ci["accuracy"][0], "acc_ci_hi": ci["accuracy"][1],
                            "pooled_f1": pm["f1_macro"], "f1_ci_lo": ci["f1_macro"][0], "f1_ci_hi": ci["f1_macro"][1],
                            "kappa": pm["kappa"], "auc_macro": pm["roc_auc_macro"], "loss": pm["loss"]})
if missing:
    raise RuntimeError("Cross-validation results are missing for: " + "; ".join(missing) + ". Run those cells first.")

FOLDS, PC, POOL = pd.concat(fold_parts, ignore_index=True), pd.concat(pc_parts, ignore_index=True), pd.DataFrame(pooled_rows)
FOLDS["gap_acc"] = FOLDS["train_acc_clean"] - FOLDS["acc"]
FOLDS.to_csv(DIRS["results"] / "cv_all_folds.csv", index=False)
PC.to_csv(DIRS["results"] / "cv_all_perclass.csv", index=False)
POOL.to_csv(DIRS["results"] / "cv_pooled_table.csv", index=False)
print(f"Loaded {len(FOLDS)} fold results ({FOLDS.groupby(['split', 'arch']).ngroups} jobs, all six present).")
print("Activations used in CV:", {f"{r.split}/{dname(r.arch)}": r.activation.upper() for r in POOL.itertuples()})

# ---- 2. Mean +/- SD tables ----
MET = ["acc", "prec_macro", "rec_macro", "f1_macro", "f1_w", "loss", "kappa", "auc_macro", "train_acc_clean", "gap_acc", "train_min"]
rows, num = [], []
for sp in SPLIT_CONFIGS:
    for arch in MANDATORY_CV_ARCHS:
        d = FOLDS[(FOLDS.split == sp) & (FOLDS.arch == arch)]
        r = {"split": sp, "model": f"{dname(arch)} {d.activation.iloc[0].upper()}", "K": len(d)}
        n = {"split": sp, "arch": arch, "activation": d.activation.iloc[0], "K": len(d)}
        for m in MET:
            r[m] = f"{d[m].mean():.4f} +/- {d[m].std(ddof=1):.4f}"; n[f"{m}_mean"] = d[m].mean(); n[f"{m}_sd"] = d[m].std(ddof=1)
        rows.append(r); num.append(n)
MS = pd.DataFrame(num); MS.to_csv(DIRS["results"] / "cv_mean_sd_table.csv", index=False)
T = pd.DataFrame(rows)
print("\nCROSS-VALIDATION, held-out folds: mean +/- sample SD over folds (core metrics)")
print(T[["split", "model", "K", "acc", "prec_macro", "rec_macro", "f1_macro", "f1_w"]].to_string(index=False))
print("\nCROSS-VALIDATION, loss / agreement / generalisation gap / time")
print(T[["split", "model", "loss", "kappa", "auc_macro", "train_acc_clean", "gap_acc", "train_min"]].to_string(index=False))
print("(gap_acc = clean training accuracy minus held-out accuracy)")
print("\nPOOLED out-of-fold results (each held-out image predicted once; CIs resample duplicate groups):")
print(POOL.round(4).to_string(index=False))
print("\nEpoch-cap hits per job:", FOLDS.groupby(["split", "arch"])["hit_cap"].sum().to_dict())

# ---- 3. EfficientNetB0 vs DenseNet121: paired by fold (same held-out images) ----
H = []
for sp in SPLIT_CONFIGS:
    e = FOLDS[(FOLDS.split == sp) & (FOLDS.arch == "efficientnetb0")].sort_values("fold")
    d = FOLDS[(FOLDS.split == sp) & (FOLDS.arch == "densenet121")].sort_values("fold")
    assert (e["n"].values == d["n"].values).all(), "fold sizes differ - the held-out folds are not paired"
    for m in ["acc", "f1_macro", "loss"]:
        diff = d[m].values - e[m].values; K = len(diff)
        better = int((diff < 0).sum()) if m == "loss" else int((diff > 0).sum())
        H.append({"split": sp, "metric": m, "densenet_minus_effnet": diff.mean(), "se": diff.std(ddof=1) / np.sqrt(K),
                  "folds_densenet_better": f"{better}/{K}", "act_effnet": e.activation.iloc[0], "act_densenet": d.activation.iloc[0]})
HH = pd.DataFrame(H); HH.to_csv(DIRS["results"] / "cv_effnet_vs_densenet.csv", index=False)
print("\nEFFICIENTNETB0 vs DENSENET121 (DenseNet121 minus EfficientNetB0; for loss, negative favours DenseNet121):")
print(HH.round(4).to_string(index=False))
print("CAUTION: SEs treat folds as independent, but fold models share most training images, so they are too small. "
      "Where the activations differ between the two models, the comparison also includes an activation difference.")

# ---- 4. Figures ----
fig, axes = plt.subplots(1, 3, figsize=(21, 5.4))
for ax, sp in zip(axes, SPLIT_CONFIGS):
    for arch in MANDATORY_CV_ARCHS:
        d = FOLDS[(FOLDS.split == sp) & (FOLDS.arch == arch)].sort_values("fold"); act = d.activation.iloc[0].upper()
        ax.plot(d.fold + 1, d.acc, "-o", color=ARCH_COL[arch], lw=2, label=f"{dname(arch)} {act}: accuracy")
        ax.plot(d.fold + 1, d.f1_macro, "--s", color=ARCH_COL[arch], lw=1.5, alpha=0.75, label=f"{dname(arch)} {act}: macro-F1")
    ax.set_title(f"{SPLIT_LABEL[sp]}: held-out performance per fold", fontsize=13)
    ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel("Score", fontsize=12)
    ax.set_xticks(range(1, SPLIT_CONFIGS[sp]["cv_folds"] + 1)); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "20_cv_foldwise.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 10)); x = np.arange(3); w = 0.38
for ax, (m, lab) in zip(axes.ravel(), [("acc", "Accuracy"), ("prec_macro", "Macro precision"), ("rec_macro", "Macro recall"), ("f1_macro", "Macro F1-score")]):
    for k, arch in enumerate(MANDATORY_CV_ARCHS):
        mu = [FOLDS[(FOLDS.split == sp) & (FOLDS.arch == arch)][m].mean() for sp in SPLIT_CONFIGS]
        sd = [FOLDS[(FOLDS.split == sp) & (FOLDS.arch == arch)][m].std(ddof=1) for sp in SPLIT_CONFIGS]
        bars = ax.bar(x + (k - 0.5) * w, mu, w, yerr=sd, capsize=4, color=ARCH_COL[arch], edgecolor="black", label=dname(arch))
        for b, v in zip(bars, mu): ax.text(b.get_x() + b.get_width() / 2, 0.762, f"{v:.3f}", ha="center", fontsize=9, color="white", fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels([SPLIT_LABEL[sp] for sp in SPLIT_CONFIGS]); ax.set_ylim(0.75, 1.0)
    ax.set_title(f"CV {lab} (mean over folds, error bars = SD)", fontsize=13); ax.set_xlabel("Experimental split", fontsize=12)
    ax.set_ylabel(lab, fontsize=12); ax.legend(fontsize=10)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "21_cv_mean_sd.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(21, 5.4)); xc = np.arange(NUM_CLASSES)
for ax, sp in zip(axes, SPLIT_CONFIGS):
    for k, arch in enumerate(MANDATORY_CV_ARCHS):
        g = PC[(PC.split == sp) & (PC.arch == arch)].groupby("class")["sensitivity"]
        ax.bar(xc + (k - 0.5) * w, g.mean().reindex(CLASS_NAMES), w, yerr=g.std().reindex(CLASS_NAMES), capsize=3,
               color=ARCH_COL[arch], edgecolor="black", label=dname(arch))
    ax.set_xticks(xc); ax.set_xticklabels([CLASS_DISPLAY[i] for i in range(NUM_CLASSES)]); ax.set_ylim(0.5, 1.02)
    ax.set_title(f"{SPLIT_LABEL[sp]}: per-class sensitivity (recall)", fontsize=13)
    ax.set_xlabel("Class", fontsize=12); ax.set_ylabel("Sensitivity (mean over folds, bars = SD)", fontsize=11); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "22_cv_per_class_sensitivity.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
for k, arch in enumerate(MANDATORY_CV_ARCHS):
    mu = [FOLDS[(FOLDS.split == sp) & (FOLDS.arch == arch)]["train_min"].mean() for sp in SPLIT_CONFIGS]
    sd = [FOLDS[(FOLDS.split == sp) & (FOLDS.arch == arch)]["train_min"].std(ddof=1) for sp in SPLIT_CONFIGS]
    tot = [FOLDS[(FOLDS.split == sp) & (FOLDS.arch == arch)]["train_min"].sum() for sp in SPLIT_CONFIGS]
    axes[0].bar(x + (k - 0.5) * w, mu, w, yerr=sd, capsize=4, color=ARCH_COL[arch], edgecolor="black", label=dname(arch))
    axes[1].bar(x + (k - 0.5) * w, tot, w, color=ARCH_COL[arch], edgecolor="black", label=dname(arch))
    for i, v in enumerate(tot): axes[1].text(i + (k - 0.5) * w, v, f"{v:.0f}", ha="center", va="bottom", fontsize=9)
for ax, (t, yl) in zip(axes, [("Training time per fold", "Minutes per fold"), ("Total CV training time", "Minutes (all folds)")]):
    ax.set_xticks(x); ax.set_xticklabels([SPLIT_LABEL[sp] for sp in SPLIT_CONFIGS]); ax.set_title(t, fontsize=13)
    ax.set_xlabel("Experimental split", fontsize=12); ax.set_ylabel(yl, fontsize=12); ax.legend()
plt.tight_layout(); plt.savefig(DIRS["figures"] / "23_cv_training_time.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5.6))
for arch in MANDATORY_CV_ARCHS:
    mu = [FOLDS[(FOLDS.split == sp) & (FOLDS.arch == arch)]["f1_macro"].mean() for sp in SPLIT_CONFIGS]
    sd = [FOLDS[(FOLDS.split == sp) & (FOLDS.arch == arch)]["f1_macro"].std(ddof=1) for sp in SPLIT_CONFIGS]
    axes[0].errorbar(x + (0.06 if arch == "densenet121" else -0.06), mu, yerr=sd, fmt="-o", capsize=5, lw=2, ms=8,
                     color=ARCH_COL[arch], label=dname(arch))
axes[0].set_xticks(x); axes[0].set_xticklabels([SPLIT_LABEL[sp] for sp in SPLIT_CONFIGS])
axes[0].set_title("EfficientNetB0 vs DenseNet121: CV macro-F1 (mean +/- SD)", fontsize=13)
axes[0].set_xlabel("Experimental split", fontsize=12); axes[0].set_ylabel("Macro-F1", fontsize=12); axes[0].legend()
for i, sp in enumerate(SPLIT_CONFIGS):
    e = FOLDS[(FOLDS.split == sp) & (FOLDS.arch == "efficientnetb0")].sort_values("fold")["f1_macro"].values
    d = FOLDS[(FOLDS.split == sp) & (FOLDS.arch == "densenet121")].sort_values("fold")["f1_macro"].values
    diff = d - e
    axes[1].scatter(np.full(len(diff), i) + np.linspace(-0.12, 0.12, len(diff)), diff, s=45, color="gray", edgecolor="black", zorder=3)
    axes[1].hlines(diff.mean(), i - 0.25, i + 0.25, color="red", lw=2.5, zorder=4)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xticks(x); axes[1].set_xticklabels([SPLIT_LABEL[sp] for sp in SPLIT_CONFIGS])
axes[1].set_title("Per-fold macro-F1 difference (DenseNet121 minus EfficientNetB0; red = mean)", fontsize=12)
axes[1].set_xlabel("Experimental split", fontsize=12); axes[1].set_ylabel("Difference in macro-F1", fontsize=12)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "24_effnet_vs_densenet_cv.png", dpi=150, bbox_inches="tight"); plt.show()
print("\nSaved: cv_all_folds.csv, cv_all_perclass.csv, cv_pooled_table.csv, cv_mean_sd_table.csv, cv_effnet_vs_densenet.csv "
      "and figures 20-24. The test sets were never touched.")

In [ ]:
# ============================================================
# CELL 55: STAGE J - EXTRA CV FOR ANY OTHER BEST-TWO ARCHITECTURE (CONDITIONAL)
# ============================================================
SEL = load_json(DIRS["logs"] / "stage_h_selection.json")
needed = [(sp, a) for sp in SPLIT_CONFIGS for a in SEL["splits"][sp]["extra_cv_needed"]]
print("Best two per split (Stage H):", {sp: SEL["splits"][sp]["top_two"] for sp in SPLIT_CONFIGS})
if not needed:
    print("\nStage J is NOT needed: EfficientNetB0 and DenseNet121 (both already cross-validated) are the best two in every split.")
    for sp in SPLIT_CONFIGS:
        print(f"   {sp}: gap between 2nd and 3rd place = {SEL['splits'][sp]['third_place_gap_to_second']:.4f}"
              + ("  (within run-to-run noise; the rule is applied as written)" if SEL['splits'][sp]['third_place_gap_to_second'] < 0.02 else ""))
    print("Optional (not required by the rule): CV for ResNet50 on 70:30, where it is only ~0.007 behind second place.")
else:
    print("\nExtra CV required by the rule:", needed)
    for sp, a in needed:
        CV_SUMMARIES[(sp, a)] = run_cv(sp, a)

In [ ]:
# ============================================================
# CELL 29: PROJECT BACKUP V2 (SUPERSEDES CELL 19; EXCLUDES MODEL-WEIGHT FILES)
# ============================================================

BACKUP_EXCLUDE_NAMES = {"X_clean_224.npy"}                 # re-created by CELL 12
BACKUP_EXCLUDE_SUFFIXES = (".weights.h5", ".keras", ".tmp")  # model weights are large; histories/results/probs are kept

def backup_project_zip(zip_name: str = "project_backup.zip", include_weights: bool = False):
    """Zips the project folder into /kaggle/working/<zip_name>. By default model-weight files and the image cache are skipped.
    Use include_weights=True only when you specifically want the weights (e.g. the final model in Stage N)."""
    if not REGISTRY_PATH.exists():
        save_json({}, REGISTRY_PATH)
    zpath = Path("/kaggle/working") / zip_name
    if zpath.exists():
        zpath.unlink()
    n_files = raw = n_skipped = skipped_bytes = 0
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(BASE.rglob("*")):
            if not p.is_file():
                continue
            skip = (p.name in BACKUP_EXCLUDE_NAMES or p.name.endswith(".tmp")
                    or ((not include_weights) and p.name.endswith(BACKUP_EXCLUDE_SUFFIXES)))
            if skip:
                n_skipped += 1; skipped_bytes += p.stat().st_size; continue
            zf.write(p, p.relative_to(BASE.parent)); n_files += 1; raw += p.stat().st_size
    print(f"Backup written : {zpath}")
    print(f"Files stored   : {n_files} | uncompressed {raw/1e6:.1f} MB | zip {zpath.stat().st_size/1e6:.1f} MB")
    print(f"Skipped        : {n_skipped} files ({skipped_bytes/1e6:.0f} MB: image cache and, unless include_weights=True, model weights)")
    return zpath

zpath = backup_project_zip()

with zipfile.ZipFile(zpath) as zf:
    assert zf.testzip() is None, "zip is corrupt"
    names = set(zf.namelist())
expected = ["logs/config_base.json", "logs/run_registry.json", "logs/leakage_verification.json", "logs/augmentation_config.json",
            "logs/custom_cnn_diagnostic.json", "splits/splits.json", "data/clean_index.csv", "data/removed_images.csv",
            "results/model_parameter_table.csv", "results/custom_cnn_diagnostic.csv"]
print("\nContents check:")
for e in expected:
    print(f"   [{'OK     ' if 'brain_tumor_project/' + e in names else 'MISSING'}] {e}")
n_runs = sum(n.startswith("brain_tumor_project/results/runs/") for n in names)
n_figs = sum(n.startswith("brain_tumor_project/figures/") for n in names)
print(f"\nRun-result files stored: {n_runs} | figures stored: {n_figs} | runs in registry: {len(list_runs())}")
print("\nDownload: right-hand panel -> Data -> Output -> /kaggle/working -> project_backup.zip -> three dots -> Download.")
print("From now on use THIS cell for backups (re-running CELL 19 would restore the old behaviour that includes weights).")

In [ ]:
# ============================================================
# CELL 61: RAISE THE GPU STOP LIMIT AND FINISH THE LAST 2 FOLDS OF RESNET50 90:10 CV
# ============================================================
GPU_STOP_MIB = 10500      # was 8000; the CELL 60 guard reads this global before every new run
print(f"GPU stop limit is now {GPU_STOP_MIB} MiB")
print(mem_line("before"))
act = cv_activation("90_10", "resnet50")
done = [f for f in range(10) if load_result(make_run_id("cv", "90_10", "resnet50", act, fold=f)) is not None]
print(f"ResNet50 90:10 ({act.upper()}) folds already saved: {done}")
CV_SUMMARIES[("90_10", "resnet50")] = run_cv("90_10", "resnet50")

In [ ]:
# ============================================================
# CELL 62: BACKUP INCLUDING THE SIX SELECTED CHECKPOINTS (RESTORE WITHOUT RETRAINING)
# ============================================================
SEL = load_json(DIRS["logs"] / "stage_h_selection.json")
assert SEL is not None, "stage_h_selection.json missing: run CELL 45 first"

sel_ck = []
for sp in SPLIT_CONFIGS:
    for arch in SEL["splits"][sp]["top_two"]:
        act = SEL["splits"][sp]["best_activation"][arch]
        p = run_paths(make_run_id("init", sp, arch, act))["ckpt"]
        sel_ck.append((sp, arch, act, p))

print("Selected models (best two per split) and their checkpoints:")
total = 0
for sp, arch, act, p in sel_ck:
    sz = p.stat().st_size if p.exists() else 0; total += sz
    print(f"   {sp:<6}{arch:<16}{act:<6} {'OK     ' if p.exists() else 'MISSING'} {sz/1e6:8.0f} MB")
missing = [str(p.name) for _, _, _, p in sel_ck if not p.exists()]
assert not missing, f"Missing selected checkpoints: {missing} (CELL 58 would have to retrain these)"
assert total < 4e9, f"Selected checkpoints total {total/1e9:.1f} GB - too large to back up comfortably"

if not REGISTRY_PATH.exists():
    save_json({}, REGISTRY_PATH)
keep = {p.resolve() for _, _, _, p in sel_ck}
zpath = Path("/kaggle/working/project_backup_selected.zip")
if zpath.exists(): zpath.unlink()
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(BASE.rglob("*")):
        if not p.is_file() or p.name.endswith(".tmp") or p.name == "X_clean_224.npy":
            continue
        if p.name.endswith((".weights.h5", ".keras")) and p.resolve() not in keep:
            continue
        zf.write(p, p.relative_to(BASE.parent)); n += 1
with zipfile.ZipFile(zpath) as zf:
    assert zf.testzip() is None
    names = set(zf.namelist())
for _, _, _, p in sel_ck:
    assert f"brain_tumor_project/checkpoints/{p.name}" in names
for must in ["logs/run_registry.json", "splits/splits.json", "splits/splits_frozen.json", "logs/stage_h_selection.json"]:
    assert f"brain_tumor_project/{must}" in names, f"missing {must}"
print(f"\nWritten: {zpath} | {n} files | {zpath.stat().st_size/1e9:.2f} GB | selected weights included, other weights and image cache excluded")
print("Download: right-hand panel -> Data -> Output -> /kaggle/working -> project_backup_selected.zip -> three dots -> Download.")
print("Restore later: upload as a Kaggle Dataset, set RESTORE_FROM in CELL 4 to its /kaggle/input/... folder.")

In [ ]:
# ============================================================
# CELL 56: STAGE K2 - OVERFITTING/UNDERFITTING, VALIDATION OPTIMISM, CLASS CONFUSIONS
# ============================================================
FOLDS = pd.read_csv(DIRS["results"] / "cv_all_folds.csv")
S = pd.read_csv(DIRS["results"] / "stage_g_validation_table.csv")
SEL = load_json(DIRS["logs"] / "stage_h_selection.json")
dname = lambda a: ARCH_SPECS[a]["display"]

def status(train_acc, gap):
    if train_acc < 0.85: return "underfit-leaning (train acc < 0.85)"
    if gap > 0.08: return "overfit-leaning (gap > 0.08)"
    return "balanced"

# ---- 1. Generalisation gap in cross-validation ----
G = FOLDS.groupby(["split", "arch", "activation"]).agg(
    train_acc=("train_acc_clean", "mean"), heldout_acc=("acc", "mean"), gap=("gap_acc", "mean"), acc_sd=("acc", "std"),
    acc_min=("acc", "min"), acc_max=("acc", "max"), f1_sd=("f1_macro", "std")).reset_index()
G["status"] = [status(a, g) for a, g in zip(G.train_acc, G.gap)]
print("CROSS-VALIDATION: generalisation gap (clean training accuracy minus held-out accuracy)")
print(G.round(4).to_string(index=False))

# ---- 2. Stage G fit status per architecture (clean training accuracy vs validation) ----
SG = S.groupby("arch").agg(train_acc=("train_acc", "mean"), val_acc=("val_acc", "mean"), gap=("gap_acc", "mean"),
                           gap_max=("gap_acc", "max")).loc[ARCHITECTURES]
SG["status"] = [status(a, g) for a, g in zip(SG.train_acc, SG.gap)]
print("\nSTAGE G (6 runs per architecture):")
print(SG.round(4).to_string())
print("Custom CNN: clean training accuracy is only about 0.75-0.78 in all six runs, so it underfits; "
      "that is a capacity/optimisation limit of this configuration (tuned with 6 short trials), not overfitting.")

# ---- 3. Is single-split validation optimistic? (Stage G validation accuracy vs CV held-out accuracy, same configuration) ----
O = G.merge(S[["split", "arch", "activation", "val_acc", "val_f1"]], on=["split", "arch", "activation"])
O["stageG_val_minus_cv_acc"] = O["val_acc"] - O["heldout_acc"]
print("\nSTAGE G VALIDATION vs CV HELD-OUT ACCURACY (same model configuration):")
print(O[["split", "arch", "activation", "val_acc", "heldout_acc", "stageG_val_minus_cv_acc", "acc_sd"]].round(4).to_string(index=False))
print(f"Mean (Stage G validation - CV): {O['stageG_val_minus_cv_acc'].mean():+.4f}. Positive = single validation sets look better "
      "than cross-validation. Two causes cannot be separated here: selection on the same set (winner's curse) and "
      "fold models training on fewer images.")

# ---- 4. Pooled confusion: which class pairs are confused? ----
print("\nLARGEST CONFUSIONS in the pooled out-of-fold matrices (share of the true class):")
for sp in SPLIT_CONFIGS:
    for arch in MANDATORY_CV_ARCHS:
        act = SEL["splits"][sp]["best_activation"][arch]
        cm = pd.read_csv(DIRS["cv"] / f"{sp}__{arch}__{act}_pooled_confusion.csv", index_col=0).values
        pairs = sorted([(cm[i, j] / cm[i].sum(), cm[i, j], i, j) for i in range(4) for j in range(4) if i != j], reverse=True)[:3]
        print(f"   {sp} {dname(arch)} {act.upper()}: " + " | ".join(
            f"{CLASS_DISPLAY[i]}->{CLASS_DISPLAY[j]} {c} ({100*r:.1f}%)" for r, c, i, j in pairs))

# ---- 5. Figure ----
fig, axes = plt.subplots(1, 3, figsize=(22, 5.6))
x = np.arange(len(ARCHITECTURES)); w = 0.38
axes[0].bar(x - w / 2, SG["train_acc"], w, label="Clean training accuracy", edgecolor="black")
axes[0].bar(x + w / 2, SG["val_acc"], w, label="Validation accuracy", edgecolor="black")
axes[0].set_xticks(x); axes[0].set_xticklabels([dname(a) for a in SG.index], rotation=15); axes[0].set_ylim(0.5, 1.02)
axes[0].set_title("Stage G: training vs validation accuracy (mean of 6 runs)", fontsize=12)
axes[0].set_xlabel("Architecture", fontsize=12); axes[0].set_ylabel("Accuracy", fontsize=12); axes[0].legend(fontsize=9)
lab = [f"{r.split.replace('_', ':')}\n{dname(r.arch)[:6]} {r.activation.upper()}" for r in G.itertuples()]; xx = np.arange(len(G))
axes[1].bar(xx - w / 2, G["train_acc"], w, label="Clean training accuracy", edgecolor="black")
axes[1].bar(xx + w / 2, G["heldout_acc"], w, yerr=G["acc_sd"], capsize=3, label="CV held-out accuracy (+/- SD)", edgecolor="black")
axes[1].set_xticks(xx); axes[1].set_xticklabels(lab, fontsize=9); axes[1].set_ylim(0.8, 1.02)
axes[1].set_title("Cross-validation: training vs held-out accuracy", fontsize=12)
axes[1].set_xlabel("Split and model", fontsize=12); axes[1].set_ylabel("Accuracy", fontsize=12); axes[1].legend(fontsize=9)
axes[2].bar(xx - w / 2, O["val_acc"], w, label="Stage G validation accuracy", edgecolor="black")
axes[2].bar(xx + w / 2, O["heldout_acc"], w, label="CV held-out accuracy (mean)", edgecolor="black")
axes[2].set_xticks(xx); axes[2].set_xticklabels(lab, fontsize=9); axes[2].set_ylim(0.8, 1.0)
axes[2].set_title("Single validation set vs cross-validation", fontsize=12)
axes[2].set_xlabel("Split and model", fontsize=12); axes[2].set_ylabel("Accuracy", fontsize=12); axes[2].legend(fontsize=9)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "25_overfitting_analysis.png", dpi=150, bbox_inches="tight"); plt.show()
G.to_csv(DIRS["results"] / "cv_generalisation_gap.csv", index=False); O.to_csv(DIRS["results"] / "validation_vs_cv_optimism.csv", index=False)

print("\nWHAT WAS (AND WAS NOT) DONE ABOUT OVER/UNDERFITTING:")
print("  done: AdamW weight decay, dropout, early stopping, LR reduction, class weights, two-stage fine-tuning, and a Stage F search "
      "over dropout, weight decay and the unfrozen fraction; conservative augmentation (the ablation showed no measurable benefit).")
print("  not done: no extra regularisation experiments beyond these; the custom CNN's underfitting was not fixed (capacity was not increased).")
print("The test sets were never touched.")

In [ ]:
# ============================================================
# CELL 57: STAGE K3 - PRE-REGISTERED FINAL SELECTION (WRITTEN BEFORE ANY TEST EVALUATION)
# ============================================================
SEL = load_json(DIRS["logs"] / "stage_h_selection.json")
FOLDS = pd.read_csv(DIRS["results"] / "cv_all_folds.csv")
S = pd.read_csv(DIRS["results"] / "stage_g_validation_table.csv")
D = pd.read_csv(DIRS["results"] / "relu_vs_gelu_paired.csv")
dname = lambda a: ARCH_SPECS[a]["display"]
assert not (DIRS["results"] / "test").exists() or not any((DIRS["results"] / "test").glob("*.json")), \
    "Test results already exist: selection must be written BEFORE any test evaluation."

RULES = {
    "architecture": "higher mean CV macro-F1 over the 3 splits; if the difference < mean fold SD: lower mean CV gap if the gap "
                    "difference >= 0.01, else fewer parameters",
    "split": "among splits with mean CV macro-F1 within one fold SD (of the best split) of the best: the largest training portion",
    "activation": "the activation used in the CV of that (architecture, split)",
    "best_activation_overall": "declared only if |mean paired Stage G delta macro-F1| >= 0.02 AND the winner takes >= 11 of 15 pairs; else no winner",
    "best_generalisation": "lowest mean Stage G train-val gap among architectures with mean val F1 within 0.05 of the best; also reported for CV",
    "best_efficiency": "lowest GPU batch-1 latency among architectures with mean Stage G val F1 within 0.02 of the best",
    "caveats": ["CV used one activation per split", "split choice is confounded with training-set size",
                "CV nested test sets are not independent", "only EfficientNetB0 and DenseNet121 have CV evidence"]}

# ---- Architecture ----
per = FOLDS.groupby(["arch", "split"]).agg(f1=("f1_macro", "mean"), sd=("f1_macro", "std"), gap=("gap_acc", "mean")).reset_index()
AS = per.groupby("arch").agg(mean_f1=("f1", "mean"), mean_sd=("sd", "mean"), mean_gap=("gap", "mean"))
AS["params_M"] = S.groupby("arch")["params_M"].first()
print("Mean CV statistics per architecture (over the 3 splits):"); print(AS.round(4).to_string())
order = AS.sort_values("mean_f1", ascending=False); a1, a2 = order.index[0], order.index[1]
diff, tol = float(order.loc[a1, "mean_f1"] - order.loc[a2, "mean_f1"]), float((order.loc[a1, "mean_sd"] + order.loc[a2, "mean_sd"]) / 2)
if diff >= tol:
    chosen, why_arch = a1, f"{dname(a1)} leads by {diff:.4f} macro-F1 (>= mean fold SD {tol:.4f})"
else:
    gd = float(order.loc[a1, "mean_gap"] - order.loc[a2, "mean_gap"])
    if abs(gd) >= 0.01:
        chosen = a2 if gd > 0 else a1
        why_arch = f"F1 difference {diff:.4f} < fold SD {tol:.4f} (tie); lower generalisation gap decides ({gd:+.4f})"
    else:
        chosen = min([a1, a2], key=lambda a: AS.loc[a, "params_M"])
        why_arch = f"F1 difference {diff:.4f} < fold SD {tol:.4f} and gap difference {abs(gd):.4f} < 0.01 (tie); fewer parameters decides"

# ---- Split ----
ps = per[per.arch == chosen].set_index("split")
best_sp = ps["f1"].idxmax(); tol_sp = float(ps.loc[best_sp, "sd"])
tied = [sp for sp in SPLIT_CONFIGS if ps.loc[sp, "f1"] >= ps.loc[best_sp, "f1"] - tol_sp]
chosen_split = max(tied, key=lambda sp: SPLIT_CONFIGS[sp]["train_frac"])
chosen_act = SEL["splits"][chosen_split]["best_activation"][chosen]
print(f"\nSplit statistics for {dname(chosen)}:"); print(ps.round(4).to_string())
print(f"Best split by mean F1: {best_sp} (SD {tol_sp:.4f}); splits within one SD: {tied}")

# ---- Other awards ----
SG = S.groupby("arch").agg(mean_f1=("val_f1", "mean"), mean_gap=("gap_acc", "mean"), params_M=("params_M", "first"),
                           train_min=("train_min", "mean"), latency_ms=("latency_ms", "mean")).loc[ARCHITECTURES]
best_val = SG["mean_f1"].idxmax(); best_cv = AS["mean_f1"].idxmax()
pool_g = SG[SG.mean_f1 >= SG.mean_f1.max() - 0.05]; best_gen_val = pool_g["mean_gap"].idxmin(); best_gen_cv = AS["mean_gap"].idxmin()
pool_e = SG[SG.mean_f1 >= SG.mean_f1.max() - 0.02]; best_eff = pool_e["latency_ms"].idxmin()
dv = D["d_f1"].values; mean_d = float(dv.mean()); wins_g, wins_r = int((dv > 0).sum()), int((dv < 0).sum())
act_verdict = ("GELU" if mean_d > 0 else "ReLU") if (abs(mean_d) >= 0.02 and max(wins_g, wins_r) >= 11) else "no winner (no detectable difference)"
print("\nStage G per-architecture means (6 runs each):"); print(SG.round(4).to_string())

FINAL = {"locked_before_test": True, "written_at": time.strftime("%Y-%m-%d %H:%M:%S"), "rules": RULES,
         "architecture_stats": AS.reset_index().to_dict("records"), "stage_g_stats": SG.reset_index().to_dict("records"),
         "final_model": {"arch": chosen, "split": chosen_split, "activation": chosen_act, "why_architecture": why_arch,
                         "why_split": f"splits within one SD of the best: {tied}; largest training portion chosen"},
         "awards": {"best_performing_validation": best_val, "best_performing_cv": best_cv,
                    "best_generalisation_validation": best_gen_val, "best_generalisation_cv": best_gen_cv,
                    "best_efficiency": best_eff, "best_activation": act_verdict,
                    "activation_stats": {"mean_delta_f1_gelu_minus_relu": mean_d, "gelu_wins": wins_g, "relu_wins": wins_r, "n_pairs": int(len(dv))},
                    "best_two_per_split": {sp: SEL["splits"][sp]["top_two"] for sp in SPLIT_CONFIGS},
                    "best_split_for_final_architecture": best_sp},
         "test_sets_accessed_before_this_file": False}
save_json(FINAL, DIRS["logs"] / "final_selection.json")

print("\n" + "=" * 90); print("SELECTION (validation and cross-validation evidence only)"); print("=" * 90)
print(f"Final model          : {dname(chosen)} | split {chosen_split} | activation {chosen_act.upper()}")
print(f"  architecture reason: {why_arch}")
print(f"Best performing      : CV {dname(best_cv)} | validation (Stage G mean) {dname(best_val)}")
print(f"Best generalisation  : validation {dname(best_gen_val)} | CV {dname(best_gen_cv)}")
print(f"Best efficiency      : {dname(best_eff)} (lowest latency among models within 0.02 F1 of the best)")
print(f"Best activation      : {act_verdict} (mean delta F1 GELU-ReLU {mean_d:+.4f}; GELU {wins_g}, ReLU {wins_r} of {len(dv)} pairs)")
print(f"Best split           : {best_sp} by mean CV F1; chosen {chosen_split} under the tie rule (confounded with training size)")
print("Best two per split   :", {sp: [dname(a) for a in SEL['splits'][sp]['top_two']] for sp in SPLIT_CONFIGS})
print("\nSaved logs/final_selection.json. The test sets have not been used. CELL 58 releases the lock.")

In [ ]:
# ============================================================
# CELL 62: BACKUP INCLUDING THE SIX SELECTED CHECKPOINTS (RESTORE WITHOUT RETRAINING)
# ============================================================
SEL = load_json(DIRS["logs"] / "stage_h_selection.json")
assert SEL is not None, "stage_h_selection.json missing: run CELL 45 first"

sel_ck = []
for sp in SPLIT_CONFIGS:
    for arch in SEL["splits"][sp]["top_two"]:
        act = SEL["splits"][sp]["best_activation"][arch]
        p = run_paths(make_run_id("init", sp, arch, act))["ckpt"]
        sel_ck.append((sp, arch, act, p))

print("Selected models (best two per split) and their checkpoints:")
total = 0
for sp, arch, act, p in sel_ck:
    sz = p.stat().st_size if p.exists() else 0; total += sz
    print(f"   {sp:<6}{arch:<16}{act:<6} {'OK     ' if p.exists() else 'MISSING'} {sz/1e6:8.0f} MB")
missing = [str(p.name) for _, _, _, p in sel_ck if not p.exists()]
assert not missing, f"Missing selected checkpoints: {missing} (CELL 58 would have to retrain these)"
assert total < 4e9, f"Selected checkpoints total {total/1e9:.1f} GB - too large to back up comfortably"

if not REGISTRY_PATH.exists():
    save_json({}, REGISTRY_PATH)
keep = {p.resolve() for _, _, _, p in sel_ck}
zpath = Path("/kaggle/working/project_backup_selected.zip")
if zpath.exists(): zpath.unlink()
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(BASE.rglob("*")):
        if not p.is_file() or p.name.endswith(".tmp") or p.name == "X_clean_224.npy":
            continue
        if p.name.endswith((".weights.h5", ".keras")) and p.resolve() not in keep:
            continue
        zf.write(p, p.relative_to(BASE.parent)); n += 1
with zipfile.ZipFile(zpath) as zf:
    assert zf.testzip() is None
    names = set(zf.namelist())
for _, _, _, p in sel_ck:
    assert f"brain_tumor_project/checkpoints/{p.name}" in names
for must in ["logs/run_registry.json", "splits/splits.json", "splits/splits_frozen.json", "logs/stage_h_selection.json"]:
    assert f"brain_tumor_project/{must}" in names, f"missing {must}"
print(f"\nWritten: {zpath} | {n} files | {zpath.stat().st_size/1e9:.2f} GB | selected weights included, other weights and image cache excluded")
print("Download: right-hand panel -> Data -> Output -> /kaggle/working -> project_backup_selected.zip -> three dots -> Download.")
print("Restore later: upload as a Kaggle Dataset, set RESTORE_FROM in CELL 4 to its /kaggle/input/... folder.")

In [ ]:
# ============================================================
# CELL 58: STAGE L - RELEASE TEST LOCK AND EVALUATE ON THE UNTOUCHED TEST SETS
# ============================================================
CONFIRM_TEST_EVAL = True   # set to True only AFTER you have read CELL 57's selection output
assert CONFIRM_TEST_EVAL, "Stopped on purpose: read CELL 57's output first, then set CONFIRM_TEST_EVAL = True and run only this cell."

FSEL = load_json(DIRS["logs"] / "final_selection.json")
assert FSEL is not None and FSEL.get("locked_before_test") is True, "Run CELL 57 first: selection must be written before test evaluation."
SEL = load_json(DIRS["logs"] / "stage_h_selection.json")
TEST_DIR = DIRS["results"] / "test"; TEST_DIR.mkdir(parents=True, exist_ok=True)
dname = lambda a: ARCH_SPECS[a]["display"]
padded_all = (df_clean["width"].values != df_clean["height"].values)     # True = image was padded to square

def is_selected(sp, arch, act):
    return arch in SEL["splits"][sp]["top_two"] and SEL["splits"][sp]["best_activation"][arch] == act

def load_model_for_test(sp, arch, act):
    """Stage G checkpoint if it exists; otherwise retrain (selected models only). Returns (model, source) or None."""
    s = get_split(sp); ck = run_paths(make_run_id("init", sp, arch, act))["ckpt"]; source = "stage_g_checkpoint"
    if not ck.exists():
        if not is_selected(sp, arch, act):
            return None
        rid = make_run_id("retrain", sp, arch, act)
        print(f"   [retrain] Stage G checkpoint missing for a SELECTED model: retraining {arch} {act} on {sp} (same configuration)")
        train_and_evaluate(rid, arch, act, sp, s["fit_idx"], s["val_idx"], hp_overrides=final_overrides(arch), verbose=0)
        ck = run_paths(rid)["ckpt"]; source = "retrained_checkpoint_missing"
    reset_session()
    hp = get_hp(arch, **final_overrides(arch))
    px = compute_pixel_stats(s["fit_idx"]) if ARCH_SPECS[arch]["kind"] == "scratch" else None
    model = build_model(arch, act, hp, pixel_stats=px)            # ImageNet weights: same layer structure as in training
    model.load_weights(str(ck))
    return model, source

def test_one(sp, arch, act):
    rid = make_run_id("test", sp, arch, act); jp = TEST_DIR / f"{rid}.json"
    if jp.exists():
        return load_json(jp)
    out = load_model_for_test(sp, arch, act)
    if out is None:
        return None
    model, source = out
    idx = get_split(sp)["test_idx"]
    probs, m = evaluate_subset(model, sp, "test", idx)            # raises if the lock is closed
    y, pred = y_all[idx], probs.argmax(1)
    ci = cluster_bootstrap_ci(y, pred, groups=groups_all[idx], n_boot=500)
    k = int((pred == y).sum()); w = wilson_ci(k, len(idx)); pm_ = padded_all[idx]; ok = pred == y
    res = {"run_id": rid, "split": sp, "arch": arch, "activation": act, "selected": bool(is_selected(sp, arch, act)),
           "source": source, "n": int(len(idx)), "metrics": m, "acc_wilson": list(w), "boot_ci": ci,
           "padded": {"n_padded": int(pm_.sum()), "n_unpadded": int((~pm_).sum()),
                      "acc_padded": float(ok[pm_].mean()) if pm_.any() else float("nan"),
                      "acc_unpadded": float(ok[~pm_].mean()) if (~pm_).any() else float("nan")}}
    np.savez_compressed(TEST_DIR / f"{rid}.npz", test_idx=idx, probs=probs)
    save_json(res, jp)
    del model; reset_session()
    return res

results, skipped = [], []
TEST_UNLOCKED = True                                              # released ONLY inside this cell
try:
    for sp in SPLIT_CONFIGS:
        for arch in ARCHITECTURES:
            for act in ACTIVATIONS:
                r = test_one(sp, arch, act)
                if r is None: skipped.append(f"{sp}/{arch}/{act}")
                else:
                    results.append(r)
                    print(f"[test] {sp} {dname(arch):15s} {act.upper():4s} {'(SELECTED)' if r['selected'] else '          '} "
                          f"acc {r['metrics']['accuracy']:.4f} | macro-F1 {r['metrics']['f1_macro']:.4f} | n={r['n']}")
finally:
    TEST_UNLOCKED = False                                         # always re-lock
assert TEST_UNLOCKED is False

rows = []
for r in results:
    m = r["metrics"]
    rows.append({"split": r["split"], "arch": r["arch"], "activation": r["activation"], "selected": r["selected"], "source": r["source"],
                 "n": r["n"], "test_acc": m["accuracy"], "acc_ci_lo": r["acc_wilson"][0], "acc_ci_hi": r["acc_wilson"][1],
                 "f1_ci_lo": r["boot_ci"]["f1_macro"][0], "f1_ci_hi": r["boot_ci"]["f1_macro"][1],
                 "kappa_ci_lo": r["boot_ci"]["kappa"][0], "kappa_ci_hi": r["boot_ci"]["kappa"][1],
                 "acc_padded": r["padded"]["acc_padded"], "acc_unpadded": r["padded"]["acc_unpadded"], "n_padded": r["padded"]["n_padded"],
                 **flatten_metrics(m)})
TD = pd.DataFrame(rows); TD.to_csv(DIRS["results"] / "test_results_all.csv", index=False)
if skipped: print("\nSkipped (no checkpoint, not selected):", skipped)

print("\n" + "=" * 100); print("TEST RESULTS: THE SIX SELECTED MODELS (independent test sets; descriptive, no feedback into selection)"); print("=" * 100)
sel = TD[TD.selected].sort_values(["split", "arch"])
print(sel[["split", "arch", "activation", "n", "test_acc", "acc_ci_lo", "acc_ci_hi", "precision_macro", "recall_macro", "f1_macro",
           "f1_ci_lo", "f1_ci_hi", "kappa", "roc_auc_macro", "pr_auc_macro", "loss", "ece"]].round(4).to_string(index=False))
print("\nPer-class sensitivity / specificity / FPR / FNR of the selected models:")
for r in sel.itertuples():
    pcs = next(x for x in results if x["split"] == r.split and x["arch"] == r.arch and x["activation"] == r.activation)["metrics"]["per_class"]
    print(f"  {r.split} {dname(r.arch)} {r.activation.upper()}: " + " | ".join(
        f"{CLASS_DISPLAY[CLASS_NAMES.index(c)]} sens {d['sensitivity']:.3f} spec {d['specificity']:.3f} FPR {d['fpr']:.3f} FNR {d['fnr']:.3f} (n={d['support']})"
        for c, d in pcs.items()))
print("\nPadded vs unpadded test images (shortcut check; a large gap would suggest dependence on image geometry):")
print(sel[["split", "arch", "n_padded", "acc_padded", "acc_unpadded"]].round(4).to_string(index=False))
print("\nALL EVALUATED MODELS (test accuracy [95% Wilson CI] / macro-F1):")
show = TD.sort_values(["split", "activation", "arch"])
print(show[["split", "activation", "arch", "selected", "test_acc", "acc_ci_lo", "acc_ci_hi", "f1_macro"]].round(4).to_string(index=False))
print("\nNOTES: the three test sets are nested (90:10 within 80:20 within 70:30), so results across splits are not independent. "
      "CIs reflect test-set size only, not training randomness (single seed). The 90:10 test set has only 30 No Tumor images. "
      "The dataset has no patient IDs, so patient-level generalisation is not measured.")
print("Saved results/test_results_all.csv and per-model JSON/NPZ files in results/test/. Test lock restored:", not TEST_UNLOCKED)

In [ ]:
# ============================================================
# CELL 63: STAGE L (CONTINUED) - TEST FIGURES AND TABLES FOR THE SIX SELECTED MODELS
# ============================================================
TEST_DIR = DIRS["results"] / "test"
SEL = load_json(DIRS["logs"] / "stage_h_selection.json")
dname = lambda a: ARCH_SPECS[a]["display"]
FIG_T = DIRS["figures"] / "test"; FIG_T.mkdir(exist_ok=True)
padded_all = (df_clean["width"].values != df_clean["height"].values)      # True = image was padded to square

items = []
for sp in SPLIT_CONFIGS:
    for arch in SEL["splits"][sp]["top_two"]:
        act = SEL["splits"][sp]["best_activation"][arch]
        rid = make_run_id("test", sp, arch, act)
        jp, zp = TEST_DIR / f"{rid}.json", TEST_DIR / f"{rid}.npz"
        assert jp.exists() and zp.exists(), f"missing test result {rid}: run CELL 58 first"
        z = np.load(zp)
        items.append({"sp": sp, "arch": arch, "act": act, "rid": rid, "res": load_json(jp),
                      "idx": z["test_idx"], "probs": z["probs"], "y": y_all[z["test_idx"]]})
for it in items:
    assert np.array_equal(it["idx"], get_split(it["sp"])["test_idx"])
    it["retrained"] = it["res"]["source"] != "stage_g_checkpoint"
    it["label"] = f"{it['sp'].replace('_', ':')} {dname(it['arch'])} {it['act'].upper()}" + (" (retrained)" if it["retrained"] else "")

# ---- per-model confusion matrix and ROC/PR figures (saved, not displayed) ----
for it in items:
    plot_confusion(it["res"]["metrics"]["confusion_matrix"], f"TEST {it['label']}", FIG_T / f"confusion_{it['rid']}.png", show=False)
    plot_roc_pr(it["y"], it["probs"], f"TEST {it['label']}", FIG_T / f"roc_pr_{it['rid']}.png", show=False)
print(f"Saved 6 confusion-matrix and 6 ROC/PR figures to {FIG_T}")

# ---- 1. Confusion matrices of all six (row-normalised colours, counts and percentages in cells) ----
names = [CLASS_DISPLAY[i] for i in range(NUM_CLASSES)]
fig, axes = plt.subplots(3, 2, figsize=(15, 20))
for ax, it in zip(axes.ravel(), items):
    cm = np.array(it["res"]["metrics"]["confusion_matrix"]); rn = cm / cm.sum(1, keepdims=True)
    annot = np.array([[f"{cm[i, j]}\n{100 * rn[i, j]:.0f}%" for j in range(NUM_CLASSES)] for i in range(NUM_CLASSES)])
    sns.heatmap(rn, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax, xticklabels=names, yticklabels=names)
    ax.set_title(f"TEST {it['label']} (n={len(it['y'])})", fontsize=12)
    ax.set_xlabel("Predicted class", fontsize=11); ax.set_ylabel("True class", fontsize=11)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "26_test_confusion_matrices.png", dpi=150, bbox_inches="tight"); plt.show()

# ---- 2. Test accuracy with Wilson 95% intervals ----
accs = np.array([it["res"]["metrics"]["accuracy"] for it in items])
lo = np.array([it["res"]["acc_wilson"][0] for it in items]); hi = np.array([it["res"]["acc_wilson"][1] for it in items])
fig, ax = plt.subplots(figsize=(14, 5.5))
bars = ax.bar(range(6), accs, yerr=[accs - lo, hi - accs], capsize=6, edgecolor="black",
              color=[ARCH_COL_ALL.get(it["arch"], "#888888") if "ARCH_COL_ALL" in globals() else "#4c72b0" for it in items])
for b, it in zip(bars, items):
    if it["retrained"]: b.set_hatch("//")
    ax.text(b.get_x() + b.get_width() / 2, 0.805, f"{it['res']['metrics']['accuracy']:.3f}", ha="center", color="white", fontweight="bold")
ax.set_xticks(range(6)); ax.set_xticklabels([it["label"].replace(" (retrained)", "") for it in items], rotation=15, fontsize=10)
ax.set_ylim(0.8, 1.0); ax.set_title("Independent test accuracy of the six selected models (bars: Wilson 95% CI; hatched = retrained)", fontsize=13)
ax.set_xlabel("Split and model", fontsize=12); ax.set_ylabel("Test accuracy", fontsize=12)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "27_test_accuracy_ci.png", dpi=150, bbox_inches="tight"); plt.show()

# ---- 3. CV estimate vs test result ----
rows = []
for it in items:
    js = load_json(DIRS["cv"] / f"{it['sp']}__{it['arch']}__{it['act']}_summary.json"); m = it["res"]["metrics"]
    rows.append({"model": it["label"], "cv_acc": js["mean"]["acc"] if js else np.nan, "cv_acc_sd": js["sd"]["acc"] if js else np.nan,
                 "test_acc": m["accuracy"], "cv_f1": js["mean"]["f1_macro"] if js else np.nan, "test_f1": m["f1_macro"]})
CVT = pd.DataFrame(rows); CVT["test_minus_cv_acc"] = CVT["test_acc"] - CVT["cv_acc"]
print("\nCROSS-VALIDATION ESTIMATE vs INDEPENDENT TEST RESULT:")
print(CVT.round(4).to_string(index=False))
print(f"Mean (test - CV) accuracy: {CVT['test_minus_cv_acc'].mean():+.4f} | range {CVT['test_minus_cv_acc'].min():+.3f} to {CVT['test_minus_cv_acc'].max():+.3f}")
fig, ax = plt.subplots(figsize=(14, 5.5)); x = np.arange(6); w = 0.38
ax.bar(x - w / 2, CVT["cv_acc"], w, yerr=CVT["cv_acc_sd"], capsize=4, edgecolor="black", label="CV held-out accuracy (mean +/- SD over folds)")
ax.bar(x + w / 2, CVT["test_acc"], w, yerr=[accs - lo, hi - accs], capsize=4, edgecolor="black", label="Independent test accuracy (Wilson 95% CI)")
ax.set_xticks(x); ax.set_xticklabels([it["label"].replace(" (retrained)", "*") for it in items], rotation=15, fontsize=10)
ax.set_ylim(0.8, 1.0); ax.set_title("Cross-validation vs test accuracy (* = retrained after the checkpoint was missing)", fontsize=13)
ax.set_xlabel("Split and model", fontsize=12); ax.set_ylabel("Accuracy", fontsize=12); ax.legend(fontsize=10)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "28_cv_vs_test.png", dpi=150, bbox_inches="tight"); plt.show()

# ---- 4. Per-class sensitivity and specificity on the test sets ----
for key in ("sensitivity", "specificity"):
    t = pd.DataFrame({it["label"]: [it["res"]["metrics"]["per_class"][c][key] for c in CLASS_NAMES] for it in items}, index=names)
    plot_bar_compare(t, f"Test per-class {key} of the six selected models", key.capitalize(),
                     DIRS["figures"] / f"29_test_per_class_{key}.png", ylim=(0.6, 1.03), figsize=(16, 5.5), xlabel="Class")

# ---- 5. Padded vs unpadded images: class mix, then accuracy with Wilson intervals ----
print("\nPADDED IMAGES BY CLASS IN EACH TEST SET (padded = original was not square):")
for sp in SPLIT_CONFIGS:
    idx = get_split(sp)["test_idx"]
    d = pd.DataFrame({"cls": [CLASS_DISPLAY[i] for i in y_all[idx]], "padded": padded_all[idx]})
    comp = d.groupby("cls")["padded"].agg(n="size", n_padded="sum"); comp["pct_padded"] = (100 * comp["n_padded"] / comp["n"]).round(1)
    print(f"  {sp}: total padded {int(padded_all[idx].sum())} of {len(idx)}"); print(comp.to_string())
prow = []
for it in items:
    pm = padded_all[it["idx"]]; ok = it["probs"].argmax(1) == it["y"]
    kp, npd, ku, nu = int(ok[pm].sum()), int(pm.sum()), int(ok[~pm].sum()), int((~pm).sum())
    wp, wu = wilson_ci(kp, npd), wilson_ci(ku, nu)
    prow.append({"model": it["label"], "n_padded": npd, "acc_padded": kp / npd, "padded_ci": f"{wp[0]:.3f}-{wp[1]:.3f}",
                 "n_unpadded": nu, "acc_unpadded": ku / nu, "unpadded_ci": f"{wu[0]:.3f}-{wu[1]:.3f}", "gap": ku / nu - kp / npd})
P = pd.DataFrame(prow)
print("\nACCURACY ON PADDED vs UNPADDED TEST IMAGES (exploratory, not pre-registered):")
print(P.round(4).to_string(index=False))
print("Reading: the gap points the same way for all six models, but the test sets are nested and the padded groups are small. "
      "A lower accuracy on padded images does not show that the models use image geometry as a shortcut; resolution, source and class mix "
      "are untested alternative explanations.")
CVT.to_csv(DIRS["results"] / "test_vs_cv_selected.csv", index=False); P.to_csv(DIRS["results"] / "test_padded_vs_unpadded.csv", index=False)
print("\nSaved figures 26-29 and the CSVs. The selection was not changed by any of this.")

In [ ]:
# ============================================================
# CELL 64: STAGE N - FINAL MODEL RETRAIN AND ONE-TIME TEST EVALUATION
# ============================================================
FSEL = load_json(DIRS["logs"] / "final_selection.json")
assert FSEL is not None and FSEL.get("locked_before_test") is True, "run CELL 57 first"
fm = FSEL["final_model"]
FINAL_ARCH, FINAL_SPLIT, FINAL_ACT = fm["arch"], fm["split"], fm["activation"]
FINAL_SEED, EXTRA_SEEDS = 42, [43, 44]          # extra seeds are DESCRIPTIVE ONLY; set EXTRA_SEEDS = [] to skip (saves ~12 min)
FINAL_DIR = DIRS["final"]; dname = lambda a: ARCH_SPECS[a]["display"]
padded_all = (df_clean["width"].values != df_clean["height"].values)

proto_path = DIRS["logs"] / "final_model_protocol.json"
if not proto_path.exists():
    save_json({"written_at": time.strftime("%Y-%m-%d %H:%M:%S"), "final_model": fm,
               "rule": f"the seed-{FINAL_SEED} model is THE final model; extra seeds {EXTRA_SEEDS} are variance information only and are never swapped in",
               "training": "same procedure as Stage G: tuned hyperparameters, fit/val of the split, early stopping on validation loss",
               "test": "each model evaluated once on the split's test set; no feedback into any decision"}, proto_path)
print(f"Final model (from logs/final_selection.json): {dname(FINAL_ARCH)} | split {FINAL_SPLIT} | {FINAL_ACT.upper()} | seed {FINAL_SEED}")

def final_rid(seed): return make_run_id("final", FINAL_SPLIT, FINAL_ARCH, FINAL_ACT, trial=f"seed{seed}")

def train_final(seed):
    rid, s = final_rid(seed), get_split(FINAL_SPLIT)
    r = train_and_evaluate(rid, FINAL_ARCH, FINAL_ACT, FINAL_SPLIT, s["fit_idx"], s["val_idx"],
                           hp_overrides=final_overrides(FINAL_ARCH), seed=seed, verbose=0)
    assert run_paths(rid)["ckpt"].exists(), f"{rid}: registered but its checkpoint is missing (restored zips exclude weights). Delete this run id from logs/run_registry.json and re-run."
    return rid, r

def test_final(rid, seed):
    jp = FINAL_DIR / f"test_{rid}.json"
    if jp.exists(): return load_json(jp), np.load(FINAL_DIR / f"test_{rid}.npz")
    s = get_split(FINAL_SPLIT); idx = s["test_idx"]
    reset_session(seed)
    model = build_model(FINAL_ARCH, FINAL_ACT, get_hp(FINAL_ARCH, **final_overrides(FINAL_ARCH)), weights=None)
    model.load_weights(str(run_paths(rid)["ckpt"]))
    global TEST_UNLOCKED
    TEST_UNLOCKED = True
    try:
        probs, m = evaluate_subset(model, FINAL_SPLIT, "test", idx)
    finally:
        TEST_UNLOCKED = False
    y, pred = y_all[idx], probs.argmax(1); ok = pred == y; pm = padded_all[idx]
    res = {"run_id": rid, "seed": seed, "n": int(len(idx)), "metrics": m, "acc_wilson": list(wilson_ci(int(ok.sum()), len(idx))),
           "boot_ci": cluster_bootstrap_ci(y, pred, groups=groups_all[idx], n_boot=1000),
           "padded": {"n_padded": int(pm.sum()), "acc_padded": float(ok[pm].mean()), "acc_unpadded": float(ok[~pm].mean())}}
    np.savez_compressed(FINAL_DIR / f"test_{rid}.npz", test_idx=idx, probs=probs)
    save_json(res, jp); del model; reset_session(seed)
    return res, np.load(FINAL_DIR / f"test_{rid}.npz")

# ---- final model (seed 42) ----
rid42, tr42 = train_final(FINAL_SEED)
res42, z42 = test_final(rid42, FINAL_SEED)
m, idx, probs = res42["metrics"], z42["test_idx"], z42["probs"]; y = y_all[idx]
assert not TEST_UNLOCKED

print("\n" + "=" * 100); print(f"FINAL MODEL TEST RESULT: {dname(FINAL_ARCH)} {FINAL_ACT.upper()} | trained on {FINAL_SPLIT} | test n={res42['n']}"); print("=" * 100)
print(f"Accuracy {m['accuracy']:.4f} [Wilson 95% CI {res42['acc_wilson'][0]:.3f}-{res42['acc_wilson'][1]:.3f}] | "
      f"macro-F1 {m['f1_macro']:.4f} [boot {res42['boot_ci']['f1_macro'][0]:.3f}-{res42['boot_ci']['f1_macro'][1]:.3f}] | kappa {m['kappa']:.4f}")
print(f"Macro precision {m['precision_macro']:.4f} | weighted precision {m['precision_weighted']:.4f} | macro recall {m['recall_macro']:.4f} | "
      f"weighted recall {m['recall_weighted']:.4f} | weighted F1 {m['f1_weighted']:.4f}")
print(f"ROC-AUC macro {m['roc_auc_macro']:.4f} | weighted {m['roc_auc_weighted']:.4f} | PR-AUC macro {m['pr_auc_macro']:.4f} | test loss {m['loss']:.4f} | ECE {m['ece']:.4f}")
print("\nPer-class metrics:")
print(pd.DataFrame(m["per_class"]).T.loc[CLASS_NAMES, ["precision", "sensitivity", "specificity", "fpr", "fnr", "f1", "roc_auc", "pr_auc", "support"]].round(4).to_string())
print("\nClassification report:\n" + classification_report_text(y, probs))
cmdf = pd.DataFrame(m["confusion_matrix"], index=[f"true {n}" for n in names] if (names := [CLASS_DISPLAY[i] for i in range(4)]) else None,
                    columns=[f"pred {n}" for n in names])
print("Confusion matrix:\n" + cmdf.to_string())
print(f"Padded test images ({res42['padded']['n_padded']}): accuracy {res42['padded']['acc_padded']:.3f} | unpadded: {res42['padded']['acc_unpadded']:.3f}")

hist = pd.read_csv(run_paths(rid42)["history"]); s42 = tr42["summary"]
print(f"\nTraining: epochs {s42['epochs_run']} (best epoch {s42['best_epoch']}) | clean train acc {s42['train_accuracy']:.4f} | "
      f"val acc {s42['val_accuracy']:.4f} F1 {s42['val_f1_macro']:.4f} loss {s42['val_loss']:.4f} | train time {s42['train_time_s']/60:.1f} min | "
      f"GPU latency {s42['latency_ms_bs1']:.1f} ms/img | params {s42['params_total']:,}")
js = load_json(DIRS["cv"] / f"{FINAL_SPLIT}__{FINAL_ARCH}__{FINAL_ACT}_summary.json")
if js: print(f"Reference: CV accuracy {js['mean']['acc']:.4f} +/- {js['sd']['acc']:.4f} | CV macro-F1 {js['mean']['f1_macro']:.4f} +/- {js['sd']['f1_macro']:.4f}")

# ---- saved files and figures ----
cmdf.to_csv(FINAL_DIR / "final_confusion_matrix.csv")
(FINAL_DIR / "final_classification_report.txt").write_text(classification_report_text(y, probs))
np.savez_compressed(FINAL_DIR / "final_test_probs.npz", test_idx=idx, probs=probs, y=y)
save_json({"run_id": rid42, **res42}, FINAL_DIR / "final_test_metrics.json")
hist.to_csv(FINAL_DIR / "final_training_history.csv", index=False)
plot_history(hist, f"FINAL {dname(FINAL_ARCH)} {FINAL_ACT.upper()} {FINAL_SPLIT}", FINAL_DIR / "final_history.png", show=True)
plot_confusion(m["confusion_matrix"], f"TEST FINAL {dname(FINAL_ARCH)} {FINAL_ACT.upper()}", FINAL_DIR / "final_confusion.png", show=True)
plot_roc_pr(y, probs, f"TEST FINAL {dname(FINAL_ARCH)} {FINAL_ACT.upper()}", FINAL_DIR / "final_roc_pr.png", show=True)

# ---- descriptive extra seeds ----
if EXTRA_SEEDS:
    rows = [{"seed": FINAL_SEED, "role": "FINAL", "test_acc": m["accuracy"], "test_f1": m["f1_macro"], "test_loss": m["loss"],
             "val_f1": s42["val_f1_macro"], "epochs": s42["epochs_run"]}]
    for sd in EXTRA_SEEDS:
        rid, tr = train_final(sd); r_, _ = test_final(rid, sd)
        rows.append({"seed": sd, "role": "descriptive only", "test_acc": r_["metrics"]["accuracy"], "test_f1": r_["metrics"]["f1_macro"],
                     "test_loss": r_["metrics"]["loss"], "val_f1": tr["summary"]["val_f1_macro"], "epochs": tr["summary"]["epochs_run"]})
    SD_TAB = pd.DataFrame(rows); SD_TAB.to_csv(FINAL_DIR / "final_seed_variability.csv", index=False)
    print("\nSEED VARIABILITY (same configuration, different seeds; the seed-42 model stays the final model):")
    print(SD_TAB.round(4).to_string(index=False))
    print(f"Test accuracy spread across seeds: {SD_TAB['test_acc'].min():.4f} to {SD_TAB['test_acc'].max():.4f}")
print("\nThe test set was evaluated once per model; TEST_UNLOCKED =", TEST_UNLOCKED)

In [ ]:
# ============================================================
# CELL 65: SAVE FINAL MODEL, LABELS, PREPROCESSING AND MODEL CONFIGS, RESULTS CSV
# ============================================================
FSEL = load_json(DIRS["logs"] / "final_selection.json"); fm = FSEL["final_model"]
FINAL_ARCH, FINAL_SPLIT, FINAL_ACT = fm["arch"], fm["split"], fm["activation"]; FINAL_SEED = 42
FINAL_DIR = DIRS["final"]; dname = lambda a: ARCH_SPECS[a]["display"]
rid42 = make_run_id("final", FINAL_SPLIT, FINAL_ARCH, FINAL_ACT, trial=f"seed{FINAL_SEED}")
ckpt = run_paths(rid42)["ckpt"]; assert ckpt.exists(), f"final checkpoint missing: {ckpt}"

hp_train = get_hp(FINAL_ARCH, **final_overrides(FINAL_ARCH)); hp_inf = {**hp_train, "aug": "none"}
reset_session()
trained = build_model(FINAL_ARCH, FINAL_ACT, hp_train, weights=None); trained.load_weights(str(ckpt))
inf = build_model(FINAL_ARCH, FINAL_ACT, hp_inf, weights=None)
n_copied = 0
for l in inf.layers:
    if l.weights:
        l.set_weights(trained.get_layer(l.name).get_weights()); n_copied += 1
print(f"Inference model built without the augmentation layer; weights copied for {n_copied} layers")

tidx = np.load(FINAL_DIR / "final_test_probs.npz")["test_idx"]; saved_probs = np.load(FINAL_DIR / "final_test_probs.npz")["probs"]
xs = X_all[tidx[:64]].astype(np.float32)
p_tr, p_inf = trained.predict(xs, verbose=0), inf.predict(xs, verbose=0)
assert np.allclose(p_tr, p_inf, atol=1e-3), "inference copy differs from the trained model"
assert np.allclose(p_inf, saved_probs[:64], atol=5e-3), "inference copy differs from the probabilities saved at test time"
print("[OK] inference copy reproduces the trained model and the saved test probabilities")

model_path = FINAL_DIR / "final_model.keras"
inf.save(model_path); inf.save_weights(FINAL_DIR / "final_model.weights.h5")
loaded = keras.models.load_model(model_path, custom_objects={"InputPreprocess": InputPreprocess}, compile=False)
assert np.allclose(loaded.predict(xs, verbose=0), p_inf, atol=1e-3), "reloaded .keras model gives different predictions"
print(f"[OK] saved and reloaded {model_path.name} ({model_path.stat().st_size/1e6:.0f} MB); predictions identical (same session)")

save_json({"index_to_class": {str(i): CLASS_NAMES[i] for i in range(NUM_CLASSES)},
           "index_to_display": {str(i): CLASS_DISPLAY[i] for i in range(NUM_CLASSES)}}, FINAL_DIR / "class_labels.json")
save_json({**PREPROC_CONFIG, "model_input": "float32 pixels 0-255, shape (224, 224, 3); the model scales them itself",
           "in_model_preprocessing": ARCH_SPECS[FINAL_ARCH]["preprocess"]}, FINAL_DIR / "preprocessing_config.json")
r42 = load_result(rid42)
save_json({"architecture": FINAL_ARCH, "display_name": dname(FINAL_ARCH), "activation_in_head": FINAL_ACT, "trained_on_split": FINAL_SPLIT,
           "seed": FINAL_SEED, "hyperparameters_training": hp_train, "hyperparameters_inference_copy": hp_inf,
           "params": r42["params_by_stage"], "epochs_run": r42["summary"]["epochs_run"], "train_time_s": r42["summary"]["train_time_s"],
           "latency_ms_bs1_gpu": r42["summary"]["latency_ms_bs1"], "augmentation_during_training": AUG_CHOICE,
           "selection": fm, "tensorflow": tf.__version__, "keras": keras.__version__, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")},
          FINAL_DIR / "model_config.json")

# ---- results CSV: final model, extra seeds, the six selected models ----
rows = []; fmx = load_json(FINAL_DIR / "final_test_metrics.json")["metrics"]
rows.append({"role": "FINAL model", "split": FINAL_SPLIT, "arch": FINAL_ARCH, "activation": FINAL_ACT, "seed": FINAL_SEED, "source": "final retrain",
             "n_test": fmx["n"], "accuracy": fmx["accuracy"], "macro_f1": fmx["f1_macro"], "kappa": fmx["kappa"], "roc_auc_macro": fmx["roc_auc_macro"], "loss": fmx["loss"]})
sv = FINAL_DIR / "final_seed_variability.csv"
if sv.exists():
    for r in pd.read_csv(sv).itertuples():
        if r.role != "FINAL":
            rows.append({"role": "seed variability (descriptive)", "split": FINAL_SPLIT, "arch": FINAL_ARCH, "activation": FINAL_ACT, "seed": r.seed,
                         "source": "final retrain", "n_test": fmx["n"], "accuracy": r.test_acc, "macro_f1": r.test_f1, "loss": r.test_loss})
for r in pd.read_csv(DIRS["results"] / "test_results_all.csv").query("selected").itertuples():
    rows.append({"role": "selected (best two)", "split": r.split, "arch": r.arch, "activation": r.activation, "seed": 42, "source": r.source,
                 "n_test": r.n, "accuracy": r.test_acc, "macro_f1": r.f1_macro, "kappa": r.kappa, "roc_auc_macro": r.roc_auc_macro, "loss": r.loss})
pd.DataFrame(rows).to_csv(FINAL_DIR / "final_results.csv", index=False)

(FINAL_DIR / "MODEL_CARD.txt").write_text(
    "EDUCATIONAL RESEARCH PROTOTYPE - NOT A CLINICALLY VALIDATED DIAGNOSTIC SYSTEM.\n\n"
    f"Model: {dname(FINAL_ARCH)} (ImageNet-pretrained, two-stage fine-tuning), 4-class brain MRI classifier "
    "(glioma, meningioma, pituitary, no tumor).\n"
    "Data: Kaggle 'brain-tumor-classification-mri' after removing exact duplicates and grouping near-duplicates; no patient IDs exist, "
    "so patient-level generalisation was not measured.\n"
    "Known limits: class cues from image source/size are present in the dataset; test sets are small (e.g. 30 No Tumor images in the "
    "90:10 test set); single seed; test accuracy is about 0.91-0.93 on a leak-free split.\n"
    "Grad-CAM maps are qualitative and are not evidence of clinical correctness.\n")
print("\nFiles in", FINAL_DIR)
for p in sorted(FINAL_DIR.iterdir()): print(f"   {p.name:<48}{p.stat().st_size/1e6:8.2f} MB")
del trained, inf, loaded; reset_session()

In [ ]:
# ============================================================
# CELL 66: GRAD-CAM FOR THE FINAL MODEL AND THE SECOND MODEL OF ITS SPLIT + BACKGROUND-ATTENTION CHECK
# ============================================================
FSEL = load_json(DIRS["logs"] / "final_selection.json"); fm = FSEL["final_model"]
FINAL_ARCH, FINAL_SPLIT, FINAL_ACT = fm["arch"], fm["split"], fm["activation"]
SEL = load_json(DIRS["logs"] / "stage_h_selection.json"); FINAL_DIR, GC_DIR = DIRS["final"], DIRS["gradcam"]
dname = lambda a: ARCH_SPECS[a]["display"]; padded_all = (df_clean["width"].values != df_clean["height"].values)

second_arch = next(a for a in SEL["splits"][FINAL_SPLIT]["top_two"] if a != FINAL_ARCH)
second_act = SEL["splits"][FINAL_SPLIT]["best_activation"][second_arch]
def find_ckpt(sp, arch, act):
    for kind in ("init", "retrain"):
        p = run_paths(make_run_id(kind, sp, arch, act))["ckpt"]
        if p.exists(): return p
    raise FileNotFoundError(f"no checkpoint for {sp}/{arch}/{act}")
zf = np.load(FINAL_DIR / "final_test_probs.npz"); tidx = zf["test_idx"]; ty = y_all[tidx]
z2 = np.load(DIRS["results"] / "test" / f"{make_run_id('test', FINAL_SPLIT, second_arch, second_act)}.npz")
assert np.array_equal(z2["test_idx"], tidx)
MODELS = [("FINAL " + dname(FINAL_ARCH), FINAL_ARCH, FINAL_ACT, run_paths(make_run_id("final", FINAL_SPLIT, FINAL_ARCH, FINAL_ACT, trial="seed42"))["ckpt"], zf["probs"]),
          ("SECOND " + dname(second_arch), second_arch, second_act, find_ckpt(FINAL_SPLIT, second_arch, second_act), z2["probs"])]

def load_f32(arch, act, ckpt):
    keras.backend.clear_session(); gc.collect(); keras.mixed_precision.set_global_policy("float32")
    m = build_model(arch, act, get_hp(arch, **final_overrides(arch)), weights=None); m.load_weights(str(ckpt)); return m

def gradcam_batch(model, x):
    """Grad-CAM on the backbone's last feature map, for the PREDICTED class, using the pre-softmax score."""
    pre, bb = model.get_layer("preprocess"), get_backbone(model)
    head = [model.get_layer(n) for n in ("gap", "head_drop1", "head_dense", "head_act", "head_drop2")]; pl = model.get_layer("predictions")
    with tf.GradientTape() as tape:
        f = bb(pre(tf.constant(x.astype(np.float32))), training=False)
        h = f
        for l in head: h = l(h, training=False)
        logits = tf.matmul(h, pl.kernel) + pl.bias
        cls = tf.argmax(logits, axis=1)
        score = tf.reduce_sum(logits * tf.one_hot(cls, NUM_CLASSES), axis=1)
    g = tape.gradient(score, f)
    cam = tf.nn.relu(tf.reduce_sum(f * tf.reduce_mean(g, axis=(1, 2))[:, None, None, :], axis=-1))
    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE), method="bilinear")[..., 0].numpy()
    cam = cam / (cam.reshape(len(cam), -1).max(1)[:, None, None] + 1e-8)
    return cam, tf.nn.softmax(logits).numpy()

bg_mask = (X_all[tidx][:, :, :, 0] <= 10); area_bg = bg_mask.mean((1, 2)); pad_t = padded_all[tidx]
rng = np.random.RandomState(SEED)

def cam_figure(title, pos, cams, probs, fname):
    rows = int(np.ceil(len(pos) / 2)); fig, axes = plt.subplots(rows, 6, figsize=(21, 3.7 * rows)); axes = np.atleast_2d(axes)
    for a in axes.ravel(): a.axis("off")
    for k, p in enumerate(pos):
        r, c0 = divmod(k, 2); c0 *= 3; g = X_all[tidx[p]][..., 0] / 255.0
        over = 0.55 * np.stack([g] * 3, -1) + 0.45 * plt.cm.jet(cams[p])[..., :3]
        pr = probs[p].argmax(); sub = f"true {CLASS_DISPLAY[ty[p]]} | pred {CLASS_DISPLAY[pr]} ({probs[p].max():.2f})"
        axes[r, c0].imshow(g, cmap="gray"); axes[r, c0].set_title(sub, fontsize=9)
        axes[r, c0 + 1].imshow(cams[p], cmap="jet", vmin=0, vmax=1); axes[r, c0 + 1].set_title("Grad-CAM heatmap", fontsize=9)
        axes[r, c0 + 2].imshow(over); axes[r, c0 + 2].set_title("overlay", fontsize=9)
        for a in axes[r, c0:c0 + 3]: a.axis("off")
    fig.suptitle(title, fontsize=14); plt.tight_layout(); plt.savefig(GC_DIR / fname, dpi=130, bbox_inches="tight"); plt.show()

summary = []
for name, arch, act, ck, saved in MODELS:
    model = load_f32(arch, act, ck); cams = np.zeros((len(tidx), IMG_SIZE, IMG_SIZE), np.float32); probs = np.zeros((len(tidx), NUM_CLASSES), np.float32)
    for i in range(0, len(tidx), 32):
        cams[i:i + 32], probs[i:i + 32] = gradcam_batch(model, X_all[tidx[i:i + 32]])
    agree = float((probs.argmax(1) == saved.argmax(1)).mean())
    print(f"\n{name}: float32 re-computation agrees with the saved test predictions on {100*agree:.1f}% of images")
    assert agree > 0.95, "float32 model does not reproduce the saved predictions"
    pred = probs.argmax(1); ok = pred == ty
    pos_ok = [int(p) for c in range(NUM_CLASSES) for p in rng.choice(np.where((ty == c) & ok)[0], min(2, int(((ty == c) & ok).sum())), replace=False)]
    wrong = np.where(~ok)[0]; pos_bad = [int(p) for p in rng.choice(wrong, min(4, len(wrong)), replace=False)]
    safe = name.replace(" ", "_")
    cam_figure(f"{name} - Grad-CAM, correctly classified test images (2 random per class)", pos_ok, cams, probs, f"gradcam_{safe}_correct.png")
    cam_figure(f"{name} - Grad-CAM, misclassified test images (random sample)", pos_bad, cams, probs, f"gradcam_{safe}_errors.png")
    mass_bg = (cams * bg_mask).sum((1, 2)) / (cams.sum((1, 2)) + 1e-8); valid = area_bg > 0.01; ratio = mass_bg / np.maximum(area_bg, 1e-6)
    for grp, msk in [("all", np.ones(len(tidx), bool)), ("unpadded", ~pad_t), ("padded", pad_t), ("correct", ok), ("wrong", ~ok)]:
        v = valid & msk
        summary.append({"model": name, "group": grp, "n": int(v.sum()), "area_in_background": float(area_bg[v].mean()),
                        "attention_in_background": float(mass_bg[v].mean()), "ratio_attention_to_area": float(ratio[v].mean())})
    np.save(GC_DIR / f"cams_{safe}.npy", cams.astype(np.float16)); del model, cams; keras.backend.clear_session(); gc.collect()

keras.mixed_precision.set_global_policy("mixed_float16" if MIXED_PRECISION else "float32")
S = pd.DataFrame(summary); S.to_csv(DIRS["results"] / "gradcam_background_attention.csv", index=False)
print("\nBACKGROUND ATTENTION (background = pixels <= 10 on a 0-255 scale; ratio 1.0 = attention proportional to area):")
print(S.round(3).to_string(index=False))
print("Caveats: the threshold is arbitrary; dark anatomy counts as background; Grad-CAM is qualitative and not proof of clinical correctness; "
      "images were sampled at random, not chosen.")

In [ ]:
# ============================================================
# CELL 67: ERROR AND CONFIDENCE ANALYSIS OF THE FINAL MODEL (TEST SET)
# ============================================================
FSEL = load_json(DIRS["logs"] / "final_selection.json"); fm = FSEL["final_model"]; FINAL_DIR = DIRS["final"]
zf = np.load(FINAL_DIR / "final_test_probs.npz"); idx, probs = zf["test_idx"], zf["probs"]; y = y_all[idx]
pred, conf = probs.argmax(1), probs.max(1); ok = pred == y; pad = (df_clean["width"].values != df_clean["height"].values)[idx]
names = [CLASS_DISPLAY[i] for i in range(NUM_CLASSES)]
print(f"Final model test set: n={len(y)} | errors {int((~ok).sum())} | accuracy {ok.mean():.4f}")

print(f"\nCONFIDENCE: mean max-probability correct {conf[ok].mean():.3f} | wrong {conf[~ok].mean():.3f} | "
      f"median correct {np.median(conf[ok]):.3f} | wrong {np.median(conf[~ok]):.3f}")
print(f"Errors with confidence >= 0.90: {int((conf[~ok] >= 0.9).sum())} of {int((~ok).sum())}")
bins = [0, 0.5, 0.7, 0.9, 0.99, 1.0001]
B = pd.DataFrame({"conf_bin": pd.cut(conf, bins, right=False), "ok": ok}).groupby("conf_bin", observed=True)["ok"].agg(n="size", accuracy="mean", mean_ok="sum")
B["mean_conf"] = pd.Series(conf).groupby(pd.cut(conf, bins, right=False), observed=True).mean().values
print("\nAccuracy by confidence bin:"); print(B[["n", "accuracy", "mean_conf"]].round(3).to_string())
cov = pd.DataFrame([{"threshold": t, "kept_images": int((conf >= t).sum()), "coverage": float((conf >= t).mean()), "accuracy_on_kept": float(ok[conf >= t].mean())}
                    for t in [0.5, 0.7, 0.9, 0.95, 0.99] if (conf >= t).any()])
print("\nIf low-confidence predictions were referred for review (descriptive only):"); print(cov.round(3).to_string(index=False))

cmx = pd.DataFrame(confusion_matrix(y, pred, labels=range(4)), index=[f"true {n}" for n in names], columns=[f"pred {n}" for n in names])
print("\nConfusion matrix:\n" + cmx.to_string())
pairs = sorted([(cmx.iloc[i, j], names[i], names[j]) for i in range(4) for j in range(4) if i != j], reverse=True)[:5]
print("Largest confusions:", " | ".join(f"{a}->{b}: {n}" for n, a, b in pairs))

print("\nPADDED vs UNPADDED by true class (exploratory; small groups):")
rows = []
for c in range(4):
    for nm, msk in [("padded", pad), ("unpadded", ~pad)]:
        sel_ = (y == c) & msk; n_ = int(sel_.sum()); k_ = int(ok[sel_].sum()); w = wilson_ci(k_, n_) if n_ else (np.nan, np.nan)
        rows.append({"class": names[c], "group": nm, "n": n_, "accuracy": k_ / n_ if n_ else np.nan, "ci_lo": w[0], "ci_hi": w[1]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# reliability diagram and confidence histogram
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2)); edges = np.linspace(0.25, 1.0, 11)
mids, accs_, ns_ = [], [], []
for lo_, hi_ in zip(edges[:-1], edges[1:]):
    m_ = (conf > lo_) & (conf <= hi_ + (1e-9 if hi_ == 1.0 else 0))
    if m_.any(): mids.append(conf[m_].mean()); accs_.append(ok[m_].mean()); ns_.append(int(m_.sum()))
axes[0].plot([0.25, 1], [0.25, 1], "k--", lw=1, label="perfect calibration"); axes[0].plot(mids, accs_, "-o", lw=2, label="final model")
for a, b_, n_ in zip(mids, accs_, ns_): axes[0].annotate(str(n_), (a, b_), textcoords="offset points", xytext=(0, 7), fontsize=8, ha="center")
axes[0].set_title("Reliability diagram (labels = images per bin)", fontsize=13); axes[0].set_xlabel("Mean confidence", fontsize=12)
axes[0].set_ylabel("Accuracy", fontsize=12); axes[0].legend()
axes[1].hist(conf[ok], bins=20, range=(0.25, 1), alpha=0.7, label=f"correct (n={int(ok.sum())})"); axes[1].hist(conf[~ok], bins=20, range=(0.25, 1), alpha=0.7, label=f"wrong (n={int((~ok).sum())})")
axes[1].set_title("Confidence of correct and wrong predictions", fontsize=13); axes[1].set_xlabel("Maximum predicted probability", fontsize=12)
axes[1].set_ylabel("Number of test images", fontsize=12); axes[1].legend()
plt.tight_layout(); plt.savefig(FINAL_DIR / "final_confidence_analysis.png", dpi=150, bbox_inches="tight"); plt.show()

# most confident errors
order = np.where(~ok)[0]; order = order[np.argsort(-conf[order])][:8]
fig, axes = plt.subplots(2, 4, figsize=(18, 9.5))
for a in axes.ravel(): a.axis("off")
for a, p in zip(axes.ravel(), order):
    a.imshow(X_all[idx[p]][..., 0], cmap="gray"); a.set_title(f"true {CLASS_DISPLAY[y[p]]} -> pred {CLASS_DISPLAY[pred[p]]} ({conf[p]:.2f})\n{'padded' if pad[p] else 'unpadded'}", fontsize=10)
fig.suptitle("The most confident errors of the final model (test set)", fontsize=14); plt.tight_layout()
plt.savefig(FINAL_DIR / "final_confident_errors.png", dpi=130, bbox_inches="tight"); plt.show()

E = df_clean.loc[idx[~ok], ["image_id", "rel_path", "orig_split"]].copy()
E["true"], E["pred"], E["confidence"], E["padded"] = [CLASS_DISPLAY[v] for v in y[~ok]], [CLASS_DISPLAY[v] for v in pred[~ok]], conf[~ok], pad[~ok]
E.sort_values("confidence", ascending=False).to_csv(FINAL_DIR / "final_misclassified_images.csv", index=False)
print(f"\nSaved final_misclassified_images.csv ({len(E)} rows), final_confidence_analysis.png, final_confident_errors.png")
print("Note: the dataset has no patient IDs, so these errors cannot be grouped by patient.")

In [ ]:
# ============================================================
# CELL 68: NEW-IMAGE PREDICTION SYSTEM (SELF-CONTAINED; EDUCATIONAL PROTOTYPE, NOT A DIAGNOSTIC TOOL)
# ============================================================
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import tensorflow as tf
from tensorflow import keras

MODEL_DIR = Path("/kaggle/working/brain_tumor_project/final")   # in a new notebook: Path("/kaggle/input/<your-dataset>/brain_tumor_project/final")
IMAGE_PATH = None                                               # set to your own MRI image (jpg/png); None = use a demo image from the dataset

try:
    _reg = keras.saving.register_keras_serializable(package="btc")
except AttributeError:
    _reg = keras.utils.register_keras_serializable(package="btc")

@_reg
class InputPreprocess(keras.layers.Layer):          # identical to the training definition (needed to load the saved model)
    def __init__(self, mode="builtin", mean=0.0, std=1.0, **kwargs):
        kwargs.setdefault("dtype", "float32"); super().__init__(**kwargs)
        self.mode, self.mean, self.std = mode, float(mean), float(std)
    def call(self, x):
        x = tf.cast(x, tf.float32)
        if self.mode == "caffe":   x = tf.reverse(x, axis=[-1]) - tf.constant([103.939, 116.779, 123.68], tf.float32)
        elif self.mode == "torch": x = (x / 255.0 - tf.constant([0.485, 0.456, 0.406], tf.float32)) / tf.constant([0.229, 0.224, 0.225], tf.float32)
        elif self.mode == "standardize": x = (x / 255.0 - self.mean) / self.std
        return x
    def compute_output_shape(self, input_shape): return input_shape
    def get_config(self):
        c = super().get_config(); c.update({"mode": self.mode, "mean": self.mean, "std": self.std}); return c

labels = json.loads((MODEL_DIR / "class_labels.json").read_text())
CLASS_KEYS = [labels["index_to_class"][str(i)] for i in range(len(labels["index_to_class"]))]
CLASS_SHOW = [labels["index_to_display"][str(i)] for i in range(len(labels["index_to_display"]))]
pre_cfg = json.loads((MODEL_DIR / "preprocessing_config.json").read_text()); SIZE = int(pre_cfg["img_size"])
model = keras.models.load_model(MODEL_DIR / "final_model.keras", custom_objects={"InputPreprocess": InputPreprocess}, compile=False)
print("Loaded model:", json.loads((MODEL_DIR / "model_config.json").read_text())["display_name"], "| classes:", CLASS_SHOW)

def preprocess_for_model(path):
    """Same steps as training: grayscale -> centred black pad to square -> bicubic resize -> 3 identical channels (uint8)."""
    with Image.open(path) as im:
        im.load(); g = im.convert("L")
    w, h = g.size; s = max(w, h)
    if w != h:
        canvas = Image.new("L", (s, s), 0); canvas.paste(g, ((s - w) // 2, (s - h) // 2)); g = canvas
    a = np.asarray(g.resize((SIZE, SIZE), Image.BICUBIC), dtype=np.uint8)
    return np.repeat(a[..., None], 3, axis=2)

def predict_mri(path, show=True):
    x = preprocess_for_model(path)
    p = model(x[None].astype(np.float32), training=False).numpy()[0].astype(np.float64); p = p / p.sum(); k = int(p.argmax())
    if show:
        with Image.open(path) as im: shown = np.asarray(im.convert("L"))
        fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.25]})
        ax[0].imshow(shown, cmap="gray"); ax[0].axis("off")
        ax[0].set_title(f"Input MRI\nPredicted: {CLASS_SHOW[k]}  (confidence {100*p[k]:.1f}%)", fontsize=12)
        bars = ax[1].barh(CLASS_SHOW[::-1], p[::-1], color=["#c44e52" if i == k else "#4c72b0" for i in range(len(p))][::-1], edgecolor="black")
        for b, v in zip(bars, p[::-1]): ax[1].text(min(v + 0.01, 0.9), b.get_y() + b.get_height() / 2, f"{100*v:.1f}%", va="center", fontsize=11)
        ax[1].set_xlim(0, 1.05); ax[1].set_xlabel("Probability", fontsize=12); ax[1].set_title("Probability across the four classes", fontsize=12)
        fig.text(0.5, -0.02, "Educational research prototype - not a clinically validated diagnostic system.", ha="center", fontsize=10, style="italic")
        plt.tight_layout(); plt.show()
    return {"class": CLASS_SHOW[k], "confidence": float(p[k]), "probabilities": {CLASS_SHOW[i]: float(p[i]) for i in range(len(p))}}

# ---- consistency check with the training notebook's cached data (only when those objects exist in this session) ----
if "X_all" in globals() and "df_clean" in globals() and (MODEL_DIR / "final_test_probs.npz").exists():
    ref = np.load(MODEL_DIR / "final_test_probs.npz"); n_ok = 0
    for j in range(5):
        i = int(ref["test_idx"][j])
        assert np.array_equal(preprocess_for_model(df_clean.loc[i, "path"]), X_all[i]), "preprocessing differs from the cached training input"
        r = predict_mri(df_clean.loc[i, "path"], show=False)
        assert np.allclose(list(r["probabilities"].values()), ref["probs"][j], atol=5e-3), "prediction differs from the saved test probabilities"
        n_ok += 1
    print(f"[OK] preprocessing is byte-identical to the cached training input and predictions match the saved test probabilities ({n_ok} images)")

# ---- predict ----
if IMAGE_PATH is None:
    cand = list(MODEL_DIR.parent.glob("data/clean_index.csv"))
    if cand:
        import pandas as pd; IMAGE_PATH = pd.read_csv(cand[0]).iloc[0]["path"]
    else:
        IMAGE_PATH = next(iter(Path("/kaggle/input").rglob("*.jpg")), None)
if IMAGE_PATH is None:
    print("Set IMAGE_PATH to an MRI image file and run this cell again.")
else:
    print("Image:", IMAGE_PATH); print(predict_mri(IMAGE_PATH))

In [ ]:
# ============================================================
# CELL 69: FINAL BACKUP (INCLUDES final/ MODEL FILES; EXCLUDES OTHER WEIGHTS AND THE IMAGE CACHE)
# ============================================================
zpath = Path("/kaggle/working/project_backup_final.zip")
if zpath.exists(): zpath.unlink()
final_root = DIRS["final"].resolve(); n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(BASE.rglob("*")):
        if not p.is_file() or p.name.endswith(".tmp") or p.name == "X_clean_224.npy": continue
        if p.name.endswith((".weights.h5", ".keras")) and final_root not in p.resolve().parents: continue
        zf.write(p, p.relative_to(BASE.parent)); n += 1
with zipfile.ZipFile(zpath) as zf:
    assert zf.testzip() is None; names_ = set(zf.namelist())
for must in ["final/final_model.keras", "final/class_labels.json", "final/preprocessing_config.json", "final/model_config.json",
             "final/final_test_metrics.json", "final/final_results.csv", "logs/run_registry.json", "splits/splits_frozen.json", "logs/final_selection.json"]:
    assert f"brain_tumor_project/{must}" in names_, f"missing {must}"
print(f"Written {zpath} | {n} files | {zpath.stat().st_size/1e6:.0f} MB")
print("Download: right-hand panel -> Data -> Output -> /kaggle/working -> project_backup_final.zip -> three dots -> Download.")